Preprocessing of training set

In [ ]:
#!/usr/bin/env python

import os
import argparse
from pathlib import Path

import numpy as np
import nibabel as nib
from scipy.ndimage import zoom


def resample_to_spacing(data, src_spacing, tgt_spacing, order):
    """
    Data: np.ndarray with shape (X, Y, Z) or (X, Y, Z, C)
    src_spacing, tgt_spacing: iterable of length 3
    order: interpolation order (3 = cubic for image, 0 = nearest for label)
    """
    src_spacing = np.array(src_spacing, dtype=np.float32)
    tgt_spacing = np.array(tgt_spacing, dtype=np.float32)
    zoom_factors = src_spacing / tgt_spacing

    if data.ndim == 3:
        factors = zoom_factors
    elif data.ndim == 4:
        factors = (*zoom_factors, 1.0)  # don't scale channels
    else:
        raise ValueError(f"Unsupported data ndim {data.ndim}, expected 3 or 4.")

    resampled = zoom(data, factors, order=order)
    return resampled


def compute_label_bbox(label, margin=0):
    """
    Compute bounding box of non-zero labels, with margin in voxels.
    label: np.ndarray (X, Y, Z) integer labels
    returns: slices or None if no foreground
    """
    if np.max(label) == 0:
        return None

    coords = np.where(label > 0)
    xmin, xmax = coords[0].min(), coords[0].max()
    ymin, ymax = coords[1].min(), coords[1].max()
    zmin, zmax = coords[2].min(), coords[2].max()

    xmin = max(xmin - margin, 0)
    ymin = max(ymin - margin, 0)
    zmin = max(zmin - margin, 0)

    xmax = min(xmax + margin, label.shape[0] - 1)
    ymax = min(ymax + margin, label.shape[1] - 1)
    zmax = min(zmax + margin, label.shape[2] - 1)

    return (slice(xmin, xmax + 1),
            slice(ymin, ymax + 1),
            slice(zmin, zmax + 1))


def zscore_normalize(img, mask=None, eps=1e-8):
    """
    Z-score normalization with mask.
    img: np.ndarray float
    mask: boolean array or None
    """
    if mask is None:
        mask = np.ones_like(img, dtype=bool)
    vals = img[mask]
    if vals.size == 0:
        return img
    mean = vals.mean()
    std = vals.std()
    if std < eps:
        std = eps
    img = (img - mean) / std
    return img


def percentile_normalize(img, p_lo=0.5, p_hi=99.5, mask=None, eps=1e-8):
    """
    Map intensities between [p_lo, p_hi] percentiles to [0,1].
    Everything below p_lo goes to 0, above p_hi to 1.
    Usually mask = (img != 0) to ignore the air background.
    """
    if mask is None:
        mask = np.ones_like(img, dtype=bool)

    vals = img[mask]
    if vals.size == 0:
        return img

    lo = np.percentile(vals, p_lo)
    hi = np.percentile(vals, p_hi)

    if hi - lo < eps:
        # almost constant volume, nothing sensible to do
        return img

    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo + eps)
    return img


def preprocess_pair(
    img_path,
    lbl_path,
    out_img_dir,
    out_lbl_dir,
    target_spacing,
    clip_range=None,
    do_zscore=False,
    crop_mode="label",
    crop_margin=10,
    percentile_norm=False,
    percentile_range=(0.5, 99.5),
):
    """
    Preprocess one image+label pair and save as .nii.gz.

    img_path, lbl_path: Path objects (input .nii/.nii.gz)
    out_img_dir, out_lbl_dir: Path objects (output directories)
    """
    print(f"Processing: {img_path.name}")

    img_nii = nib.load(str(img_path))
    img = img_nii.get_fdata().astype(np.float32)
    src_spacing = img_nii.header.get_zooms()[:3]

    # Labels
    if lbl_path is not None:
        lbl_nii = nib.load(str(lbl_path))
        lbl = lbl_nii.get_fdata().astype(np.int16)
    else:
        lbl_nii = None
        lbl = None

    # Resample image and label to target spacing
    if target_spacing is not None:
        img = resample_to_spacing(img, src_spacing, target_spacing, order=3)
        if lbl is not None:
            lbl = resample_to_spacing(lbl, src_spacing, target_spacing, order=0)
        spacing = target_spacing
    else:
        spacing = src_spacing

    # Intensity normalisation
    # HU clipping
    if clip_range is not None:
        lo, hi = clip_range
    else:
        lo, hi = -1000.0, 600.0  # default chest CT window

    img = np.clip(img, lo, hi)

    # Choose one of: percentile mapping, z-score, or simple [lo,hi] to [0,1]
    if percentile_norm:
        # Ignore pure-air voxels when computing percentiles
        mask = img != lo
        p_lo, p_hi = percentile_range
        img = percentile_normalize(img, p_lo=p_lo, p_hi=p_hi, mask=mask)
    elif do_zscore:
        mask = img != lo
        img = zscore_normalize(img, mask=mask)
    else:
        # Simple linear windowing [lo,hi] to [0,1]
        img = (img - lo) / (hi - lo + 1e-8)

    img = img.astype(np.float32)

    # Cropping

    # For inference, use crop_mode="none" so volume size stays global.
    if crop_mode == "label" and lbl is not None:
        bbox = compute_label_bbox(lbl, margin=crop_margin)
        if bbox is not None:
            img = img[bbox]
            lbl = lbl[bbox]
    elif crop_mode == "none":
        # No cropping
        pass
    elif crop_mode == "body":
        # Simple body mask
        mask = img != 0
        if np.any(mask):
            coords = np.where(mask)
            xmin, xmax = coords[0].min(), coords[0].max()
            ymin, ymax = coords[1].min(), coords[1].max()
            zmin, zmax = coords[2].min(), coords[2].max()
            xmin = max(xmin - crop_margin, 0)
            ymin = max(ymin - crop_margin, 0)
            zmin = max(zmin - crop_margin, 0)
            xmax = min(xmax + crop_margin, img.shape[0] - 1)
            ymax = min(ymax + crop_margin, img.shape[1] - 1)
            zmax = min(zmax + crop_margin, img.shape[2] - 1)
            bbox = (slice(xmin, xmax + 1),
                    slice(ymin, ymax + 1),
                    slice(zmin, zmax + 1))
            img = img[bbox]
            if lbl is not None:
                lbl = lbl[bbox]
    else:
        raise ValueError(f"Unknown crop_mode: {crop_mode}")

    affine = np.eye(4, dtype=np.float32)
    affine[0, 0] = spacing[0]
    affine[1, 1] = spacing[1]
    affine[2, 2] = spacing[2]

    out_img_dir.mkdir(parents=True, exist_ok=True)
    if out_lbl_dir is not None:
        out_lbl_dir.mkdir(parents=True, exist_ok=True)

    out_img_path = out_img_dir / img_path.name
    img_out = nib.Nifti1Image(img.astype(np.float32), affine)
    nib.save(img_out, str(out_img_path))

    if lbl is not None and out_lbl_dir is not None:
        out_lbl_path = out_lbl_dir / lbl_path.name
        lbl_out = nib.Nifti1Image(lbl.astype(np.int16), affine)
        nib.save(lbl_out, str(out_lbl_path))

    print(f"  -> saved image to {out_img_path}")
    if lbl is not None and out_lbl_dir is not None:
        print(f"  -> saved label to {out_lbl_path}")

def run_preprocess_av_ct_nii(
    images,
    labels=None,
    out_images=None,
    out_labels=None,
    target_spacing=(1.0, 1.0, 1.0),
    clip=None,                          # e.g. (-1000, 400)
    percentile_norm=False,
    percentile_range=(0.5, 99.5),
    zscore=False,
    crop_mode="label",
    crop_margin=10,
):
    """
    Notebook wrapper for preprocess_av_ct_nii. All paths are strings.
    """
    images_dir = Path(images)
    labels_dir = Path(labels) if labels is not None else None
    out_images_dir = Path(out_images) if out_images is not None else images_dir
    out_labels_dir = Path(out_labels) if out_labels is not None else labels_dir

    if labels_dir is None and crop_mode == "label":
        raise ValueError("crop_mode='label' requires labels directory.")

    img_files = sorted(
        [
            p for p in images_dir.iterdir()
            if p.is_file() and (p.name.endswith(".nii") or p.name.endswith(".nii.gz"))
        ]
    )

    if not img_files:
        raise RuntimeError(f"No NIfTI files found in {images_dir}")

    for img_path in img_files:
        img_name = img_path.name
        base = img_name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]

        lbl_path = None
        if labels_dir is not None:
            candidates = []
            candidates.append(labels_dir / (base + ".nii.gz"))
            candidates.append(labels_dir / (base + ".nii"))

            if base.startswith("image_"):
                idx = base[len("image_"):]
                candidates.append(labels_dir / f"label_{idx}.nii.gz")
                candidates.append(labels_dir / f"label_{idx}.nii")

            for cand in candidates:
                if cand.exists():
                    lbl_path = cand
                    break

            if lbl_path is None:
                raise FileNotFoundError(
                    f"Missing label for {img_path.name}. "
                    f"Tried: {[str(c) for c in candidates]}"
                )

        preprocess_pair(
            img_path=img_path,
            lbl_path=lbl_path,
            out_img_dir=out_images_dir,
            out_lbl_dir=out_labels_dir,
            target_spacing=target_spacing,
            clip_range=clip,
            do_zscore=zscore and not percentile_norm,
            crop_mode=crop_mode,
            crop_margin=crop_margin,
            percentile_norm=percentile_norm,
            percentile_range=percentile_range,
        )

In [ ]:
run_preprocess_av_ct_nii(
    images="/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/VesselFMAdaptationMethod/vesselFM-main/vesselFM-main/data/imagesTr",
    labels="/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/VesselFMAdaptationMethod/vesselFM-main/vesselFM-main/data/labelsTr",
    out_images="/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/VesselFMAdaptationMethod/vesselFM-main/vesselFM-main/data/imagesTr_pre_3",
    out_labels="/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/VesselFMAdaptationMethod/vesselFM-main/vesselFM-main/data/labelsTr_pre_3",
    target_spacing=(1.0, 1.0, 1.0),
    clip=(-1000, 400),
    percentile_norm=True,
    percentile_range=(0.5, 99.5),
    zscore=False,
    crop_mode="label",
    crop_margin=10,
)

cldice_utils.py for SoftCLDiceLoss and hard_cldice imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from skimage.morphology import skeletonize, skeletonize_3d


# Soft skeletonization (for soft-clDice loss)

def _soft_erode(x: torch.Tensor) -> torch.Tensor:
    """
    x: (B, C, D, H, W) or (B, C, H, W)
    """
    if x.ndim == 4:  # 2D: B,C,H,W
        p1 = -F.max_pool2d(-x, kernel_size=(3, 1), stride=1, padding=(1, 0))
        p2 = -F.max_pool2d(-x, kernel_size=(1, 3), stride=1, padding=(0, 1))
        return torch.min(p1, p2)
    elif x.ndim == 5:  # 3D: B,C,D,H,W
        p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
        p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
        p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
        return torch.min(torch.min(p1, p2), p3)
    else:
        raise ValueError(f"Expected 4D or 5D tensor, got shape {x.shape}")


def _soft_dilate(x: torch.Tensor) -> torch.Tensor:
    if x.ndim == 4:
        return F.max_pool2d(x, kernel_size=3, stride=1, padding=1)
    elif x.ndim == 5:
        return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)
    else:
        raise ValueError(f"Expected 4D or 5D tensor, got shape {x.shape}")


def _soft_open(x: torch.Tensor) -> torch.Tensor:
    return _soft_dilate(_soft_erode(x))


def soft_skeleton(x: torch.Tensor, iters: int) -> torch.Tensor:
    """
    Differentiable soft skeletonization from the clDice paper (Shit et al., CVPR 2021).
    x: probability map in [0,1], shape (B,1,...) or (B,C,...)
    """
    img = x
    img1 = _soft_open(img)
    skel = F.relu(img - img1)

    for _ in range(iters):
        img = _soft_erode(img)
        img1 = _soft_open(img)
        delta = F.relu(img - img1)
        skel = skel + F.relu(delta - skel * delta)

    return skel


class SoftCLDiceLoss(nn.Module):
    """
    Soft clDice loss (1 - clDice) as in Shit et al. (CVPR 2021).
    You should usually call this on a SINGLE-CHANNEL vessel mask (e.g. A∪V vs BG).
    """

    def __init__(self, iter_: int = 3, smooth: float = 1.0):
        super().__init__()
        self.iters = iter_
        self.smooth = float(smooth)

    def forward(self, y_true: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
        """
        y_true, y_pred: (B,1,D,H,W) or (B,1,H,W), values in [0,1]
        """
        if y_true.shape != y_pred.shape:
            raise ValueError(f"SoftCLDiceLoss: shape mismatch {y_true.shape} vs {y_pred.shape}")

        skel_pred = soft_skeleton(y_pred, self.iters)
        skel_true = soft_skeleton(y_true, self.iters)

        # Sum over spatial (and channel) dims
        dims = tuple(range(1, y_true.ndim))

        tprec = ( (skel_pred * y_true).sum(dim=dims) + self.smooth ) / (
                skel_pred.sum(dim=dims) + self.smooth
        )
        tsens = ( (skel_true * y_pred).sum(dim=dims) + self.smooth ) / (
                skel_true.sum(dim=dims) + self.smooth
        )

        cl_dice = 2.0 * tprec * tsens / (tprec + tsens + self.smooth)
        # Loss = 1 - clDice, averaged over batch
        return 1.0 - cl_dice.mean()


# Hard clDice metric (for eval/inference)

def hard_cldice(pred: np.ndarray, target: np.ndarray, eps: float = 1e-6) -> float:
    """
    Hard clDice metric using binary skeletonization.

    pred, target: boolean numpy arrays, shape (D,H,W) or (H,W)
    returns: scalar clDice in [0,1]
    """
    if pred.shape != target.shape:
        raise ValueError(f"hard_cldice: shape mismatch {pred.shape} vs {target.shape}")

    if pred.ndim == 3:
        skel_pred = skeletonize_3d(pred)
        skel_true = skeletonize_3d(target)
    elif pred.ndim == 2:
        skel_pred = skeletonize(pred)
        skel_true = skeletonize(target)
    else:
        raise ValueError(f"hard_cldice expects 2D or 3D arrays, got {pred.ndim}D")

    def _cl_score(v: np.ndarray, s: np.ndarray) -> float:
        denom = s.sum()
        if denom == 0:
            return 0.0
        return float((v & s).sum()) / float(denom)

    tprec = _cl_score(pred, skel_true)   # skeleton(gt) inside pred
    tsens = _cl_score(target, skel_pred) # skeleton(pred) inside gt

    if tprec + tsens < eps:
        return 0.0

    return float(2.0 * tprec * tsens / (tprec + tsens + eps))


data.py for generate_transforms import

In [ ]:
import logging

from monai import transforms
from monai.transforms import Compose

logger = logging.getLogger(__name__)


def generate_transforms(
    transforms_config: list[dict],
) -> list[transforms.Transform]:
    """
    Generate a list of transforms from a list of transform configurations.

    Args:
        transforms_config (list[dict]): List of transform configurations.

    Returns:
        list: List of transforms.
    """

    transform_list = []
    logger.debug(f"Generating {len(transforms_config)} transforms")

    for transform_config in transforms_config:
        transform_name = next(iter(transform_config))
        transform_kwargs = transform_config[transform_name]
        logger.debug(
            f"Generating transform {transform_name} with kwargs {transform_kwargs}"
        )
        transform: transforms.Transform = getattr(transforms, transform_name)(
            **transform_kwargs
        )  # type: ignore
        transform_list.append(transform)

    return Compose(transform_list)  # type: ignore


io.py for determine_reader_writer import

In [ ]:
"""Reader/writer classes; we follow https://github.com/MIC-DKFZ/nnUNet/tree/master/nnunetv2/imageio"""

import logging
import os
import json
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import SimpleITK as sitk

logger = logging.getLogger(__name__)


class BaseReaderWriter(ABC):
    supported_file_formats = []
    
    @staticmethod
    def _check_all_same(lst: List) -> bool:
        """
        Check if all elements in a list are the same
        Args:
            lst: List of elements
        Returns:
            Boolean indicating if all elements are the same
        """
        return all(x == lst[0] for x in lst)
    
    @abstractmethod
    def read_images(self, image_fnames: Union[List[str], Tuple[str, ...]]) -> Tuple[np.ndarray, dict]:
        """
        Read images from disk
        Args:
            image_fnames: List of image filenames
        Returns:
            Tuple of numpy array and dictionary
        """
        pass

    @abstractmethod
    def read_segs(self, seg_fnames: Union[List[str], Tuple[str, ...]]) -> Tuple[np.ndarray, dict]:
        """
        Read segmentations from disk
        Args:
            seg_fnames: List of segmentation filenames
        Returns:
            Tuple of numpy array and dictionary
        """
        pass

    @abstractmethod
    def write_seg(self, seg: np.ndarray, seg_fname: str, metadata: Optional[Dict[str, Any]] = None) -> None:
        """
        Write segmentation to disk
        Args:
            seg: Segmentation array
            seg_fname: Segmentation filename
            metadata: Metadata dictionary
        """
        pass


class NumpyReaderWriter(BaseReaderWriter):
    supported_file_formats = ["npy", "npz"]

    def __init__(self):
        super().__init__()

    def read_images(
        self, image_fnames: Union[str, list[str]], metdata_path: Optional[str] = None
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        """
        Read images from disk

        Args:
            image_fnames: List of image filenames
        Returns:
            Tuple of numpy array and dictionary
        """
        image_data = []
        if type(image_fnames) is str:
            image_fnames = [image_fnames]

        for image_fname in image_fnames:

            file_extension = os.path.basename(image_fname).split(".")[-1]
            if file_extension not in self.supported_file_formats:
                raise RuntimeError(f"File format not supported for {image_fname}")

            if file_extension == "npy":
                image_data.append(self._load_npy(image_fname))
                if image_data[-1].ndim != 3:
                    raise RuntimeError(
                        f"Image {image_fname} has dimension {image_data[-1].ndim}, expected 3"
                    )
            elif file_extension == "npz":
                new_images = self._load_npz(image_fname)
                for image in new_images:
                    if image.ndim != 3:
                        raise RuntimeError(
                            f"Image in {image_fname} has dimension {image.ndim}, expected 3"
                        )
                image_data.extend(new_images)

        if len(image_data) > 1:
            image_data = np.vstack(image_data)
        else:
            image_data = image_data[0]

        if metdata_path is not None:
            with open(metdata_path, "r") as f:
                metadata = json.load(f)
            spacing = metadata["spacing"]
            return image_data, {"spacing": spacing, "other": metadata}
        else:
            spacing = [1, 1, 1]
            return image_data, {"spacing": spacing}

    def _load_npy(self, fname: str) -> np.ndarray:
        return np.load(fname)

    def _load_npz(self, fname: str) -> list[np.ndarray]:
        file = np.load(fname)
        array = []
        for key in file.files:
            array.append(file[key])
        return array

    def read_segs(
        self, seg_fnames: Union[str, list[str]]
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        return self.read_images(seg_fnames)

    def write_seg(
        self,
        seg: np.ndarray,
        seg_fname: str,
        metadata: Optional[Dict[str, Any]] = None,
    ):
        file_extension = os.path.basename(seg_fname).split(".")[-1]
        if file_extension != ".npy":
            raise RuntimeError(
                f"File format {file_extension} not supported,  saving {seg_fname} failed!"
            )
        if metadata is not None:
            spacing = metadata["spacing"]
            with open(seg_fname, "wb") as f:
                np.save(f, seg)
            with open(seg_fname.replace(".npy", ".json"), "w") as f:
                json.dump(metadata, f)
        else:
            with open(seg_fname, "wb") as f:
                np.save(f, seg)


class NumpySeriesReaderWriter(BaseReaderWriter):
    supported_file_formats = ["npy_series", "npz_series"]
    slice_formats = ["npy", "npz"]

    def __init__(self):
        super().__init__()

    def read_images(
        self,
        image_folder: str,
        metdata_path: Optional[str] = None,
        start_idx: Optional[int] = None,
        end_idx: Optional[int] = None,
    ):
        """
        Read a series of images from disk given a the folder and optionally a start and end index.

        Args:
            image_folder: Folder containing the images
            start_idx: Start index of the images to read
            end_idx: End index of the images to read
        Returns:
            Tuple of numpy array and dictionary
        """
        image_files = os.listdir(image_folder)
        image_files = [
            f for f in image_files if f.endswith(".npy") or f.endswith(".npz")
        ]
        if start_idx is not None:
            image_files = [f for f in image_files if int(f.split(".")[0]) >= start_idx]
        if end_idx is not None:
            image_files = [f for f in image_files if int(f.split(".")[0]) < end_idx]
        image_files.sort()
        image_files = [os.path.join(image_folder, f) for f in image_files]
        image_data = []
        for image_fname in image_files:
            file_extension = os.path.basename(image_fname).split(".")[-1].lower()
            if file_extension not in self.slice_formats:
                raise RuntimeError(f"File format not supported for {image_fname}")
            if file_extension == "npy":
                image = self._load_npy(image_fname)
                image_data.append(image)
                if image_data[-1].ndim != 2:
                    raise RuntimeError(
                        f"Image {image_fname} has dimension {image_data[-1].ndim}, expected 2"
                    )
            elif file_extension == "npz":
                new_images = self._load_npz(image_fname)
                for image in new_images:
                    if image.ndim != 2:
                        raise RuntimeError(
                            f"Image in {image_fname} has dimension {image.ndim}, expected 2"
                        )
                image_data.extend(new_images)
        if metdata_path is not None:
            with open(metdata_path, "r") as f:
                metadata = json.load(f)
            return np.stack(image_data, axis=0), metadata
        else:
            spacing = [1, 1, 1]
            return np.stack(image_data, axis=0), {"spacing": spacing}

    def _load_npy(self, fname: str) -> np.ndarray:
        return np.load(fname)

    def _load_npz(self, fname: str) -> list[np.ndarray]:
        file = np.load(fname)
        array = []
        for key in file.files:
            array.append(file[key])
        return array

    def read_segs(
        self,
        seg_fnames: Union[str, list[str]],
        metadata: Optional[Dict[str, Any]] = None,
        start_idx: Optional[int] = None,
        end_idx: Optional[int] = None,
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        return self.read_images(seg_fnames, metadata, start_idx, end_idx)

    def _save_npy_series(self, array: np.ndarray, output_folder: str):
        os.makedirs(output_folder, exist_ok=True)
        for i in range(array.shape[0]):
            np.save(os.path.join(output_folder, f"{i}.npy"), array[i])

    def write_seg(
        self,
        seg: np.ndarray,
        seg_fname: str,
        metadata: Optional[Dict[str, Any]] = None,
    ):
        if metadata is not None:
            self._save_npy_series(seg, seg_fname)
            json.dump(metadata, os.path.join(seg_fname, "metadata.json"))
        else:
            self._save_npy_series(seg, seg_fname)


class SimpleITKReaderWriter(BaseReaderWriter):
    supported_file_formats = ["nii", "nii.gz", "mha", "mhd", "nrrd", "gz"]

    def __init__(self):
        super().__init__()

    def read_images(
        self, image_fnames: Union[str, list[str]]
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        """
        Read images from disk

        Args:
            image_fnames: List of image filenames
        Returns:
            Tuple of numpy array and dictionary
        """
        if type(image_fnames) is not list:
            image_fnames = [image_fnames]
        image_data = []
        image_metadata = {"spacing": [], "origin": [], "direction": []}
        for image_fname in image_fnames:

            if (
                os.path.basename(image_fname).split(".")[-1]
                not in self.supported_file_formats
            ):
                raise RuntimeError(f"File format not supported for {image_fname}")

            image = sitk.ReadImage(image_fname)
            logger.debug(f"Image {image_fname} has shape {image.GetSize()}")
            image_data.append(sitk.GetArrayFromImage(image))

            if image_data[-1].ndim != 3:
                raise RuntimeError(
                    f"Image {image_fname} has dimension {image_data[-1].ndim}, expected 3"
                )

            image_metadata["spacing"].append(image.GetSpacing())
            image_metadata["origin"].append(image.GetOrigin())
            image_metadata["direction"].append(image.GetDirection())

        if not self._check_all_same(image_metadata["spacing"]):
            logger.error("Spacing is not the same for all images")
            raise RuntimeError("Spacing is not the same for all images")
        if not self._check_all_same(image_metadata["origin"]):
            logger.warning("Origin is not the same for all images")
            logger.warning("Please check if this is expected behavior")
        if not self._check_all_same(image_metadata["direction"]):
            logger.warning("Direction is not the same for all images")
            logger.warning("Please check if this is expected behavior")

        sitk_metadata = {}
        for key in image_metadata.keys():
            sitk_metadata[key] = image_metadata[key][0]
        spacing = [
            sitk_metadata["spacing"][2],
            sitk_metadata["spacing"][0],
            sitk_metadata["spacing"][1],
        ]
        meta_data = {"spacing": spacing, "other": sitk_metadata}
        logger.debug(f"Spacing: {spacing}")
        logger.debug(f"Final shape: {np.vstack(image_data).shape}")
        return np.vstack(image_data), meta_data

    def read_segs(
        self, seg_fnames: Union[str, list[str]]
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        """
        Read segmentations from disk
        Args:
            seg_fnames: List of segmentation filenames
        Returns:
            Tuple of numpy array and dictionary
        """
        return self.read_images(seg_fnames)

    def write_seg(
        self,
        seg: np.ndarray,
        seg_fname: str,
        metadata: Optional[Dict[str, Any]] = None,
        compression: bool = True,
    ):
        """
        Write segmentation to disk
        Args:
            seg: Segmentation array
            seg_fname: Segmentation filename
            metadata: Metadata dictionary
        """
        seg = sitk.GetImageFromArray(seg)
        if metadata is not None:
            seg.SetSpacing(metadata["spacing"])
            seg.SetOrigin(metadata["origin"])
            seg.SetDirection(metadata["direction"])

        sitk.WriteImage(seg, seg_fname, compression)


def determine_reader_writer(file_ending: str):
    LIST_OF_READERS_WRITERS = [
        NumpyReaderWriter,
        SimpleITKReaderWriter,
        NumpySeriesReaderWriter,
    ]

    for reader_writer in LIST_OF_READERS_WRITERS:
        if file_ending.lower() in reader_writer.supported_file_formats:
            logger.debug(
                f"Automatically determined reader_writer: {reader_writer.__name__} for file ending: {file_ending}"
            )
            return reader_writer

    raise ValueError(f"No reader_writer found for file ending: {file_ending}")

evaluation.py for Evaluator and calculate_mean_metrics imports

In [ ]:
from pathlib import Path

import numpy as np
import torch
from skimage.morphology import skeletonize, skeletonize_3d
from skimage.measure import euler_number, label
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
import SimpleITK as sitk
from torch.utils.data import Dataset


class PretrainEvaluationDataset(Dataset):
    def __init__(self, data_path):
        data_dir = Path(data_path).resolve()
        self.val_data = {
            "deepvess": [
                torch.tensor(read_nifti(data_dir / "deepvess.nii"))[None],
                torch.tensor(read_nifti(data_dir / "deepvess_mask.nii"))[None],
            ],
            "deepvesselnet": [
                torch.tensor(read_nifti(data_dir / "deepvesselnet.nii"))[None],
                torch.tensor(read_nifti(data_dir / "deepvesselnet_mask.nii"))[None],
            ],
            "lightsheet": [
                torch.tensor(read_nifti(data_dir / "lightsheet.nii"))[None],
                torch.tensor(read_nifti(data_dir / "lightsheet_mask.nii"))[None],
            ],
            "minivess": [
                torch.tensor(read_nifti(data_dir / "minivess.nii"))[None],
                torch.tensor(read_nifti(data_dir / "minivess_mask.nii"))[None],
            ],
            "tubetk": [
                torch.tensor(read_nifti(data_dir / "tubetk.nii"))[None],
                torch.tensor(read_nifti(data_dir / "tubetk_mask.nii"))[None],
            ],
        }
        self._samples = list(self.val_data.keys())

    def __len__(self):
        return len(self._samples)

    def __getitem__(self, idx):
        name = self._samples[idx]
        image, mask = self.val_data[self._samples[idx]]
        return image, mask, name
        

class Evaluator:
    def extract_labels(self, gt_array, pred_array):
        """
        Adapted from https://github.com/CoWBenchmark/TopCoW_Eval_Metrics/blob/master/metric_functions.py#L18.
        """
        labels_gt = np.unique(gt_array)
        labels_pred = np.unique(pred_array)
        labels = list(set().union(labels_gt, labels_pred))
        labels = [int(x) for x in labels]
        return labels

    def betti_number_error(self, gt, pred):
        """
        Adapted from https://github.com/CoWBenchmark/TopCoW_Eval_Metrics/blob/master/metric_functions.py#L250.
        """
        labels = self.extract_labels(gt_array=gt, pred_array=pred)
        labels.remove(0)

        if len(labels) == 0:
            return 0, 0
        assert len(labels) == 1 and 1 in labels, "Invalid binary segmentatio.n"

        gt_betti_numbers = self.betti_number(gt)
        pred_betti_numbers = self.betti_number(pred)
        betti_0_error = abs(pred_betti_numbers[0] - gt_betti_numbers[0])
        betti_1_error = abs(pred_betti_numbers[1] - gt_betti_numbers[1])
        return betti_0_error, betti_1_error

    def betti_number(self, img):
        """
        Adapted from https://github.com/CoWBenchmark/TopCoW_Eval_Metrics/blob/master/metric_functions.py#L186.
        """
        assert img.ndim == 3
        N6 = 1
        N26 = 3

        padded = np.pad(img, pad_width=1)
        assert set(np.unique(padded)).issubset({0, 1})

        _, b0 = label(padded, return_num=True, connectivity=N26)
        euler_char_num = euler_number(padded, connectivity=N26)
        _, b2 = label(1 - padded, return_num=True, connectivity=N6)

        b2 -= 1
        b1 = b0 + b2 - euler_char_num
        return [b0, b1, b2]

    def cl_dice(self, v_p, v_l):
        """
        Adapted from https://github.com/jocpae/clDice/blob/master/cldice_metric/cldice.py.
        """
        def cl_score(v, s):
            return np.sum(v * s) / np.sum(s)

        if len(v_p.shape) == 2:
            tprec = cl_score(v_p, skeletonize(v_l))
            tsens = cl_score(v_l, skeletonize(v_p))
        elif len(v_p.shape) == 3:
            tprec = cl_score(v_p, skeletonize_3d(v_l))
            tsens = cl_score(v_l, skeletonize_3d(v_p))
        else:
            raise ValueError(f"Invalid shape for cl_dice: {v_p.shape}")
        return 2 * tprec * tsens / (tprec + tsens + np.finfo(float).eps)

    def estimate_metrics(self, pred_seg, gt_seg, threshold=0.5, fast=False):
        metrics = {}
        pred_seg_thresh = (pred_seg >= threshold).float().cpu()

        # estimate metrics
        tn, fp, fn, tp = confusion_matrix(
            gt_seg.flatten().cpu().clone().numpy(),
            pred_seg_thresh.flatten().cpu().clone().numpy(),
            labels=[0, 1],
        ).ravel()

        if fast:
            metrics["dice"] = (2 * tp) / (2 * tp + fp + fn)
            return metrics

        roc_auc = roc_auc_score(
            gt_seg.flatten().cpu().clone().detach().numpy(),
            pred_seg.flatten().cpu().clone().detach().numpy(),
        )

        pr_auc = average_precision_score(
            gt_seg.flatten().cpu().clone().detach().numpy(),
            pred_seg.flatten().cpu().clone().detach().numpy(),
        )

        cldice = self.cl_dice(
            pred_seg_thresh.squeeze().cpu().clone().detach().byte().numpy(),
            gt_seg.squeeze().cpu().clone().detach().byte().numpy(),
        )

        betti_0_error, betti_1_error = self.betti_number_error(
            gt_seg.squeeze().cpu().clone().detach().int().numpy(),
            pred_seg_thresh.squeeze().cpu().clone().detach().int().numpy(),
        )
        betti_0, betti_1, betti_2 = self.betti_number(
            pred_seg_thresh.squeeze().cpu().clone().detach().int().numpy()
        )

        metrics["recall_tpr_sensitivity"] = tp / (tp + fn)
        metrics["fpr"] = fp / (fp + tn)
        metrics["precision"] = tp / (tp + fp)
        metrics["specificity"] = tn / (tn + fp)
        metrics["jaccard_iou"] = tp / (tp + fp + fn)
        metrics["dice"] = (2 * tp) / (2 * tp + fp + fn)
        metrics["cldice"] = cldice
        metrics["accuracy"] = (tp + tn) / (tn + fp + tp + fn)
        metrics["roc_auc"] = roc_auc
        metrics["pr_auc_ap"] = pr_auc
        metrics["betti_0_error"] = betti_0_error
        metrics["betti_1_error"] = betti_1_error
        metrics["betti_0"] = betti_0
        metrics["betti_1"] = betti_1
        metrics["betti_2"] = betti_2
        return metrics


def read_nifti(path: str):
    return sitk.GetArrayFromImage(sitk.ReadImage(path))

def calculate_mean_metrics(results):
    """
    Compute mean over metrics.

    Accepts either:
      - dict: {case_id -> {metric_name -> value}}
      - list/tuple: [ {metric_name -> value}, ... ]
    Returns:
      dict: {metric_name -> mean_value}
    """
    import numpy as np

    # Handle dict input: {case_id: metrics_dict}
    if isinstance(results, dict):
        if not results:
            return {}
        results_list = list(results.values())
    else:
        # Assume it's an iterable of metrics dicts
        results_list = list(results)
        if not results_list:
            return {}

    mean_metrics = {}
    # Use keys from first case’s metrics
    for k in results_list[0].keys():
        vals = [float(r[k]) for r in results_list if k in r]
        if vals:
            mean_metrics[k] = float(np.mean(vals))

    return mean_metrics

inference.py for build model import

In [ ]:
""" Script to perform inference with vesselFM."""

import logging
import warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import hydra
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

import numpy as np
import json
import nibabel as nib

from tqdm import tqdm
from huggingface_hub import hf_hub_download
from monai.inferers import SlidingWindowInfererAdapt
from skimage.morphology import remove_small_objects
from skimage.exposure import equalize_hist
from scipy.ndimage import zoom as nd_zoom

from omegaconf import OmegaConf
from pathlib import Path


warnings.filterwarnings("ignore")
logger = logging.getLogger(__name__)

def build_model(num_classes=3, dropout=0.0):
    # Load inference config to get the same model definition with ckpt_path
    here = Path(__file__).resolve().parent
    config_dir = here / "configs"

    # Compose the full inference config
    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=str(config_dir), job_name="av_model"):
        cfg_inf = compose(config_name="inference")

    if "model" not in cfg_inf:
        raise ValueError(
            f"'model' key not found in composed config. Top-level keys: {list(cfg_inf.keys())}"
        )

    mcfg = cfg_inf.model

    # Set number of output channels to num_classes
    if "out_channels" in mcfg:
        logger.info(f"[build_model] Setting model.out_channels -> {num_classes}")
        mcfg.out_channels = num_classes
    elif "num_classes" in mcfg:
        logger.info(f"[build_model] Setting model.num_classes -> {num_classes}")
        mcfg.num_classes = num_classes
    else:
        logger.warning(
            "[build_model] Neither 'out_channels' nor 'num_classes' found in model config; "
            "leaving output channels as-is."
        )

    # Dropout override
    if "dropout" in mcfg:
        logger.info(f"[build_model] Setting model.dropout -> {dropout}")
        mcfg.dropout = dropout

    # Instantiate MONAI DynUNet
    model = hydra.utils.instantiate(cfg_inf.model)

    # Load pretrained VesselFM weights
    try:
        logger.info(f"[build_model] Loading pretrained weights from {cfg_inf.ckpt_path}.")
        ckpt = torch.load(Path(cfg_inf.ckpt_path), map_location="cpu", weights_only=True)
    except Exception as e:
        logger.info(
            f"[build_model] Could not load ckpt from cfg_inf.ckpt_path ({e}). "
            "Falling back to Hugging Face vesselFM_base.pt."
        )
        hf_hub_download(repo_id="bwittmann/vesselFM", filename="meta.yaml")
        ckpt = torch.load(
            hf_hub_download(repo_id="bwittmann/vesselFM", filename="vesselFM_base.pt"),
            map_location="cpu",
            weights_only=True,
        )

    # Drop old head weights (1-channel) to avoid size mismatch
    head_keys = [k for k in ckpt.keys() if k.startswith("output_block.")]
    if head_keys:
        logger.info(
            f"[build_model] Removing {len(head_keys)} head params from checkpoint "
            f"to accommodate new 3-class head: {head_keys}"
        )
        for k in head_keys:
            ckpt.pop(k)

    # Now load backbone weights (strict=False allows missing head params)
    missing, unexpected = model.load_state_dict(ckpt, strict=False)
    logger.info(
        f"[build_model] Loaded pretrained VesselFM weights with "
        f"{len(missing)} missing and {len(unexpected)} unexpected keys "
        f"(expected when swapping to a 3-class head)."
    )

    # Add an explicit vessel head as a second physical head.
    if not hasattr(model, "vessel_head"):
        logger.info("[build_model] Adding 1x1x1 vessel_head on top of A/V logits.")
        model.vessel_head = nn.Conv3d(num_classes, 1, kernel_size=1)

    return model


def load_model(cfg, device):
    """
    Load the final A/V model (including Stage-3 av_refine_head) from ckpt_path.
    Assumes ckpt_path points to the av_ct_best_cldice.pt written by train_av.py.
    """
    # Load checkpoint
    try:
        logger.info(f"Loading model from {cfg.ckpt_path}.")
        ckpt = torch.load(Path(cfg.ckpt_path), map_location=device, weights_only=True)
    except Exception as e:
        logger.info(
            f"Could not load {cfg.ckpt_path} ({e}). "
            "Falling back to Hugging Face vesselFM_base.pt."
        )
        hf_hub_download(repo_id="bwittmann/vesselFM", filename="meta.yaml")
        ckpt = torch.load(
            hf_hub_download(repo_id="bwittmann/vesselFM", filename="vesselFM_base.pt"),
            map_location=device,
            weights_only=True,
        )

    # nstantiate the same backbone as in training
    model = hydra.utils.instantiate(cfg.model)

    # Figure out how many output channels the backbone has (should be 3)
    if "out_channels" in cfg.model:
        out_ch = cfg.model.out_channels
    elif "num_classes" in cfg.model:
        out_ch = cfg.model.num_classes
    else:
        out_ch = 3  # sensible default for your A/V/BG setup

    # Attach heads exactly like in train_av.py
    # Vessel head (not strictly needed for inference right now, but harmless)
    if not hasattr(model, "vessel_head"):
        logger.info("[load_model] Adding vessel_head for union-of-vessels output.")
        model.vessel_head = nn.Conv3d(out_ch, 1, kernel_size=1)

    # AV refine head: Stage-3 2-class A/V classifier on top of logits
    if not hasattr(model, "av_refine_head"):
        logger.info("[load_model] Adding av_refine_head (A/V refine) on top of logits.")
        model.av_refine_head = nn.Conv3d(out_ch, 2, kernel_size=1)

    # Load weights into this full architecture
    if isinstance(ckpt, dict):
        # Works for both raw state_dict and {'state_dict': ...}
        state = ckpt.get("state_dict", ckpt)
    else:
        state = ckpt

    missing, unexpected = model.load_state_dict(state, strict=False)
    logger.info(
        f"[load_model] Loaded checkpoint with {len(missing)} missing and "
        f"{len(unexpected)} unexpected keys."
    )

    return model


def get_paths(cfg):
    """
    Collect image and mask paths.

    Supports config layouts:
      - cfg.image_dir / cfg.mask_dir
      - cfg.image_path / cfg.mask_path
      - cfg.data.image_dir / cfg.data.mask_dir
      - cfg.data.image_path / cfg.data.mask_path

    Supports filename conventions:
      1) Same-name masks:
           image_004.nii.gz -> image_004.nii.gz
      2) image/label naming:
           image_004.nii.gz -> label_004.nii.gz
    """
    import os

    # Read directories from config with fallbacks
    image_dir_str = (
        OmegaConf.select(cfg, "data.image_dir")
        or OmegaConf.select(cfg, "image_dir")
        or OmegaConf.select(cfg, "data.image_path")
        or OmegaConf.select(cfg, "image_path")
    )

    mask_dir_str = (
        OmegaConf.select(cfg, "data.mask_dir")
        or OmegaConf.select(cfg, "mask_dir")
        or OmegaConf.select(cfg, "data.mask_path")
        or OmegaConf.select(cfg, "mask_path")
    )

    if image_dir_str is None:
        raise RuntimeError(
            "image directory not set in config "
            "(looked for 'image_dir', 'data.image_dir', "
            "'image_path', and 'data.image_path')."
        )

    image_dir = Path(image_dir_str)

    # Normalize mask_dir: either a Path or None
    if mask_dir_str is None or mask_dir_str in ("", "null"):
        mask_dir = None
    else:
        mask_dir = Path(mask_dir_str)

    # Collect images as Path objects
    # Use *.nii* so it works for .nii and .nii.gz
    image_paths = sorted(image_dir.glob("*.nii*"))
    if not image_paths:
        raise RuntimeError(f"No images found in {image_dir}")

    # If no mask_dir (pure inference), just return images
    if mask_dir is None:
        return image_paths, None

    # --- 3. Build mask paths with both naming schemes (also as Path objects) ---
    mask_paths = []

    for img_path in image_paths:
        # img_path is a Path
        img_name = img_path.name  # e.g. "image_004.nii.gz"

        # First try: mask has EXACT same basename as image
        same_name_mask = mask_dir / img_name
        if same_name_mask.exists():
            mask_paths.append(same_name_mask)
            continue

        # Second try: image_XXX.nii.gz -> label_XXX.nii.gz
        alt_mask = None
        if img_name.startswith("image_"):
            suffix = img_name[len("image_"):]          # "004.nii.gz"
            alt_mask = mask_dir / f"label_{suffix}"
            if alt_mask.exists():
                mask_paths.append(alt_mask)
                continue

        # No matching mask was found for this image
        msg = (
            f"Could not find a mask for image:\n  {img_path}\n"
            f"Tried:\n  {same_name_mask}"
        )
        if alt_mask is not None:
            msg += f"\n  {alt_mask}"
        raise FileNotFoundError(msg)

    return image_paths, mask_paths



def resample(image, factor=None, target_shape=None):
    if factor == 1:
        return image
    
    if target_shape:
        _, _, new_d, new_h, new_w = target_shape
    else:
        _, _, d, h, w = image.shape
        new_d, new_h, new_w = int(round(d / factor)), int(round(h / factor)), int(round(w / factor))
    return F.interpolate(image, size=(new_d, new_h, new_w), mode="trilinear", align_corners=False)


def resample_mask_to_spacing(mask, src_spacing, tgt_spacing, order=0):
    """
    Resample a 3D mask from src_spacing to tgt_spacing using nearest-neighbor (order=0).

    mask: (X, Y, Z) np.ndarray (integer labels)
    src_spacing, tgt_spacing: iterables of length 3 (sx, sy, sz)
    """
    src_spacing = np.array(src_spacing, dtype=np.float32)
    tgt_spacing = np.array(tgt_spacing, dtype=np.float32)
    zoom_factors = src_spacing / tgt_spacing  # Same convention as preprocess_av_ct_nii
    return nd_zoom(mask, zoom_factors, order=order)


@hydra.main(config_path="configs", config_name="inference", version_base="1.3.2")
def main(cfg):
    # seed libraries
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    torch.cuda.manual_seed_all(cfg.seed)

    # set device
    logger.info(f"Using device {cfg.device}.")
    device = cfg.device

    # load model and ckpt
    model = load_model(cfg, device)
    model.to(device)
    model.eval()

    # init pre-processing transforms
    transforms = generate_transforms(cfg.transforms_config)

    # i/o
    output_folder = Path(cfg.output_folder)
    output_folder.mkdir(exist_ok=True)

    image_paths, mask_paths = get_paths(cfg)
    logger.info(f"Found {len(image_paths)} images in {cfg.image_path}.")

    file_ending = (cfg.image_file_ending if cfg.image_file_ending else image_paths[0].suffix)
    image_reader_writer = determine_reader_writer(file_ending)()
    save_writer = determine_reader_writer(file_ending)()

    # init sliding window inferer
    logger.debug(f"Sliding window patch size: {cfg.patch_size}")
    logger.debug(f"Sliding window batch size: {cfg.batch_size}.")
    logger.debug(f"Sliding window overlap: {cfg.overlap}.")
    inferer = SlidingWindowInfererAdapt(
        roi_size=cfg.patch_size, sw_batch_size=cfg.batch_size, overlap=cfg.overlap, 
        mode=cfg.mode, sigma_scale=cfg.sigma_scale, padding_mode=cfg.padding_mode
    )

    # loop over images
    metrics_dict = {}
    with torch.no_grad():
        for idx, image_path in tqdm(
            enumerate(image_paths),
            total=len(image_paths),
            desc="Processing images.",
        ):
            preds = []  # per-scale logits (kept on device)
            mask_np = None

            for scale in cfg.tta.scales:
                # read image (and mask if available)
                image_np = image_reader_writer.read_images(image_path)[0].astype(np.float32)
                image = transforms(image_np)[None].to(device)  # (1,1,D,H,W) on device

                if mask_paths is not None and mask_np is None:
                    # Load 3-class GT: 0=bg,1=artery,2=vein, keep on CPU
                    mask_np = image_reader_writer.read_images(mask_paths[idx])[0].astype(np.int16)

                # TTA intensity transforms
                if cfg.tta.invert:
                    if image.mean() > cfg.tta.invert_mean_thresh:
                        image = 1 - image
                if cfg.tta.equalize_hist:
                    image_np = image.cpu().squeeze().numpy()
                    image_equal_hist_np = equalize_hist(image_np, nbins=cfg.tta.hist_bins)
                    image = torch.from_numpy(image_equal_hist_np).to(device)[None][None]

                # resample for scale, run model, resample back
                original_shape = image.shape
                image_scaled = resample(image, factor=scale)          # on device
                logits = inferer(image_scaled, model)                 # (1,3,D,H,W) on device
                logits = resample(logits, target_shape=original_shape)
                preds.append(logits.squeeze(0))                       # (3,D,H,W) on device

            # Preds is a list of per-scale logits, each (3,D,H,W) on device
            logits_ensemble = torch.stack(preds).mean(dim=0)          # (3,D,H,W) on device

            if hasattr(model, "av_refine_head"):
                # Stage-3 A/V refinement (same as eval_epoch(use_av_refine=True))
                base_probs = F.softmax(logits_ensemble.unsqueeze(0), dim=1)  # (1,3,D,H,W)
                p_bg = base_probs[:, 0:1, ...]
                p_union = base_probs[:, 1:3, ...].sum(dim=1, keepdim=True).clamp(0.0, 1.0)

                av_logits = model.av_refine_head(logits_ensemble.unsqueeze(0))  # (1,2,D,H,W)
                av_probs = F.softmax(av_logits, dim=1)
                p_art_cond = av_probs[:, 0:1, ...]
                p_vein_cond = av_probs[:, 1:2, ...]

                p_art = p_union * p_art_cond
                p_vein = p_union * p_vein_cond

                denom = p_bg + p_art + p_vein + 1e-8
                probs_final = torch.cat(
                    [p_bg / denom, p_art / denom, p_vein / denom],
                    dim=1,
                )[0]  # (3,D,H,W) on device
            else:
                # Fallback: original vesselFM A/V logits fusion
                if cfg.merging.max:
                    probs_final = torch.stack([F.softmax(p, dim=0) for p in preds]).max(dim=0)[0]
                else:
                    probs_final = torch.stack([F.softmax(p, dim=0) for p in preds]).mean(dim=0)

            # Move to CPU only when converting to numpy for saving / metrics
            label = probs_final.argmax(0).cpu().numpy().astype(np.uint8)

            # Class-wise CC cleanup
            if cfg.post.apply:
                cleaned = np.zeros_like(label, dtype=np.uint8)
                for c in (1, 2):  # artery, vein
                    cm = (label == c)
                    cm = remove_small_objects(
                        cm,
                        min_size=cfg.post.small_objects_min_size,
                        connectivity=cfg.post.small_objects_connectivity,
                    )
                    cleaned[cm] = c
                label = cleaned

            # Label is a numpy array (D, H, W) in model/reader order
            label_np = label.astype(np.uint8)

            # Geometry handling
            # Preprocessed reference (the volume actually fed into the model)
            pre_nii = nib.load(str(image_path))
            pre_shape = pre_nii.shape
            pre_spacing = pre_nii.header.get_zooms()[:3]

            # RAW reference: same filename but in cfg.raw_image_dir
            raw_image_dir = getattr(cfg, "raw_image_dir", None)
            raw_nii = None
            label_for_save = label_np  # default: save in preprocessed space
            ref_nii = pre_nii          # default reference

            if raw_image_dir not in (None, "", "null"):
                raw_dir = Path(raw_image_dir)
                raw_path = raw_dir / image_path.name

                if raw_path.exists():
                    raw_nii = nib.load(str(raw_path))
                    raw_spacing = raw_nii.header.get_zooms()[:3]

                    # Resample mask from pre spacing -> raw spacing
                    mask_raw = resample_mask_to_spacing(
                        label_np,
                        src_spacing=pre_spacing,
                        tgt_spacing=raw_spacing,
                        order=0,  # nearest neighbor for labels
                    ).astype(np.uint8)

                    # If small size mismatch (rounding), crop/pad to raw_nii.shape
                    if mask_raw.shape != raw_nii.shape:
                        logger.warning(
                            f"Resampled mask shape {mask_raw.shape} != raw image shape {raw_nii.shape}; "
                            "cropping/padding to match."
                        )
                        out = np.zeros(raw_nii.shape, dtype=np.uint8)
                        common = tuple(min(a, b) for a, b in zip(mask_raw.shape, out.shape))
                        slices_out = tuple(slice(0, c) for c in common)
                        slices_in = tuple(slice(0, c) for c in common)
                        out[slices_out] = mask_raw[slices_in]
                        mask_raw = out

                    label_for_save = mask_raw
                    ref_nii = raw_nii
                else:
                    logger.warning(
                        f"Raw image not found at {raw_path}; "
                        "saving prediction in preprocessed space instead."
                    )
                    ref_nii = pre_nii
                    label_for_save = label_np
            else:
                # No raw_image_dir provided: save in preprocessed space,
                if label_np.shape != pre_shape:
                    # Common case: (D,H,W) vs (X,Y,Z) where only axis 0 and 2 differ
                    if (
                        label_np.shape[0] == pre_shape[2]
                        and label_np.shape[1] == pre_shape[1]
                        and label_np.shape[2] == pre_shape[0]
                    ):
                        label_np = np.transpose(label_np, (2, 1, 0))
                        logger.info(
                            f"Transposed prediction to match preprocessed shape {pre_shape}."
                        )
                    else:
                        raise RuntimeError(
                            f"Predicted mask shape {label_np.shape} does not match preprocessed image shape "
                            f"{pre_shape} and is not a simple 0<->2 axis swap."
                        )
                label_for_save = label_np
                ref_nii = pre_nii

            # Save prediction in ref_nii space (raw if available, otherwise preprocessed)
            pred_nii = nib.Nifti1Image(
                label_for_save,
                affine=ref_nii.affine,
                header=ref_nii.header,
            )

            # Keep sform/qform consistent with reference image
            pred_nii.set_sform(
                ref_nii.get_sform(),
                code=ref_nii.get_sform(coded=True)[1] or 1
            )
            pred_nii.set_qform(
                ref_nii.get_qform(),
                code=ref_nii.get_qform(coded=True)[1] or 1
            )

            out_path = output_folder / f"{image_path.name.split('.')[0]}_{cfg.file_app}pred.nii.gz"
            nib.save(pred_nii, str(out_path))

            # Metrics if GT masks are available
            if mask_paths is not None and mask_np is not None:
                """
                label:   (D, H, W) with {0:bg, 1:artery, 2:vein}
                mask_np: (D, H, W) with {0:bg, 1:artery, 2:vein}

                We report:
                  - Dice / clDice for union, artery, vein
                  - FP% and FN%:
                      * FP%_union_pred:   FP / (TP + FP)   for union (A∪V)
                      * FN%_union_gt:     FN / (TP + FN)   for union (A∪V)
                    and analogous metrics for artery and vein.
                """

                # -------------------------
                # UNION (A ∪ V) as binary
                # -------------------------
                union_pred = label > 0    # (D,H,W) bool
                union_gt   = mask_np > 0  # (D,H,W) bool

                tp_u = np.logical_and(union_pred, union_gt).sum()
                fp_u = np.logical_and(union_pred, np.logical_not(union_gt)).sum()
                fn_u = np.logical_and(np.logical_not(union_pred), union_gt).sum()
                tn_u = np.logical_and(np.logical_not(union_pred), np.logical_not(union_gt)).sum()

                denom_u = union_pred.sum() + union_gt.sum()
                dice_union = 2.0 * tp_u / (denom_u + 1e-5) if denom_u > 0 else 0.0
                cldice_union = hard_cldice(union_pred.astype(bool), union_gt.astype(bool))

                pred_pos_u = tp_u + fp_u
                gt_pos_u   = tp_u + fn_u

                fp_pct_union_pred = 100.0 * fp_u / (pred_pos_u + 1e-5) if pred_pos_u > 0 else 0.0
                fn_pct_union_gt   = 100.0 * fn_u / (gt_pos_u + 1e-5)   if gt_pos_u > 0 else 0.0

                # -------------------------
                # ARTERY (class = 1)
                # -------------------------
                g_art = (mask_np == 1)
                if g_art.any():
                    p_art = (label == 1)

                    tp_a = np.logical_and(p_art, g_art).sum()
                    fp_a = np.logical_and(p_art, np.logical_not(g_art)).sum()
                    fn_a = np.logical_and(np.logical_not(p_art), g_art).sum()

                    denom_a = p_art.sum() + g_art.sum()
                    dice_art = 2.0 * tp_a / (denom_a + 1e-5) if denom_a > 0 else 0.0
                    cldice_art = hard_cldice(p_art.astype(bool), g_art.astype(bool))

                    pred_pos_a = tp_a + fp_a
                    gt_pos_a   = tp_a + fn_a

                    fp_pct_art_pred = 100.0 * fp_a / (pred_pos_a + 1e-5) if pred_pos_a > 0 else 0.0
                    fn_pct_art_gt   = 100.0 * fn_a / (gt_pos_a + 1e-5)   if gt_pos_a > 0 else 0.0
                else:
                    dice_art = 0.0
                    cldice_art = 0.0
                    fp_pct_art_pred = 0.0
                    fn_pct_art_gt = 0.0

                # -------------------------
                # VEIN (class = 2)
                # -------------------------
                g_vein = (mask_np == 2)
                if g_vein.any():
                    p_vein = (label == 2)

                    tp_v = np.logical_and(p_vein, g_vein).sum()
                    fp_v = np.logical_and(p_vein, np.logical_not(g_vein)).sum()
                    fn_v = np.logical_and(np.logical_not(p_vein), g_vein).sum()

                    denom_v = p_vein.sum() + g_vein.sum()
                    dice_vein = 2.0 * tp_v / (denom_v + 1e-5) if denom_v > 0 else 0.0
                    cldice_vein = hard_cldice(p_vein.astype(bool), g_vein.astype(bool))

                    pred_pos_v = tp_v + fp_v
                    gt_pos_v   = tp_v + fn_v

                    fp_pct_vein_pred = 100.0 * fp_v / (pred_pos_v + 1e-5) if pred_pos_v > 0 else 0.0
                    fn_pct_vein_gt   = 100.0 * fn_v / (gt_pos_v + 1e-5)   if gt_pos_v > 0 else 0.0
                else:
                    dice_vein = 0.0
                    cldice_vein = 0.0
                    fp_pct_vein_pred = 0.0
                    fn_pct_vein_gt = 0.0

                case_name = image_path.name.split(".")[0]

                logger.info(
                    f"{case_name}: "
                    f"Dice(A∪V)={dice_union:.4f} clDice(A∪V)={cldice_union:.4f} "
                    f"Dice(art)={dice_art:.4f} clDice(art)={cldice_art:.4f} "
                    f"Dice(vein)={dice_vein:.4f} clDice(vein)={cldice_vein:.4f} | "
                    f"FP%(A∪V|pred)={fp_pct_union_pred:.2f} FN%(A∪V|gt)={fn_pct_union_gt:.2f} | "
                    f"FP%(art|pred)={fp_pct_art_pred:.2f} FN%(art|gt)={fn_pct_art_gt:.2f} | "
                    f"FP%(vein|pred)={fp_pct_vein_pred:.2f} FN%(vein|gt)={fn_pct_vein_gt:.2f}"
                )

                # Store metrics for JSON + summary
                metrics_dict[case_name] = {
                    # union
                    "dice":               torch.tensor(dice_union),
                    "cldice":             torch.tensor(cldice_union),
                    "fp_pct_union_pred":  torch.tensor(fp_pct_union_pred),
                    "fn_pct_union_gt":    torch.tensor(fn_pct_union_gt),
                    # artery
                    "dice_art":           torch.tensor(dice_art),
                    "cldice_art":         torch.tensor(cldice_art),
                    "fp_pct_art_pred":    torch.tensor(fp_pct_art_pred),
                    "fn_pct_art_gt":      torch.tensor(fn_pct_art_gt),
                    # vein
                    "dice_vein":          torch.tensor(dice_vein),
                    "cldice_vein":        torch.tensor(cldice_vein),
                    "fp_pct_vein_pred":   torch.tensor(fp_pct_vein_pred),
                    "fn_pct_vein_gt":     torch.tensor(fn_pct_vein_gt),
                }

    # Summarize over all images
    if mask_paths is not None and len(metrics_dict) > 0:
        # Compute mean for every metric key we stored
        metric_names = list(next(iter(metrics_dict.values())).keys())
        mean_metrics = {}
        for m in metric_names:
            vals = [metrics_dict[k][m].item() for k in metrics_dict]
            mean_metrics[m] = float(np.mean(vals))

        logger.info(f"Mean Dice(A∪V): {mean_metrics['dice']:.4f}")
        logger.info(f"Mean clDice(A∪V): {mean_metrics['cldice']:.4f}")
        logger.info(f"Mean Dice(art): {mean_metrics['dice_art']:.4f}")
        logger.info(f"Mean clDice(art): {mean_metrics['cldice_art']:.4f}")
        logger.info(f"Mean Dice(vein): {mean_metrics['dice_vein']:.4f}")
        logger.info(f"Mean clDice(vein): {mean_metrics['cldice_vein']:.4f}")

        logger.info(f"Mean FP%(A∪V|pred): {mean_metrics['fp_pct_union_pred']:.2f}")
        logger.info(f"Mean FN%(A∪V|gt):   {mean_metrics['fn_pct_union_gt']:.2f}")
        logger.info(f"Mean FP%(art|pred): {mean_metrics['fp_pct_art_pred']:.2f}")
        logger.info(f"Mean FN%(art|gt):   {mean_metrics['fn_pct_art_gt']:.2f}")
        logger.info(f"Mean FP%(vein|pred): {mean_metrics['fp_pct_vein_pred']:.2f}")
        logger.info(f"Mean FN%(vein|gt):   {mean_metrics['fn_pct_vein_gt']:.2f}")

        with open(output_folder / "metrics_per_volume.json", "w") as f:
            json.dump(
                {k: {m: float(v[m].item()) for m in v} for k, v in metrics_dict.items()},
                f,
                indent=2,
            )

        with open(output_folder / "metrics_mean.json", "w") as f:
            json.dump(mean_metrics, f, indent=2)

if __name__ == "__main__":
    main()

dataio.py for NiftiVolume and make_aug_transforms imports

In [ ]:
import numpy as np
import nibabel as nib
import torch
from torch.utils.data import Dataset
from scipy.ndimage import rotate as ndi_rotate, zoom as ndi_zoom


def _center_crop_or_pad(arr, target_shape):
    """
    Center-crop or pad a 3D array to target_shape = (D,H,W).
    Pads with zeros if arr is smaller; crops centrally if larger.
    """
    out = np.zeros(target_shape, dtype=arr.dtype)

    in_shape = arr.shape
    in_slices = []
    out_slices = []

    for in_size, out_size in zip(in_shape, target_shape):
        if in_size >= out_size:
            # crop in the center
            start_in = (in_size - out_size) // 2
            end_in = start_in + out_size
            start_out = 0
            end_out = out_size
        else:
            # pad in the center
            start_in = 0
            end_in = in_size
            start_out = (out_size - in_size) // 2
            end_out = start_out + in_size

        in_slices.append(slice(start_in, end_in))
        out_slices.append(slice(start_out, end_out))

    out[tuple(out_slices)] = arr[tuple(in_slices)]
    return out


def rotate_3d(image, label, angle_deg, axis="z"):
    """
    Small 3D rotation around one axis.
    image, label: (D,H,W)
    axis: 'x', 'y', or 'z' (CT-wise, 'z' = axial plane rotation).
    """
    if axis == "z":
        axes = (1, 2)  # rotate in (H,W)
    elif axis == "y":
        axes = (0, 2)
    elif axis == "x":
        axes = (0, 1)
    else:
        raise ValueError(f"Unknown axis {axis}, expected 'x','y','z'.")

    img_rot = ndi_rotate(
        image, angle_deg, axes=axes, reshape=False,
        order=1, mode="nearest"
    )
    lab_rot = ndi_rotate(
        label, angle_deg, axes=axes, reshape=False,
        order=0, mode="nearest"
    ).astype(label.dtype)
    return img_rot, lab_rot


def zoom_3d(image, label, zoom_factor):
    """
    Isotropic zoom in 3D, then center-crop / pad back to original shape.
    """
    orig_shape = image.shape

    img_zoom = ndi_zoom(image, zoom_factor, order=1)
    lab_zoom = ndi_zoom(label, zoom_factor, order=0).astype(label.dtype)

    img_zoom = _center_crop_or_pad(img_zoom, orig_shape)
    lab_zoom = _center_crop_or_pad(lab_zoom, orig_shape)

    return img_zoom, lab_zoom


class NiftiVolume(Dataset):
    """
    Simple NIfTI dataset that:
    - loads image/label volumes from disk,
    - applies basic CT preprocessing (HU clipping + scaling),
    - samples 3D patches for training,
    - returns tensors in MONAI / DynUNet-friendly format:
        image: (1, D, H, W), float32
        label: (D, H, W),     long (class indices 0,1,2)
    """

    def __init__(self, items, cfg, train: bool = True):
        """
        Args:
            items: list of (image_path, label_path) tuples.
            cfg:   the full av_ct.yaml config (dict-like).
            train: True for training, False for validation.
        """
        self.items = items
        self.cfg = cfg
        self.train = train

        data_cfg = cfg.get("data", {})

        # Patch sampling / preprocessing hyperparams (from av_ct.yaml)
        self.patch_size = tuple(data_cfg.get("patch_size", [96, 96, 96]))
        self.samples_per_volume = int(data_cfg.get("samples_per_volume", 4))
        self.min_fg_fraction = float(data_cfg.get("min_fg_fraction", 0.0))

        # A/V-specific sampling hyperparams
        self.artery_prob = float(data_cfg.get("artery_patch_prob", 0.0))
        self.vein_prob   = float(data_cfg.get("vein_patch_prob", 0.0))
        self.mixed_prob  = float(data_cfg.get("mixed_av_patch_prob", 0.0))
        self.min_artery_fraction = float(data_cfg.get("min_artery_fraction", 0.0))
        self.min_vein_fraction   = float(data_cfg.get("min_vein_fraction", 0.0))


        # Flag to indicate images are already preprocessed by preprocess_av.py
        self.preprocessed = bool(data_cfg.get("preprocessed", False))

        if self.preprocessed:
            # Images are already resampled + HU-clipped + scaled to [0,1],
            # Do NOT apply any further intensity preprocessing here.
            self.clip_hu = None
            self.zscore = False
        else:
            # Fallback: keep old online intensity preprocessing behavior
            self.clip_hu = data_cfg.get("clip_hu", None)   # e.g. [-1000, 600]
            self.zscore = bool(data_cfg.get("zscore", False))

        # For training, we define length as (#volumes * samples_per_volume)
        # so each epoch sees multiple patches per volume.
        if self.train:
            self.length = len(self.items) * self.samples_per_volume
        else:
            # For validation, one sample per volume (still patch-based).
            self.length = len(self.items)

        # Simple in-memory cache so we don't reload NIfTI every time
        self._cache = {}  # img_path -> (image_array, label_array)

        # Optional transform hook (kept to match train_av.py API)
        self._transform = None

        # Augmentation hyperparameters from config av_ct.yaml
        aug_cfg = cfg.get("augment", {})

        # Flip probability per axis
        self.flip_prob = float(aug_cfg.get("flip_prob", 0.5))

        # Small-angle rotation & zoom probabilities
        self.rotate_prob = float(aug_cfg.get("rotate_prob", 0.3))
        self.zoom_prob = float(aug_cfg.get("zoom_prob", 0.3))

        # Gamma augmentation
        self.gamma_range = aug_cfg.get("gamma", None)

        # Gaussian noise std (on image only, after all geometric augs)
        self.noise_std = float(aug_cfg.get("noise_std", 0.0))


    def set_transform(self, transform):
        """Keep a hook if you later want to plug extra transforms."""
        self._transform = transform

    def __len__(self):
        return self.length

    # Helpers

    def _load_image_label(self, vol_idx):
        img_path, lab_path = self.items[vol_idx]

        if img_path in self._cache:
            return self._cache[img_path]

        img_nii = nib.load(img_path)
        lab_nii = nib.load(lab_path)

        image = img_nii.get_fdata().astype(np.float32)
        label = lab_nii.get_fdata().astype(np.int16)

        # Intensity preprocessing: HU clipping + scaling to [0,1]
        if self.clip_hu is not None:
            lo, hi = float(self.clip_hu[0]), float(self.clip_hu[1])
            image = np.clip(image, lo, hi)
            image = (image - lo) / (hi - lo + 1e-8)

        # Optional z-score after HU scaling (if enabled)
        if self.zscore:
            m = image.mean()
            s = image.std()
            if s > 0:
                image = (image - m) / s

        self._cache[img_path] = (image, label)
        return image, label

    def _sample_patch(self, image, label):
        D, H, W = image.shape
        pD, pH, pW = self.patch_size
        min_fg      = self.min_fg_fraction
        art_prob    = self.artery_prob
        mixed_prob  = self.mixed_prob
        min_art     = self.min_artery_fraction
        min_vein = self.min_vein_fraction
        max_tries   = 32

        # Never request a patch bigger than the volume
        pD = min(pD, D); pH = min(pH, H); pW = min(pW, W)

        # Turn (artery_prob, vein_prob, mixed_prob) into a proper distribution
        probs = np.array(
            [self.artery_prob, self.vein_prob, self.mixed_prob],
            dtype=float,
        )
        if probs.sum() > 0:
            probs = probs / probs.sum()
            modes = ["art", "vein", "mix"]
        else:
            # No special A/V preference, just foreground-biased
            probs = None
            modes = ["fg"]

        fallback_img = None
        fallback_lab = None

        for _ in range(max_tries):
            # Random crop coords
            z = np.random.randint(0, max(1, D - pD + 1))
            y = np.random.randint(0, max(1, H - pH + 1))
            x = np.random.randint(0, max(1, W - pW + 1))

            img_patch = image[z:z+pD, y:y+pH, x:x+pW]
            lab_patch = label[z:z+pD, y:y+pH, x:x+pW]

            if fallback_img is None:
                fallback_img, fallback_lab = img_patch, lab_patch

            # Decide what type of patch we want
            if probs is not None:
                mode = np.random.choice(modes, p=probs)
            else:
                mode = "fg"

            fg_fraction   = (lab_patch > 0).mean()
            art_fraction  = (lab_patch == 1).mean()
            vein_fraction = (lab_patch == 2).mean()

            if mode == "mix":
                # want both artery and vein
                if art_fraction >= min_art and vein_fraction >= min_vein:
                    return img_patch, lab_patch
            elif mode == "art":
                if art_fraction >= min_art:
                    return img_patch, lab_patch
            elif mode == "vein":
                # reuse min_art as min_vein_fraction; you can split later if needed
                if vein_fraction >= min_vein:
                    return img_patch, lab_patch
            else:  # "fg" (fallback foreground-biased)
                if fg_fraction >= min_fg:
                    return img_patch, lab_patch

        # Fallback if constraints not met
        return fallback_img, fallback_lab



    def _augment(self, image, label):
        """Lightweight spatial + intensity augmentation."""
        if not self.train:
            return image, label

        # Random flips along each axis
        if np.random.rand() < self.flip_prob:
            image = image[::-1, :, :]
            label = label[::-1, :, :]
        if np.random.rand() < self.flip_prob:
            image = image[:, ::-1, :]
            label = label[:, ::-1, :]
        if np.random.rand() < self.flip_prob:
            image = image[:, :, ::-1]
            label = label[:, :, ::-1]

        # Small random rotation around z-axis (axial plane)
        if np.random.rand() < self.rotate_prob:
            angle = np.random.uniform(-7.0, 7.0)  # degrees
            image, label = rotate_3d(
                image,
                label,
                angle_deg=angle,
                axis="z",
            )

        # Small isotropic zoom
        if np.random.rand() < self.zoom_prob:
            zoom_factor = np.random.uniform(0.9, 1.1)
            image, label = zoom_3d(
                image,
                label,
                zoom_factor,
            )

        # Add gamma augmentation
        if self.gamma_range is not None and np.random.rand() < 0.5:
            g_lo, g_hi = self.gamma_range
            gamma = np.random.uniform(g_lo, g_hi)
            # assume image in [0,1]
            image = np.clip(image, 0.0, 1.0) ** gamma

        # Add Gaussian noise on image only
        if self.noise_std > 0.0 and np.random.rand() < 0.5:
            noise = np.random.normal(0.0, self.noise_std, size=image.shape).astype(image.dtype)
            image = image + noise

            # If your preprocessed intensities are in [0,1], keep them there:
            image = np.clip(image, 0.0, 1.0)

        return image, label


    def __getitem__(self, idx):
        # Map global index to volume index
        if self.train:
            vol_idx = idx // self.samples_per_volume
        else:
            vol_idx = idx
        vol_idx = int(vol_idx % len(self.items))

        # Load and preprocess volume
        image, label = self._load_image_label(vol_idx)

        # For training: sample patch + augment
        if self.train:
            image, label = self._sample_patch(image, label)
            image, label = self._augment(image, label)
        # For validation: use full volume (no patching, no aug)

        sample = {"image": image, "label": label}

        if self._transform is not None:
            sample = self._transform(sample)
            image = sample["image"]
            label = sample["label"]

        # Ensure positive strides / contiguous arrays
        image = np.ascontiguousarray(image)
        label = np.ascontiguousarray(label)

        # Convert to tensors, channel-first for image
        img_tensor = torch.as_tensor(image[None, ...], dtype=torch.float32)  # (1, D, H, W)
        lab_tensor = torch.as_tensor(label, dtype=torch.long)                # (D, H, W)

        return {"image": img_tensor, "label": lab_tensor}



def make_aug_transforms(cfg, train: bool = True):
    """
    Placeholder hook for additional transforms.

    """
    def _identity(sample):
        return sample

    return _identity

Step 1 of Training: Fine tune the base checkpoint so that it is adapted to our training set

In [ ]:
import os, random, argparse, pathlib
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from monai.inferers import sliding_window_inference
import multiprocessing as mp


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def _compute_cldice_worker(p_np, g_np, cldice_metric):
    return float(cldice_metric(p_np, g_np))


def make_items_from_dirs(image_dir, label_dir):
    image_dir = pathlib.Path(image_dir)
    label_dir = pathlib.Path(label_dir)
    items = []

    for img_path in sorted(image_dir.glob("*.nii*")):
        img_name = img_path.name
        if img_name.startswith("image_"):
            lbl_name = "label_" + img_name[len("image_"):]
        else:
            lbl_name = img_name

        lab_path = label_dir / lbl_name
        if not lab_path.exists():
            print(f"WARNING: no label for {img_name}, expected {lab_path}")
            continue
        items.append((str(img_path), str(lab_path)))

    if not items:
        raise RuntimeError(
            f"No image/label pairs found in {image_dir} and {label_dir}."
        )

    return items


def make_loader(kind, cfg, train=True):
    if kind == "train":
        items = make_items_from_dirs(
            cfg["data"]["train_images"], cfg["data"]["train_labels"]
        )
    else:
        items = make_items_from_dirs(
            cfg["data"]["val_images"], cfg["data"]["val_labels"]
        )

    ds = NiftiVolume(items, cfg, train=train)
    aug = make_aug_transforms(cfg, train=train)
    ds.set_transform(aug)

    batch_size = cfg["optim"]["batch_size"] if train else 1
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=train,
        num_workers=cfg["data"].get("num_workers", 8),
        pin_memory=True,
    )


def vessel_loss(
    vessel_logits,
    lab,
    bce_weight: float,
    cldice_weight: float,
    cldice_loss_fn=None,
):
    """
    lab: (B,D,H,W) int {0,1,2}, union-of-vessels = (lab>0)
    vessel_logits: (B,1,D,H,W)
    """
    vessel_gt = (lab > 0).float().unsqueeze(1)    # (B,1,D,H,W)
    if vessel_gt.sum() == 0:
        return vessel_logits.new_tensor(0.0)

    loss = 0.0

    if bce_weight > 0.0:
        bce = F.binary_cross_entropy_with_logits(
            vessel_logits, vessel_gt
        )
        loss = loss + bce_weight * bce

    if cldice_loss_fn is not None and cldice_weight > 0.0:
        vessel_probs = torch.sigmoid(vessel_logits)
        cl = cldice_loss_fn(vessel_gt, vessel_probs)
        loss = loss + cldice_weight * cl

    return loss


@torch.no_grad()
def eval_epoch(model, loader, device, patch_size, cldice_metric=None, num_workers: int = 0):
    model.eval()

    dice_scores = []
    cldice_cases = []

    ds_factor = 1

    for batch in loader:
        img, lab = batch["image"].to(device), batch["label"].to(device).long()

        # Full-volume via sliding window
        logits = sliding_window_inference(
            img, roi_size=patch_size, sw_batch_size=2, predictor=model
        )

        # Vessel head
        vessel_logits = model.vessel_head(logits)      # (B,1,D,H,W)
        vessel_probs = torch.sigmoid(vessel_logits)
        pred = (vessel_probs > 0.5).squeeze(1)        # (B,D,H,W)

        gt = (lab > 0)                                # (B,D,H,W)

        B = pred.shape[0]
        for b in range(B):
            pb = pred[b]
            gb = gt[b]
            if gb.sum() == 0:
                continue

            inter = (pb & gb).sum().float()
            denom = pb.sum().float() + gb.sum().float()
            if denom > 0:
                dice = (2.0 * inter) / (denom + 1e-5)
                dice_scores.append(dice.item())

            if cldice_metric is not None:
                p_small = pb[::ds_factor, ::ds_factor, ::ds_factor]
                g_small = gb[::ds_factor, ::ds_factor, ::ds_factor]
                cldice_cases.append(
                    (p_small.cpu().numpy().astype(bool),
                     g_small.cpu().numpy().astype(bool))
                )

    def _compute_cldice_list(cases):
        if cldice_metric is None or not cases:
            return []
        if num_workers > 0:
            with mp.Pool(processes=num_workers) as pool:
                return pool.starmap(
                    _compute_cldice_worker,
                    [(p, g, cldice_metric) for (p, g) in cases],
                )
        else:
            return [
                _compute_cldice_worker(p, g, cldice_metric)
                for (p, g) in cases
            ]

    cldice_scores = _compute_cldice_list(cldice_cases)

    mean_dice = float(np.mean(dice_scores)) if dice_scores else 0.0
    mean_cl   = float(np.mean(cldice_scores)) if cldice_scores else 0.0
    return mean_dice, mean_cl


def main(cfg):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    set_seed(cfg["seed"])

    val_cldice_workers = cfg["optim"].get("val_cldice_workers", 0)
    train_loader = make_loader("train", cfg, train=True)
    val_loader   = make_loader("val", cfg, train=False)

    # Just for sanity
    first_batch = next(iter(train_loader))
    print("DEBUG vessel: first_batch image:", first_batch["image"].shape)
    print("DEBUG vessel: first_batch label:", first_batch["label"].shape)

    # Model with 3-class head + vessel_head
    model = build_model(
        num_classes=cfg["model"]["num_classes"],
        dropout=cfg["model"].get("dropout", 0.0),
    )

    # Optional pretrain_ckpt from YAML
    pre_ckpt = cfg["model"].get("pretrain_ckpt", None)
    if pre_ckpt:
        print(f"[Stage1] Loading pretrain_ckpt from {pre_ckpt}")
        ckpt = torch.load(pre_ckpt, map_location="cpu")
        state = ckpt.get("state_dict", ckpt)
        model_state = model.state_dict()
        filtered = {}
        for k, v in state.items():
            if k in model_state and v.shape == model_state[k].shape:
                filtered[k] = v
            else:
                print(f"[Stage1] Skipping {k}: {v.shape} vs {model_state.get(k, torch.empty(0)).shape}")
        model.load_state_dict(filtered, strict=False)

    model.to(device)

    # Loss hyperparams
    bce_w = cfg["loss"].get("vessel_bce_weight", 0.5)
    cl_w  = cfg["loss"].get("vessel_cldice_weight", 1.0)

    cldice_loss_fn = SoftCLDiceLoss(
        iter_=cfg["loss"]["soft_cldice_iters"], smooth=1.0
    ).to(device)

    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["optim"]["lr"],
        weight_decay=cfg["optim"]["weight_decay"],
    )
    scaler = GradScaler(enabled=cfg["optim"]["amp"])

    patch_size = cfg["data"]["patch_size"]
    best_cl = -1.0

    history = {"epoch": [], "train_loss": [], "val_dice": [], "val_clDice": []}

    for epoch in range(cfg["optim"]["epochs"]):
        model.train()
        running = []

        for i, batch in enumerate(train_loader, start=1):
            img, lab = batch["image"].to(device), batch["label"].to(device).long()
            opt.zero_grad(set_to_none=True)

            with autocast(enabled=cfg["optim"]["amp"]):
                logits = model(img)                    # (B,3,D,H,W)
                vessel_logits = model.vessel_head(logits)  # (B,1,D,H,W)

                loss = vessel_loss(
                    vessel_logits,
                    lab,
                    bce_weight=bce_w,
                    cldice_weight=cl_w,
                    cldice_loss_fn=cldice_loss_fn,
                )

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running.append(loss.item())

            if i == 1:
                print("[Stage1] DEBUG train patch:", img.shape, lab.shape, flush=True)

            if i % 50 == 0 or i == 1:
                print(f"[Stage1] epoch {epoch+1} batch {i}/{len(train_loader)} loss={loss.item():.4f}")

        mean_train = float(np.mean(running))
        val_dice, val_cl = eval_epoch(
            model, val_loader, device,
            patch_size=patch_size,
            cldice_metric=hard_cldice,
            num_workers=val_cldice_workers,
        )

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(mean_train)
        history["val_dice"].append(val_dice)
        history["val_clDice"].append(val_cl)

        print(
            f"[Stage1][{epoch+1}/{cfg['optim']['epochs']}] "
            f"train={mean_train:.4f} valDice(vessel)={val_dice:.4f} valClDice(vessel)={val_cl:.4f}"
        )

        if val_cl > best_cl:
            best_cl = val_cl
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(
                model.state_dict(),
                f"checkpoints/{cfg['experiment']}_vessel_best.pt",
            )

    try:
        import pandas as pd
        os.makedirs("checkpoints", exist_ok=True)
        pd.DataFrame(history).to_csv(
            f"checkpoints/{cfg['experiment']}_vessel_curve.csv", index=False
        )
    except Exception as e:
        print("Could not save vessel training curve CSV:", e)

In [ ]:
import yaml
import os
import sys

repo_root = "/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/VesselFMAdaptationMethod/vesselFM-main/vesselFM-main"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from vesselfm.seg.train_vessel import main

config_path = os.path.join(repo_root, "configs", "vessel_ct.yaml")
with open(config_path) as f:
    cfg = yaml.safe_load(f)

main(cfg)

DEBUG vessel: first_batch image: torch.Size([2, 1, 96, 96, 96])
DEBUG vessel: first_batch label: torch.Size([2, 96, 96, 96])
[Stage1] Loading pretrain_ckpt from /projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/VesselFMAdaptationMethod/vesselFM-main/vesselFM-main/checkpoints/vesselFM_base.pt
[Stage1] Skipping output_block.conv.conv.weight: torch.Size([1, 32, 1, 1, 1]) vs torch.Size([3, 32, 1, 1, 1])
[Stage1] Skipping output_block.conv.conv.bias: torch.Size([1]) vs torch.Size([3])
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 1 batch 1/560 loss=1.3239
[Stage1] epoch 1 batch 50/560 loss=0.8271
[Stage1] epoch 1 batch 100/560 loss=0.8205
[Stage1] epoch 1 batch 150/560 loss=0.7043
[Stage1] epoch 1 batch 200/560 loss=0.6739
[Stage1] epoch 1 batch 250/560 loss=0.7365
[Stage1] epoch 1 batch 300/560 loss=0.6566
[Stage1] epoch 1 batch 350/560 loss=0.7538
[Stage1] epoch 1 batch 400/560 loss=0.5693
[Stage1] epoch 1 batch 450/560 loss=0.5653
[Stage1] epoch 1 batch 500/560 loss=0.5473
[Stage1] epoch 1 batch 550/560 loss=0.5981
[Stage1][1/150] train=0.6720 valDice(vessel)=0.2615 valClDice(vessel)=0.3887
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 2 batch 1/560 loss=0.5340
[Stage1] epoch 2 batch 50/560 loss=0.5431
[Stage1] epoch 2 batch 100/560 loss=0.5023
[Stage1] epoch 2 batch 150/560 loss=0.5542
[Stage1] epoch 2 batch 200/560 loss=0.5339
[Stage1] epoch 2 batch 250/560 loss=0.5776
[Stage1] epoch 2 batch 300/560 loss=0.5080
[Stage1] epoch 2 batch 350/560 loss=0.5947
[Stage1] epoch 2 batch 400/560 loss=0.5260
[Stage1] epoch 2 batch 450/560 loss=0.5033
[Stage1] epoch 2 batch 500/560 loss=0.5808
[Stage1] epoch 2 batch 550/560 loss=0.5455
[Stage1][2/150] train=0.5495 valDice(vessel)=0.2600 valClDice(vessel)=0.4052
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 3 batch 1/560 loss=0.5794
[Stage1] epoch 3 batch 50/560 loss=0.5841
[Stage1] epoch 3 batch 100/560 loss=0.4893
[Stage1] epoch 3 batch 150/560 loss=0.4806
[Stage1] epoch 3 batch 200/560 loss=0.4927
[Stage1] epoch 3 batch 250/560 loss=0.4989
[Stage1] epoch 3 batch 300/560 loss=0.4466
[Stage1] epoch 3 batch 350/560 loss=0.5178
[Stage1] epoch 3 batch 400/560 loss=0.5056
[Stage1] epoch 3 batch 450/560 loss=0.4810
[Stage1] epoch 3 batch 500/560 loss=0.4783
[Stage1] epoch 3 batch 550/560 loss=0.5172
[Stage1][3/150] train=0.5126 valDice(vessel)=0.3734 valClDice(vessel)=0.5992
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 4 batch 1/560 loss=0.5417
[Stage1] epoch 4 batch 50/560 loss=0.5561
[Stage1] epoch 4 batch 100/560 loss=0.4869
[Stage1] epoch 4 batch 150/560 loss=0.5283
[Stage1] epoch 4 batch 200/560 loss=0.4470
[Stage1] epoch 4 batch 250/560 loss=0.5088
[Stage1] epoch 4 batch 300/560 loss=0.4466
[Stage1] epoch 4 batch 350/560 loss=0.4647
[Stage1] epoch 4 batch 400/560 loss=0.5054
[Stage1] epoch 4 batch 450/560 loss=0.4833
[Stage1] epoch 4 batch 500/560 loss=0.4288
[Stage1] epoch 4 batch 550/560 loss=0.4919
[Stage1][4/150] train=0.5022 valDice(vessel)=0.3620 valClDice(vessel)=0.5425
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 5 batch 1/560 loss=0.4671
[Stage1] epoch 5 batch 50/560 loss=0.4862
[Stage1] epoch 5 batch 100/560 loss=0.4326
[Stage1] epoch 5 batch 150/560 loss=0.4812
[Stage1] epoch 5 batch 200/560 loss=0.4500
[Stage1] epoch 5 batch 250/560 loss=0.4475
[Stage1] epoch 5 batch 300/560 loss=0.4910
[Stage1] epoch 5 batch 350/560 loss=0.4683
[Stage1] epoch 5 batch 400/560 loss=0.4401
[Stage1] epoch 5 batch 450/560 loss=0.4889
[Stage1] epoch 5 batch 500/560 loss=0.4911
[Stage1] epoch 5 batch 550/560 loss=0.4452
[Stage1][5/150] train=0.4915 valDice(vessel)=0.3966 valClDice(vessel)=0.5910
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 6 batch 1/560 loss=0.4270
[Stage1] epoch 6 batch 50/560 loss=0.4426
[Stage1] epoch 6 batch 100/560 loss=0.4863
[Stage1] epoch 6 batch 150/560 loss=0.4612
[Stage1] epoch 6 batch 200/560 loss=0.4821
[Stage1] epoch 6 batch 250/560 loss=0.4636
[Stage1] epoch 6 batch 300/560 loss=0.4574
[Stage1] epoch 6 batch 350/560 loss=0.4708
[Stage1] epoch 6 batch 400/560 loss=0.4983
[Stage1] epoch 6 batch 450/560 loss=0.4529
[Stage1] epoch 6 batch 500/560 loss=0.4809
[Stage1] epoch 6 batch 550/560 loss=0.4818
[Stage1][6/150] train=0.4834 valDice(vessel)=0.3893 valClDice(vessel)=0.5930
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 7 batch 1/560 loss=0.4550
[Stage1] epoch 7 batch 50/560 loss=0.5095
[Stage1] epoch 7 batch 100/560 loss=0.4945
[Stage1] epoch 7 batch 150/560 loss=0.4641
[Stage1] epoch 7 batch 200/560 loss=0.4679
[Stage1] epoch 7 batch 250/560 loss=0.5205
[Stage1] epoch 7 batch 300/560 loss=0.4700
[Stage1] epoch 7 batch 350/560 loss=0.4927
[Stage1] epoch 7 batch 400/560 loss=0.4806
[Stage1] epoch 7 batch 450/560 loss=0.4863
[Stage1] epoch 7 batch 500/560 loss=0.4677
[Stage1] epoch 7 batch 550/560 loss=0.4263
[Stage1][7/150] train=0.4818 valDice(vessel)=0.3926 valClDice(vessel)=0.5847
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 8 batch 1/560 loss=0.4776
[Stage1] epoch 8 batch 50/560 loss=0.4190
[Stage1] epoch 8 batch 100/560 loss=0.4776
[Stage1] epoch 8 batch 150/560 loss=0.4612
[Stage1] epoch 8 batch 200/560 loss=0.5022
[Stage1] epoch 8 batch 250/560 loss=0.4680
[Stage1] epoch 8 batch 300/560 loss=0.4631
[Stage1] epoch 8 batch 350/560 loss=0.4709
[Stage1] epoch 8 batch 400/560 loss=0.4625
[Stage1] epoch 8 batch 450/560 loss=0.4386
[Stage1] epoch 8 batch 500/560 loss=0.4860
[Stage1] epoch 8 batch 550/560 loss=0.4524
[Stage1][8/150] train=0.4724 valDice(vessel)=0.3716 valClDice(vessel)=0.5775
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 9 batch 1/560 loss=0.4349
[Stage1] epoch 9 batch 50/560 loss=0.4809
[Stage1] epoch 9 batch 100/560 loss=0.4510
[Stage1] epoch 9 batch 150/560 loss=0.4886
[Stage1] epoch 9 batch 200/560 loss=0.4503
[Stage1] epoch 9 batch 250/560 loss=0.5003
[Stage1] epoch 9 batch 300/560 loss=0.4855
[Stage1] epoch 9 batch 350/560 loss=0.4492
[Stage1] epoch 9 batch 400/560 loss=0.4552
[Stage1] epoch 9 batch 450/560 loss=0.4659
[Stage1] epoch 9 batch 500/560 loss=0.4796
[Stage1] epoch 9 batch 550/560 loss=0.4456
[Stage1][9/150] train=0.4788 valDice(vessel)=0.3438 valClDice(vessel)=0.5391
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 10 batch 1/560 loss=0.4425
[Stage1] epoch 10 batch 50/560 loss=0.4423
[Stage1] epoch 10 batch 100/560 loss=0.4466
[Stage1] epoch 10 batch 150/560 loss=0.4584
[Stage1] epoch 10 batch 200/560 loss=0.4762
[Stage1] epoch 10 batch 250/560 loss=0.4816
[Stage1] epoch 10 batch 300/560 loss=0.4907
[Stage1] epoch 10 batch 350/560 loss=0.4712
[Stage1] epoch 10 batch 400/560 loss=0.4691
[Stage1] epoch 10 batch 450/560 loss=0.4888
[Stage1] epoch 10 batch 500/560 loss=0.9693
[Stage1] epoch 10 batch 550/560 loss=0.4320
[Stage1][10/150] train=0.4702 valDice(vessel)=0.3619 valClDice(vessel)=0.5806
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 11 batch 1/560 loss=0.4758
[Stage1] epoch 11 batch 50/560 loss=0.4251
[Stage1] epoch 11 batch 100/560 loss=0.4437
[Stage1] epoch 11 batch 150/560 loss=0.4655
[Stage1] epoch 11 batch 200/560 loss=0.4538
[Stage1] epoch 11 batch 250/560 loss=0.4330
[Stage1] epoch 11 batch 300/560 loss=0.4597
[Stage1] epoch 11 batch 350/560 loss=0.4416
[Stage1] epoch 11 batch 400/560 loss=0.4821
[Stage1] epoch 11 batch 450/560 loss=0.4684
[Stage1] epoch 11 batch 500/560 loss=0.4700
[Stage1] epoch 11 batch 550/560 loss=0.5012
[Stage1][11/150] train=0.4681 valDice(vessel)=0.3305 valClDice(vessel)=0.5216
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 12 batch 1/560 loss=0.4906
[Stage1] epoch 12 batch 50/560 loss=0.4686
[Stage1] epoch 12 batch 100/560 loss=0.4475
[Stage1] epoch 12 batch 150/560 loss=0.4707
[Stage1] epoch 12 batch 200/560 loss=0.4482
[Stage1] epoch 12 batch 250/560 loss=0.4449
[Stage1] epoch 12 batch 300/560 loss=0.4217
[Stage1] epoch 12 batch 350/560 loss=0.4626
[Stage1] epoch 12 batch 400/560 loss=0.4799
[Stage1] epoch 12 batch 450/560 loss=0.4369
[Stage1] epoch 12 batch 500/560 loss=0.4701
[Stage1] epoch 12 batch 550/560 loss=0.4904
[Stage1][12/150] train=0.4719 valDice(vessel)=0.3845 valClDice(vessel)=0.5716
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 13 batch 1/560 loss=0.5607
[Stage1] epoch 13 batch 50/560 loss=0.4451
[Stage1] epoch 13 batch 100/560 loss=1.2132
[Stage1] epoch 13 batch 150/560 loss=0.4857
[Stage1] epoch 13 batch 200/560 loss=0.4947
[Stage1] epoch 13 batch 250/560 loss=0.4436
[Stage1] epoch 13 batch 300/560 loss=0.4791
[Stage1] epoch 13 batch 350/560 loss=0.4404
[Stage1] epoch 13 batch 400/560 loss=0.4467
[Stage1] epoch 13 batch 450/560 loss=0.4238
[Stage1] epoch 13 batch 500/560 loss=0.4894
[Stage1] epoch 13 batch 550/560 loss=0.4465
[Stage1][13/150] train=0.4677 valDice(vessel)=0.3538 valClDice(vessel)=0.5203
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 14 batch 1/560 loss=0.4930
[Stage1] epoch 14 batch 50/560 loss=0.4208
[Stage1] epoch 14 batch 100/560 loss=0.4681
[Stage1] epoch 14 batch 150/560 loss=0.4745
[Stage1] epoch 14 batch 200/560 loss=0.4339
[Stage1] epoch 14 batch 250/560 loss=0.4503
[Stage1] epoch 14 batch 300/560 loss=0.4805
[Stage1] epoch 14 batch 350/560 loss=0.4103
[Stage1] epoch 14 batch 400/560 loss=0.4814
[Stage1] epoch 14 batch 450/560 loss=0.4675
[Stage1] epoch 14 batch 500/560 loss=0.4585
[Stage1] epoch 14 batch 550/560 loss=0.4825
[Stage1][14/150] train=0.4654 valDice(vessel)=0.3332 valClDice(vessel)=0.5224
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 15 batch 1/560 loss=0.4493
[Stage1] epoch 15 batch 50/560 loss=0.4891
[Stage1] epoch 15 batch 100/560 loss=0.4519
[Stage1] epoch 15 batch 150/560 loss=0.5027
[Stage1] epoch 15 batch 200/560 loss=0.5353
[Stage1] epoch 15 batch 250/560 loss=0.4632
[Stage1] epoch 15 batch 300/560 loss=0.4380
[Stage1] epoch 15 batch 350/560 loss=0.4206
[Stage1] epoch 15 batch 400/560 loss=0.4201
[Stage1] epoch 15 batch 450/560 loss=0.4388
[Stage1] epoch 15 batch 500/560 loss=0.4072
[Stage1] epoch 15 batch 550/560 loss=0.4179
[Stage1][15/150] train=0.4624 valDice(vessel)=0.4019 valClDice(vessel)=0.5912
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 16 batch 1/560 loss=0.4614
[Stage1] epoch 16 batch 50/560 loss=0.4616
[Stage1] epoch 16 batch 100/560 loss=0.4274
[Stage1] epoch 16 batch 150/560 loss=0.4841
[Stage1] epoch 16 batch 200/560 loss=0.5161
[Stage1] epoch 16 batch 250/560 loss=0.4443
[Stage1] epoch 16 batch 300/560 loss=0.4903
[Stage1] epoch 16 batch 350/560 loss=0.4926
[Stage1] epoch 16 batch 400/560 loss=0.4438
[Stage1] epoch 16 batch 450/560 loss=0.4489
[Stage1] epoch 16 batch 500/560 loss=0.4786
[Stage1] epoch 16 batch 550/560 loss=0.4314
[Stage1][16/150] train=0.4610 valDice(vessel)=0.3779 valClDice(vessel)=0.5453
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 17 batch 1/560 loss=0.4723
[Stage1] epoch 17 batch 50/560 loss=0.4846
[Stage1] epoch 17 batch 100/560 loss=0.4272
[Stage1] epoch 17 batch 150/560 loss=0.4478
[Stage1] epoch 17 batch 200/560 loss=0.4740
[Stage1] epoch 17 batch 250/560 loss=0.4545
[Stage1] epoch 17 batch 300/560 loss=0.4357
[Stage1] epoch 17 batch 350/560 loss=0.4140
[Stage1] epoch 17 batch 400/560 loss=0.4928
[Stage1] epoch 17 batch 450/560 loss=0.5430
[Stage1] epoch 17 batch 500/560 loss=0.4373
[Stage1] epoch 17 batch 550/560 loss=0.4348
[Stage1][17/150] train=0.4600 valDice(vessel)=0.3621 valClDice(vessel)=0.5364
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 18 batch 1/560 loss=0.4504
[Stage1] epoch 18 batch 50/560 loss=0.4582
[Stage1] epoch 18 batch 100/560 loss=0.4598
[Stage1] epoch 18 batch 150/560 loss=0.5292
[Stage1] epoch 18 batch 200/560 loss=0.4862
[Stage1] epoch 18 batch 250/560 loss=0.4728
[Stage1] epoch 18 batch 300/560 loss=0.4724
[Stage1] epoch 18 batch 350/560 loss=0.4663
[Stage1] epoch 18 batch 400/560 loss=0.4248
[Stage1] epoch 18 batch 450/560 loss=0.4580
[Stage1] epoch 18 batch 500/560 loss=0.4491
[Stage1] epoch 18 batch 550/560 loss=0.4280
[Stage1][18/150] train=0.4589 valDice(vessel)=0.3865 valClDice(vessel)=0.5413
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 19 batch 1/560 loss=0.4502
[Stage1] epoch 19 batch 50/560 loss=0.4130
[Stage1] epoch 19 batch 100/560 loss=0.4464
[Stage1] epoch 19 batch 150/560 loss=0.4421
[Stage1] epoch 19 batch 200/560 loss=0.4073
[Stage1] epoch 19 batch 250/560 loss=0.4292
[Stage1] epoch 19 batch 300/560 loss=0.4709
[Stage1] epoch 19 batch 350/560 loss=0.4763
[Stage1] epoch 19 batch 400/560 loss=0.4218
[Stage1] epoch 19 batch 450/560 loss=0.4626
[Stage1] epoch 19 batch 500/560 loss=0.4412
[Stage1] epoch 19 batch 550/560 loss=0.4111
[Stage1][19/150] train=0.4567 valDice(vessel)=0.3995 valClDice(vessel)=0.5624
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 20 batch 1/560 loss=0.4264
[Stage1] epoch 20 batch 50/560 loss=0.4267
[Stage1] epoch 20 batch 100/560 loss=0.4434
[Stage1] epoch 20 batch 150/560 loss=0.4529
[Stage1] epoch 20 batch 200/560 loss=0.4627
[Stage1] epoch 20 batch 250/560 loss=0.4255
[Stage1] epoch 20 batch 300/560 loss=0.4401
[Stage1] epoch 20 batch 350/560 loss=0.4387
[Stage1] epoch 20 batch 400/560 loss=0.4590
[Stage1] epoch 20 batch 450/560 loss=0.4247
[Stage1] epoch 20 batch 500/560 loss=0.4393
[Stage1] epoch 20 batch 550/560 loss=0.4336
[Stage1][20/150] train=0.4579 valDice(vessel)=0.4036 valClDice(vessel)=0.5852
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 21 batch 1/560 loss=0.4815
[Stage1] epoch 21 batch 50/560 loss=0.4182
[Stage1] epoch 21 batch 100/560 loss=0.4224
[Stage1] epoch 21 batch 150/560 loss=0.4380
[Stage1] epoch 21 batch 200/560 loss=0.4212
[Stage1] epoch 21 batch 250/560 loss=0.4331
[Stage1] epoch 21 batch 300/560 loss=0.4364
[Stage1] epoch 21 batch 350/560 loss=0.4695
[Stage1] epoch 21 batch 400/560 loss=0.4692
[Stage1] epoch 21 batch 450/560 loss=0.4667
[Stage1] epoch 21 batch 500/560 loss=0.4909
[Stage1] epoch 21 batch 550/560 loss=0.5032
[Stage1][21/150] train=0.4590 valDice(vessel)=0.3972 valClDice(vessel)=0.5625
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 22 batch 1/560 loss=0.4320
[Stage1] epoch 22 batch 50/560 loss=0.4751
[Stage1] epoch 22 batch 100/560 loss=0.4313
[Stage1] epoch 22 batch 150/560 loss=0.4662
[Stage1] epoch 22 batch 200/560 loss=0.4265
[Stage1] epoch 22 batch 250/560 loss=0.4370
[Stage1] epoch 22 batch 300/560 loss=0.4398
[Stage1] epoch 22 batch 350/560 loss=0.4348
[Stage1] epoch 22 batch 400/560 loss=0.4669
[Stage1] epoch 22 batch 450/560 loss=0.4452
[Stage1] epoch 22 batch 500/560 loss=0.4813
[Stage1] epoch 22 batch 550/560 loss=0.4164
[Stage1][22/150] train=0.4567 valDice(vessel)=0.4315 valClDice(vessel)=0.6605
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 23 batch 1/560 loss=0.4702
[Stage1] epoch 23 batch 50/560 loss=0.4481
[Stage1] epoch 23 batch 100/560 loss=0.4389
[Stage1] epoch 23 batch 150/560 loss=0.4467
[Stage1] epoch 23 batch 200/560 loss=0.4225
[Stage1] epoch 23 batch 250/560 loss=0.4260
[Stage1] epoch 23 batch 300/560 loss=0.4665
[Stage1] epoch 23 batch 350/560 loss=0.4301
[Stage1] epoch 23 batch 400/560 loss=0.4472
[Stage1] epoch 23 batch 450/560 loss=0.4293
[Stage1] epoch 23 batch 500/560 loss=0.4318
[Stage1] epoch 23 batch 550/560 loss=0.4353
[Stage1][23/150] train=0.4528 valDice(vessel)=0.4100 valClDice(vessel)=0.6090
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 24 batch 1/560 loss=0.4275
[Stage1] epoch 24 batch 50/560 loss=0.8118
[Stage1] epoch 24 batch 100/560 loss=0.4622
[Stage1] epoch 24 batch 150/560 loss=0.4064
[Stage1] epoch 24 batch 200/560 loss=0.4442
[Stage1] epoch 24 batch 250/560 loss=0.4384
[Stage1] epoch 24 batch 300/560 loss=0.5009
[Stage1] epoch 24 batch 350/560 loss=0.4152
[Stage1] epoch 24 batch 400/560 loss=0.4592
[Stage1] epoch 24 batch 450/560 loss=0.5164
[Stage1] epoch 24 batch 500/560 loss=0.4499
[Stage1] epoch 24 batch 550/560 loss=0.4418
[Stage1][24/150] train=0.4512 valDice(vessel)=0.4042 valClDice(vessel)=0.5609
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 25 batch 1/560 loss=0.4713
[Stage1] epoch 25 batch 50/560 loss=0.4317
[Stage1] epoch 25 batch 100/560 loss=0.4749
[Stage1] epoch 25 batch 150/560 loss=0.4442
[Stage1] epoch 25 batch 200/560 loss=0.4140
[Stage1] epoch 25 batch 250/560 loss=0.4906
[Stage1] epoch 25 batch 300/560 loss=0.4212
[Stage1] epoch 25 batch 350/560 loss=0.4220
[Stage1] epoch 25 batch 400/560 loss=0.4201
[Stage1] epoch 25 batch 450/560 loss=0.4629
[Stage1] epoch 25 batch 500/560 loss=0.4448
[Stage1] epoch 25 batch 550/560 loss=0.4164
[Stage1][25/150] train=0.4532 valDice(vessel)=0.3677 valClDice(vessel)=0.5192
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 26 batch 1/560 loss=0.4712
[Stage1] epoch 26 batch 50/560 loss=0.4391
[Stage1] epoch 26 batch 100/560 loss=0.4213
[Stage1] epoch 26 batch 150/560 loss=0.5729
[Stage1] epoch 26 batch 200/560 loss=0.4241
[Stage1] epoch 26 batch 250/560 loss=0.4187
[Stage1] epoch 26 batch 300/560 loss=0.4533
[Stage1] epoch 26 batch 350/560 loss=0.4266
[Stage1] epoch 26 batch 400/560 loss=0.4325
[Stage1] epoch 26 batch 450/560 loss=0.4526
[Stage1] epoch 26 batch 500/560 loss=0.4230
[Stage1] epoch 26 batch 550/560 loss=0.4422
[Stage1][26/150] train=0.4525 valDice(vessel)=0.3466 valClDice(vessel)=0.5186
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 27 batch 1/560 loss=0.4248
[Stage1] epoch 27 batch 50/560 loss=0.5069
[Stage1] epoch 27 batch 100/560 loss=0.4305
[Stage1] epoch 27 batch 150/560 loss=0.4296
[Stage1] epoch 27 batch 200/560 loss=0.4254
[Stage1] epoch 27 batch 250/560 loss=0.4159
[Stage1] epoch 27 batch 300/560 loss=0.4108
[Stage1] epoch 27 batch 350/560 loss=0.4300
[Stage1] epoch 27 batch 400/560 loss=0.4243
[Stage1] epoch 27 batch 450/560 loss=0.4143
[Stage1] epoch 27 batch 500/560 loss=0.8710
[Stage1] epoch 27 batch 550/560 loss=0.4349
[Stage1][27/150] train=0.4485 valDice(vessel)=0.4248 valClDice(vessel)=0.6116
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 28 batch 1/560 loss=0.4375
[Stage1] epoch 28 batch 50/560 loss=0.4204
[Stage1] epoch 28 batch 100/560 loss=0.4433
[Stage1] epoch 28 batch 150/560 loss=0.4384
[Stage1] epoch 28 batch 200/560 loss=0.4249
[Stage1] epoch 28 batch 250/560 loss=0.4350
[Stage1] epoch 28 batch 300/560 loss=0.4452
[Stage1] epoch 28 batch 350/560 loss=0.4771
[Stage1] epoch 28 batch 400/560 loss=0.4454
[Stage1] epoch 28 batch 450/560 loss=0.4376
[Stage1] epoch 28 batch 500/560 loss=0.4369
[Stage1] epoch 28 batch 550/560 loss=0.4208
[Stage1][28/150] train=0.4485 valDice(vessel)=0.3892 valClDice(vessel)=0.5173
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 29 batch 1/560 loss=0.4223
[Stage1] epoch 29 batch 50/560 loss=0.4135
[Stage1] epoch 29 batch 100/560 loss=0.4134
[Stage1] epoch 29 batch 150/560 loss=0.4479
[Stage1] epoch 29 batch 200/560 loss=0.4686
[Stage1] epoch 29 batch 250/560 loss=0.4469
[Stage1] epoch 29 batch 300/560 loss=0.4548
[Stage1] epoch 29 batch 350/560 loss=0.8978
[Stage1] epoch 29 batch 400/560 loss=0.4448
[Stage1] epoch 29 batch 450/560 loss=0.4766
[Stage1] epoch 29 batch 500/560 loss=0.4322
[Stage1] epoch 29 batch 550/560 loss=0.4314
[Stage1][29/150] train=0.4483 valDice(vessel)=0.3625 valClDice(vessel)=0.4713
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 30 batch 1/560 loss=0.4450
[Stage1] epoch 30 batch 50/560 loss=0.4564
[Stage1] epoch 30 batch 100/560 loss=0.4678
[Stage1] epoch 30 batch 150/560 loss=0.4247
[Stage1] epoch 30 batch 200/560 loss=0.4174
[Stage1] epoch 30 batch 250/560 loss=0.4610
[Stage1] epoch 30 batch 300/560 loss=0.4384
[Stage1] epoch 30 batch 350/560 loss=0.4188
[Stage1] epoch 30 batch 400/560 loss=0.4460
[Stage1] epoch 30 batch 450/560 loss=0.4410
[Stage1] epoch 30 batch 500/560 loss=0.4648
[Stage1] epoch 30 batch 550/560 loss=0.4482
[Stage1][30/150] train=0.4490 valDice(vessel)=0.4053 valClDice(vessel)=0.5768
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 31 batch 1/560 loss=0.4308
[Stage1] epoch 31 batch 50/560 loss=0.4180
[Stage1] epoch 31 batch 100/560 loss=0.4398
[Stage1] epoch 31 batch 150/560 loss=0.4487
[Stage1] epoch 31 batch 200/560 loss=0.4194
[Stage1] epoch 31 batch 250/560 loss=0.4884
[Stage1] epoch 31 batch 300/560 loss=0.4390
[Stage1] epoch 31 batch 350/560 loss=0.3972
[Stage1] epoch 31 batch 400/560 loss=0.4026
[Stage1] epoch 31 batch 450/560 loss=0.4227
[Stage1] epoch 31 batch 500/560 loss=0.4360
[Stage1] epoch 31 batch 550/560 loss=0.4560
[Stage1][31/150] train=0.4479 valDice(vessel)=0.4134 valClDice(vessel)=0.5757
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 32 batch 1/560 loss=0.4957
[Stage1] epoch 32 batch 50/560 loss=0.4164
[Stage1] epoch 32 batch 100/560 loss=0.4388
[Stage1] epoch 32 batch 150/560 loss=0.4345
[Stage1] epoch 32 batch 200/560 loss=0.9600
[Stage1] epoch 32 batch 250/560 loss=0.4569
[Stage1] epoch 32 batch 300/560 loss=0.4478
[Stage1] epoch 32 batch 350/560 loss=0.7851
[Stage1] epoch 32 batch 400/560 loss=0.4094
[Stage1] epoch 32 batch 450/560 loss=0.4348
[Stage1] epoch 32 batch 500/560 loss=0.4386
[Stage1] epoch 32 batch 550/560 loss=0.4157
[Stage1][32/150] train=0.4463 valDice(vessel)=0.4081 valClDice(vessel)=0.5285
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 33 batch 1/560 loss=0.4454
[Stage1] epoch 33 batch 50/560 loss=0.4593
[Stage1] epoch 33 batch 100/560 loss=0.4156
[Stage1] epoch 33 batch 150/560 loss=0.4694
[Stage1] epoch 33 batch 200/560 loss=0.4911
[Stage1] epoch 33 batch 250/560 loss=0.4518
[Stage1] epoch 33 batch 300/560 loss=0.4463
[Stage1] epoch 33 batch 350/560 loss=0.4192
[Stage1] epoch 33 batch 400/560 loss=0.4531
[Stage1] epoch 33 batch 450/560 loss=0.9404
[Stage1] epoch 33 batch 500/560 loss=0.8045
[Stage1] epoch 33 batch 550/560 loss=0.4478
[Stage1][33/150] train=0.4483 valDice(vessel)=0.3810 valClDice(vessel)=0.5177
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 34 batch 1/560 loss=0.4263
[Stage1] epoch 34 batch 50/560 loss=0.4424
[Stage1] epoch 34 batch 100/560 loss=0.4486
[Stage1] epoch 34 batch 150/560 loss=0.4373
[Stage1] epoch 34 batch 200/560 loss=0.4345
[Stage1] epoch 34 batch 250/560 loss=0.4813
[Stage1] epoch 34 batch 300/560 loss=0.4597
[Stage1] epoch 34 batch 350/560 loss=0.4765
[Stage1] epoch 34 batch 400/560 loss=0.4223
[Stage1] epoch 34 batch 450/560 loss=0.4657
[Stage1] epoch 34 batch 500/560 loss=0.4364
[Stage1] epoch 34 batch 550/560 loss=0.4296
[Stage1][34/150] train=0.4502 valDice(vessel)=0.4228 valClDice(vessel)=0.6399
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 35 batch 1/560 loss=0.4247
[Stage1] epoch 35 batch 50/560 loss=0.5107
[Stage1] epoch 35 batch 100/560 loss=0.4437
[Stage1] epoch 35 batch 150/560 loss=0.4330
[Stage1] epoch 35 batch 200/560 loss=0.4070
[Stage1] epoch 35 batch 250/560 loss=0.4810
[Stage1] epoch 35 batch 300/560 loss=0.4029
[Stage1] epoch 35 batch 350/560 loss=0.4180
[Stage1] epoch 35 batch 400/560 loss=0.4363
[Stage1] epoch 35 batch 450/560 loss=0.4152
[Stage1] epoch 35 batch 500/560 loss=0.4600
[Stage1] epoch 35 batch 550/560 loss=0.4184
[Stage1][35/150] train=0.4464 valDice(vessel)=0.4228 valClDice(vessel)=0.6305
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 36 batch 1/560 loss=0.4322
[Stage1] epoch 36 batch 50/560 loss=0.4114
[Stage1] epoch 36 batch 100/560 loss=0.4123
[Stage1] epoch 36 batch 150/560 loss=0.4379
[Stage1] epoch 36 batch 200/560 loss=0.4818
[Stage1] epoch 36 batch 250/560 loss=0.4256
[Stage1] epoch 36 batch 300/560 loss=0.4645
[Stage1] epoch 36 batch 350/560 loss=0.4278
[Stage1] epoch 36 batch 400/560 loss=0.4173
[Stage1] epoch 36 batch 450/560 loss=0.4137
[Stage1] epoch 36 batch 500/560 loss=0.4827
[Stage1] epoch 36 batch 550/560 loss=0.4166
[Stage1][36/150] train=0.4434 valDice(vessel)=0.4459 valClDice(vessel)=0.6386
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 37 batch 1/560 loss=0.4287
[Stage1] epoch 37 batch 50/560 loss=0.4810
[Stage1] epoch 37 batch 100/560 loss=0.4269
[Stage1] epoch 37 batch 150/560 loss=0.4028
[Stage1] epoch 37 batch 200/560 loss=0.4395
[Stage1] epoch 37 batch 250/560 loss=0.4439
[Stage1] epoch 37 batch 300/560 loss=0.4253
[Stage1] epoch 37 batch 350/560 loss=0.4626
[Stage1] epoch 37 batch 400/560 loss=0.4442
[Stage1] epoch 37 batch 450/560 loss=0.4545
[Stage1] epoch 37 batch 500/560 loss=0.4390
[Stage1] epoch 37 batch 550/560 loss=0.4411
[Stage1][37/150] train=0.4440 valDice(vessel)=0.3994 valClDice(vessel)=0.6041
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 38 batch 1/560 loss=0.4073
[Stage1] epoch 38 batch 50/560 loss=0.4491
[Stage1] epoch 38 batch 100/560 loss=0.4981
[Stage1] epoch 38 batch 150/560 loss=0.4411
[Stage1] epoch 38 batch 200/560 loss=0.4238
[Stage1] epoch 38 batch 250/560 loss=0.4076
[Stage1] epoch 38 batch 300/560 loss=0.4128
[Stage1] epoch 38 batch 350/560 loss=0.4250
[Stage1] epoch 38 batch 400/560 loss=0.4250
[Stage1] epoch 38 batch 450/560 loss=0.4580
[Stage1] epoch 38 batch 500/560 loss=0.4190
[Stage1] epoch 38 batch 550/560 loss=0.4883
[Stage1][38/150] train=0.4458 valDice(vessel)=0.3725 valClDice(vessel)=0.5940
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 39 batch 1/560 loss=0.4895
[Stage1] epoch 39 batch 50/560 loss=0.4396
[Stage1] epoch 39 batch 100/560 loss=0.4495
[Stage1] epoch 39 batch 150/560 loss=0.4129
[Stage1] epoch 39 batch 200/560 loss=0.4854
[Stage1] epoch 39 batch 250/560 loss=0.4217
[Stage1] epoch 39 batch 300/560 loss=0.4043
[Stage1] epoch 39 batch 350/560 loss=0.4555
[Stage1] epoch 39 batch 400/560 loss=0.4762
[Stage1] epoch 39 batch 450/560 loss=0.4125
[Stage1] epoch 39 batch 500/560 loss=0.4222
[Stage1] epoch 39 batch 550/560 loss=0.4275
[Stage1][39/150] train=0.4434 valDice(vessel)=0.3823 valClDice(vessel)=0.5693
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 40 batch 1/560 loss=0.4448
[Stage1] epoch 40 batch 50/560 loss=0.4364
[Stage1] epoch 40 batch 100/560 loss=0.4288
[Stage1] epoch 40 batch 150/560 loss=0.4076
[Stage1] epoch 40 batch 200/560 loss=0.4151
[Stage1] epoch 40 batch 250/560 loss=0.4223
[Stage1] epoch 40 batch 300/560 loss=0.4320
[Stage1] epoch 40 batch 350/560 loss=0.4174
[Stage1] epoch 40 batch 400/560 loss=0.4311
[Stage1] epoch 40 batch 450/560 loss=0.4045
[Stage1] epoch 40 batch 500/560 loss=0.4183
[Stage1] epoch 40 batch 550/560 loss=0.4241
[Stage1][40/150] train=0.4418 valDice(vessel)=0.4061 valClDice(vessel)=0.5543
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 41 batch 1/560 loss=0.4286
[Stage1] epoch 41 batch 50/560 loss=0.4155
[Stage1] epoch 41 batch 100/560 loss=0.4441
[Stage1] epoch 41 batch 150/560 loss=0.4227
[Stage1] epoch 41 batch 200/560 loss=0.4330
[Stage1] epoch 41 batch 250/560 loss=0.4483
[Stage1] epoch 41 batch 300/560 loss=0.4210
[Stage1] epoch 41 batch 350/560 loss=0.4490
[Stage1] epoch 41 batch 400/560 loss=0.4169
[Stage1] epoch 41 batch 450/560 loss=0.4106
[Stage1] epoch 41 batch 500/560 loss=0.4346
[Stage1] epoch 41 batch 550/560 loss=0.4512
[Stage1][41/150] train=0.4408 valDice(vessel)=0.3984 valClDice(vessel)=0.5661
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 42 batch 1/560 loss=0.4230
[Stage1] epoch 42 batch 50/560 loss=0.4967
[Stage1] epoch 42 batch 100/560 loss=0.3968
[Stage1] epoch 42 batch 150/560 loss=0.4329
[Stage1] epoch 42 batch 200/560 loss=0.4180
[Stage1] epoch 42 batch 250/560 loss=0.4832
[Stage1] epoch 42 batch 300/560 loss=0.4582
[Stage1] epoch 42 batch 350/560 loss=0.4377
[Stage1] epoch 42 batch 400/560 loss=0.4535
[Stage1] epoch 42 batch 450/560 loss=0.4974
[Stage1] epoch 42 batch 500/560 loss=0.4268
[Stage1] epoch 42 batch 550/560 loss=0.4476
[Stage1][42/150] train=0.4435 valDice(vessel)=0.4276 valClDice(vessel)=0.6105
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 43 batch 1/560 loss=0.4521
[Stage1] epoch 43 batch 50/560 loss=0.4205
[Stage1] epoch 43 batch 100/560 loss=0.4301
[Stage1] epoch 43 batch 150/560 loss=0.4522
[Stage1] epoch 43 batch 200/560 loss=0.4727
[Stage1] epoch 43 batch 250/560 loss=0.4252
[Stage1] epoch 43 batch 300/560 loss=0.3948
[Stage1] epoch 43 batch 350/560 loss=0.4212
[Stage1] epoch 43 batch 400/560 loss=0.4591
[Stage1] epoch 43 batch 450/560 loss=0.4360
[Stage1] epoch 43 batch 500/560 loss=0.4037
[Stage1] epoch 43 batch 550/560 loss=0.4034
[Stage1][43/150] train=0.4414 valDice(vessel)=0.3541 valClDice(vessel)=0.4845
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 44 batch 1/560 loss=0.4166
[Stage1] epoch 44 batch 50/560 loss=0.4661
[Stage1] epoch 44 batch 100/560 loss=0.4325
[Stage1] epoch 44 batch 150/560 loss=0.3992
[Stage1] epoch 44 batch 200/560 loss=0.4950
[Stage1] epoch 44 batch 250/560 loss=0.4068
[Stage1] epoch 44 batch 300/560 loss=0.4265
[Stage1] epoch 44 batch 350/560 loss=0.4780
[Stage1] epoch 44 batch 400/560 loss=0.4867
[Stage1] epoch 44 batch 450/560 loss=0.4109
[Stage1] epoch 44 batch 500/560 loss=0.4228
[Stage1] epoch 44 batch 550/560 loss=0.4223
[Stage1][44/150] train=0.4387 valDice(vessel)=0.3756 valClDice(vessel)=0.4996
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 45 batch 1/560 loss=0.4743
[Stage1] epoch 45 batch 50/560 loss=0.4166
[Stage1] epoch 45 batch 100/560 loss=0.4002
[Stage1] epoch 45 batch 150/560 loss=0.4045
[Stage1] epoch 45 batch 200/560 loss=0.4277
[Stage1] epoch 45 batch 250/560 loss=0.4420
[Stage1] epoch 45 batch 300/560 loss=0.4190
[Stage1] epoch 45 batch 350/560 loss=0.4279
[Stage1] epoch 45 batch 400/560 loss=0.4265
[Stage1] epoch 45 batch 450/560 loss=0.4251
[Stage1] epoch 45 batch 500/560 loss=0.4080
[Stage1] epoch 45 batch 550/560 loss=0.4628
[Stage1][45/150] train=0.4372 valDice(vessel)=0.4460 valClDice(vessel)=0.6294
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 46 batch 1/560 loss=0.8617
[Stage1] epoch 46 batch 50/560 loss=0.4063
[Stage1] epoch 46 batch 100/560 loss=0.4501
[Stage1] epoch 46 batch 150/560 loss=0.4479
[Stage1] epoch 46 batch 200/560 loss=0.4343
[Stage1] epoch 46 batch 250/560 loss=0.4162
[Stage1] epoch 46 batch 300/560 loss=0.4226
[Stage1] epoch 46 batch 350/560 loss=0.4318
[Stage1] epoch 46 batch 400/560 loss=0.4287
[Stage1] epoch 46 batch 450/560 loss=0.5066
[Stage1] epoch 46 batch 500/560 loss=0.7958
[Stage1] epoch 46 batch 550/560 loss=0.4153
[Stage1][46/150] train=0.4387 valDice(vessel)=0.3931 valClDice(vessel)=0.5409
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 47 batch 1/560 loss=0.4677
[Stage1] epoch 47 batch 50/560 loss=0.4194
[Stage1] epoch 47 batch 100/560 loss=0.4106
[Stage1] epoch 47 batch 150/560 loss=0.4039
[Stage1] epoch 47 batch 200/560 loss=0.4468
[Stage1] epoch 47 batch 250/560 loss=0.4160
[Stage1] epoch 47 batch 300/560 loss=0.4199
[Stage1] epoch 47 batch 350/560 loss=0.4770
[Stage1] epoch 47 batch 400/560 loss=0.4635
[Stage1] epoch 47 batch 450/560 loss=0.4232
[Stage1] epoch 47 batch 500/560 loss=0.4367
[Stage1] epoch 47 batch 550/560 loss=0.4110
[Stage1][47/150] train=0.4372 valDice(vessel)=0.3960 valClDice(vessel)=0.5806
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 48 batch 1/560 loss=0.4209
[Stage1] epoch 48 batch 50/560 loss=0.4420
[Stage1] epoch 48 batch 100/560 loss=0.4135
[Stage1] epoch 48 batch 150/560 loss=0.4126
[Stage1] epoch 48 batch 200/560 loss=0.5021
[Stage1] epoch 48 batch 250/560 loss=0.4252
[Stage1] epoch 48 batch 300/560 loss=0.4159
[Stage1] epoch 48 batch 350/560 loss=0.4449
[Stage1] epoch 48 batch 400/560 loss=0.4316
[Stage1] epoch 48 batch 450/560 loss=0.4153
[Stage1] epoch 48 batch 500/560 loss=0.4243
[Stage1] epoch 48 batch 550/560 loss=0.4151
[Stage1][48/150] train=0.4370 valDice(vessel)=0.4564 valClDice(vessel)=0.6396
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 49 batch 1/560 loss=0.4148
[Stage1] epoch 49 batch 50/560 loss=0.4385
[Stage1] epoch 49 batch 100/560 loss=0.4281
[Stage1] epoch 49 batch 150/560 loss=0.4479
[Stage1] epoch 49 batch 200/560 loss=0.4185
[Stage1] epoch 49 batch 250/560 loss=0.4053
[Stage1] epoch 49 batch 300/560 loss=0.4103
[Stage1] epoch 49 batch 350/560 loss=0.4540
[Stage1] epoch 49 batch 400/560 loss=0.4124
[Stage1] epoch 49 batch 450/560 loss=0.4387
[Stage1] epoch 49 batch 500/560 loss=0.4523
[Stage1] epoch 49 batch 550/560 loss=0.4216
[Stage1][49/150] train=0.4354 valDice(vessel)=0.4498 valClDice(vessel)=0.6946
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 50 batch 1/560 loss=0.4459
[Stage1] epoch 50 batch 50/560 loss=0.4112
[Stage1] epoch 50 batch 100/560 loss=0.5075
[Stage1] epoch 50 batch 150/560 loss=0.4359
[Stage1] epoch 50 batch 200/560 loss=0.4222
[Stage1] epoch 50 batch 250/560 loss=0.4470
[Stage1] epoch 50 batch 300/560 loss=0.4648
[Stage1] epoch 50 batch 350/560 loss=0.4066
[Stage1] epoch 50 batch 400/560 loss=0.4135
[Stage1] epoch 50 batch 450/560 loss=0.3953
[Stage1] epoch 50 batch 500/560 loss=0.4020
[Stage1] epoch 50 batch 550/560 loss=0.4351
[Stage1][50/150] train=0.4351 valDice(vessel)=0.4043 valClDice(vessel)=0.5727
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 51 batch 1/560 loss=0.4070
[Stage1] epoch 51 batch 50/560 loss=0.4418
[Stage1] epoch 51 batch 100/560 loss=0.4918
[Stage1] epoch 51 batch 150/560 loss=0.4189
[Stage1] epoch 51 batch 200/560 loss=0.4197
[Stage1] epoch 51 batch 250/560 loss=0.4599
[Stage1] epoch 51 batch 300/560 loss=0.4313
[Stage1] epoch 51 batch 350/560 loss=0.4283
[Stage1] epoch 51 batch 400/560 loss=0.4743
[Stage1] epoch 51 batch 450/560 loss=0.4209
[Stage1] epoch 51 batch 500/560 loss=0.4137
[Stage1] epoch 51 batch 550/560 loss=0.4359
[Stage1][51/150] train=0.4373 valDice(vessel)=0.3840 valClDice(vessel)=0.5322
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 52 batch 1/560 loss=0.4478
[Stage1] epoch 52 batch 50/560 loss=0.4284
[Stage1] epoch 52 batch 100/560 loss=0.4335
[Stage1] epoch 52 batch 150/560 loss=0.3944
[Stage1] epoch 52 batch 200/560 loss=0.4410
[Stage1] epoch 52 batch 250/560 loss=0.4043
[Stage1] epoch 52 batch 300/560 loss=0.4600
[Stage1] epoch 52 batch 350/560 loss=0.4703
[Stage1] epoch 52 batch 400/560 loss=0.3950
[Stage1] epoch 52 batch 450/560 loss=0.4599
[Stage1] epoch 52 batch 500/560 loss=0.4426
[Stage1] epoch 52 batch 550/560 loss=0.4387
[Stage1][52/150] train=0.4371 valDice(vessel)=0.4366 valClDice(vessel)=0.6700
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 53 batch 1/560 loss=0.4209
[Stage1] epoch 53 batch 50/560 loss=0.4622
[Stage1] epoch 53 batch 100/560 loss=0.4243
[Stage1] epoch 53 batch 150/560 loss=0.3981
[Stage1] epoch 53 batch 200/560 loss=0.4249
[Stage1] epoch 53 batch 250/560 loss=0.4205
[Stage1] epoch 53 batch 300/560 loss=0.4435
[Stage1] epoch 53 batch 350/560 loss=0.4185
[Stage1] epoch 53 batch 400/560 loss=0.4526
[Stage1] epoch 53 batch 450/560 loss=0.4104
[Stage1] epoch 53 batch 500/560 loss=0.4267
[Stage1] epoch 53 batch 550/560 loss=0.4201
[Stage1][53/150] train=0.4337 valDice(vessel)=0.4383 valClDice(vessel)=0.6047
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 54 batch 1/560 loss=0.4299
[Stage1] epoch 54 batch 50/560 loss=0.4583
[Stage1] epoch 54 batch 100/560 loss=0.4141
[Stage1] epoch 54 batch 150/560 loss=0.4567
[Stage1] epoch 54 batch 200/560 loss=0.4286
[Stage1] epoch 54 batch 250/560 loss=0.4318
[Stage1] epoch 54 batch 300/560 loss=0.4652
[Stage1] epoch 54 batch 350/560 loss=0.4343
[Stage1] epoch 54 batch 400/560 loss=0.4059
[Stage1] epoch 54 batch 450/560 loss=0.4443
[Stage1] epoch 54 batch 500/560 loss=0.4317
[Stage1] epoch 54 batch 550/560 loss=0.4046
[Stage1][54/150] train=0.4342 valDice(vessel)=0.4306 valClDice(vessel)=0.5922
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 55 batch 1/560 loss=0.4378
[Stage1] epoch 55 batch 50/560 loss=0.4333
[Stage1] epoch 55 batch 100/560 loss=0.4328
[Stage1] epoch 55 batch 150/560 loss=0.4055
[Stage1] epoch 55 batch 200/560 loss=0.4334
[Stage1] epoch 55 batch 250/560 loss=0.4331
[Stage1] epoch 55 batch 300/560 loss=0.4180
[Stage1] epoch 55 batch 350/560 loss=0.4127
[Stage1] epoch 55 batch 400/560 loss=0.4078
[Stage1] epoch 55 batch 450/560 loss=0.4141
[Stage1] epoch 55 batch 500/560 loss=0.4496
[Stage1] epoch 55 batch 550/560 loss=0.5666
[Stage1][55/150] train=0.4375 valDice(vessel)=0.4175 valClDice(vessel)=0.6203
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 56 batch 1/560 loss=0.4448
[Stage1] epoch 56 batch 50/560 loss=0.4309
[Stage1] epoch 56 batch 100/560 loss=0.4101
[Stage1] epoch 56 batch 150/560 loss=0.4114
[Stage1] epoch 56 batch 200/560 loss=0.8882
[Stage1] epoch 56 batch 250/560 loss=0.4141
[Stage1] epoch 56 batch 300/560 loss=0.8004
[Stage1] epoch 56 batch 350/560 loss=0.4900
[Stage1] epoch 56 batch 400/560 loss=0.4196
[Stage1] epoch 56 batch 450/560 loss=0.4176
[Stage1] epoch 56 batch 500/560 loss=0.4074
[Stage1] epoch 56 batch 550/560 loss=0.4155
[Stage1][56/150] train=0.4374 valDice(vessel)=0.4402 valClDice(vessel)=0.6633
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 57 batch 1/560 loss=0.4066
[Stage1] epoch 57 batch 50/560 loss=0.4341
[Stage1] epoch 57 batch 100/560 loss=0.4175
[Stage1] epoch 57 batch 150/560 loss=0.4082
[Stage1] epoch 57 batch 200/560 loss=0.4717
[Stage1] epoch 57 batch 250/560 loss=0.4479
[Stage1] epoch 57 batch 300/560 loss=0.4241
[Stage1] epoch 57 batch 350/560 loss=0.4261
[Stage1] epoch 57 batch 400/560 loss=0.4326
[Stage1] epoch 57 batch 450/560 loss=0.4256
[Stage1] epoch 57 batch 500/560 loss=0.4306
[Stage1] epoch 57 batch 550/560 loss=0.4303
[Stage1][57/150] train=0.4330 valDice(vessel)=0.4405 valClDice(vessel)=0.6461
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 58 batch 1/560 loss=0.4041
[Stage1] epoch 58 batch 50/560 loss=0.4147
[Stage1] epoch 58 batch 100/560 loss=0.4240
[Stage1] epoch 58 batch 150/560 loss=0.4270
[Stage1] epoch 58 batch 200/560 loss=0.4265
[Stage1] epoch 58 batch 250/560 loss=0.4219
[Stage1] epoch 58 batch 300/560 loss=0.4317
[Stage1] epoch 58 batch 350/560 loss=0.4468
[Stage1] epoch 58 batch 400/560 loss=0.4060
[Stage1] epoch 58 batch 450/560 loss=0.4083
[Stage1] epoch 58 batch 500/560 loss=0.4104
[Stage1] epoch 58 batch 550/560 loss=0.4207
[Stage1][58/150] train=0.4329 valDice(vessel)=0.4763 valClDice(vessel)=0.6728
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 59 batch 1/560 loss=0.4195
[Stage1] epoch 59 batch 50/560 loss=0.4349
[Stage1] epoch 59 batch 100/560 loss=0.4093
[Stage1] epoch 59 batch 150/560 loss=0.4375
[Stage1] epoch 59 batch 200/560 loss=0.4244
[Stage1] epoch 59 batch 250/560 loss=0.4405
[Stage1] epoch 59 batch 300/560 loss=0.4464
[Stage1] epoch 59 batch 350/560 loss=0.4245
[Stage1] epoch 59 batch 400/560 loss=0.3871
[Stage1] epoch 59 batch 450/560 loss=0.3978
[Stage1] epoch 59 batch 500/560 loss=0.4161
[Stage1] epoch 59 batch 550/560 loss=0.4290
[Stage1][59/150] train=0.4333 valDice(vessel)=0.4191 valClDice(vessel)=0.6120
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 60 batch 1/560 loss=0.4702
[Stage1] epoch 60 batch 50/560 loss=0.4893
[Stage1] epoch 60 batch 100/560 loss=0.4103
[Stage1] epoch 60 batch 150/560 loss=0.4351
[Stage1] epoch 60 batch 200/560 loss=0.4392
[Stage1] epoch 60 batch 250/560 loss=0.4194
[Stage1] epoch 60 batch 300/560 loss=0.4063
[Stage1] epoch 60 batch 350/560 loss=0.4093
[Stage1] epoch 60 batch 400/560 loss=0.4555
[Stage1] epoch 60 batch 450/560 loss=0.4239
[Stage1] epoch 60 batch 500/560 loss=0.4117
[Stage1] epoch 60 batch 550/560 loss=0.4880
[Stage1][60/150] train=0.4325 valDice(vessel)=0.4322 valClDice(vessel)=0.6182
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 61 batch 1/560 loss=0.4368
[Stage1] epoch 61 batch 50/560 loss=0.4004
[Stage1] epoch 61 batch 100/560 loss=0.3930
[Stage1] epoch 61 batch 150/560 loss=0.4341
[Stage1] epoch 61 batch 200/560 loss=0.4397
[Stage1] epoch 61 batch 250/560 loss=0.4414
[Stage1] epoch 61 batch 300/560 loss=0.4544
[Stage1] epoch 61 batch 350/560 loss=0.4114
[Stage1] epoch 61 batch 400/560 loss=0.3965
[Stage1] epoch 61 batch 450/560 loss=0.4247
[Stage1] epoch 61 batch 500/560 loss=0.9141
[Stage1] epoch 61 batch 550/560 loss=0.4200
[Stage1][61/150] train=0.4333 valDice(vessel)=0.3499 valClDice(vessel)=0.5073
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 62 batch 1/560 loss=0.4202
[Stage1] epoch 62 batch 50/560 loss=0.4578
[Stage1] epoch 62 batch 100/560 loss=0.4280
[Stage1] epoch 62 batch 150/560 loss=0.4194
[Stage1] epoch 62 batch 200/560 loss=0.4201
[Stage1] epoch 62 batch 250/560 loss=0.4122
[Stage1] epoch 62 batch 300/560 loss=0.4013
[Stage1] epoch 62 batch 350/560 loss=0.7350
[Stage1] epoch 62 batch 400/560 loss=0.4136
[Stage1] epoch 62 batch 450/560 loss=0.4202
[Stage1] epoch 62 batch 500/560 loss=0.4618
[Stage1] epoch 62 batch 550/560 loss=0.4154
[Stage1][62/150] train=0.4327 valDice(vessel)=0.4118 valClDice(vessel)=0.5807
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 63 batch 1/560 loss=0.4600
[Stage1] epoch 63 batch 50/560 loss=0.4467
[Stage1] epoch 63 batch 100/560 loss=0.4256
[Stage1] epoch 63 batch 150/560 loss=0.4255
[Stage1] epoch 63 batch 200/560 loss=0.4097
[Stage1] epoch 63 batch 250/560 loss=0.4277
[Stage1] epoch 63 batch 300/560 loss=0.4138
[Stage1] epoch 63 batch 350/560 loss=0.4295
[Stage1] epoch 63 batch 400/560 loss=0.3960
[Stage1] epoch 63 batch 450/560 loss=0.4107
[Stage1] epoch 63 batch 500/560 loss=0.4253
[Stage1] epoch 63 batch 550/560 loss=0.4142
[Stage1][63/150] train=0.4335 valDice(vessel)=0.4448 valClDice(vessel)=0.6242
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 64 batch 1/560 loss=0.4396
[Stage1] epoch 64 batch 50/560 loss=0.4452
[Stage1] epoch 64 batch 100/560 loss=0.4031
[Stage1] epoch 64 batch 150/560 loss=0.4130
[Stage1] epoch 64 batch 200/560 loss=0.4241
[Stage1] epoch 64 batch 250/560 loss=0.4186
[Stage1] epoch 64 batch 300/560 loss=0.3910
[Stage1] epoch 64 batch 350/560 loss=0.4060
[Stage1] epoch 64 batch 400/560 loss=0.4396
[Stage1] epoch 64 batch 450/560 loss=0.3996
[Stage1] epoch 64 batch 500/560 loss=0.4483
[Stage1] epoch 64 batch 550/560 loss=0.4666
[Stage1][64/150] train=0.4319 valDice(vessel)=0.4551 valClDice(vessel)=0.6514
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 65 batch 1/560 loss=0.4020
[Stage1] epoch 65 batch 50/560 loss=0.4230
[Stage1] epoch 65 batch 100/560 loss=0.4084
[Stage1] epoch 65 batch 150/560 loss=0.4479
[Stage1] epoch 65 batch 200/560 loss=0.4030
[Stage1] epoch 65 batch 250/560 loss=0.4250
[Stage1] epoch 65 batch 300/560 loss=0.5227
[Stage1] epoch 65 batch 350/560 loss=0.4117
[Stage1] epoch 65 batch 400/560 loss=0.4171
[Stage1] epoch 65 batch 450/560 loss=0.4195
[Stage1] epoch 65 batch 500/560 loss=0.4139
[Stage1] epoch 65 batch 550/560 loss=0.4134
[Stage1][65/150] train=0.4327 valDice(vessel)=0.4511 valClDice(vessel)=0.6665
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 66 batch 1/560 loss=0.4429
[Stage1] epoch 66 batch 50/560 loss=0.4358
[Stage1] epoch 66 batch 100/560 loss=0.3888
[Stage1] epoch 66 batch 150/560 loss=0.4479
[Stage1] epoch 66 batch 200/560 loss=0.4337
[Stage1] epoch 66 batch 250/560 loss=0.4412
[Stage1] epoch 66 batch 300/560 loss=0.4190
[Stage1] epoch 66 batch 350/560 loss=0.4104
[Stage1] epoch 66 batch 400/560 loss=0.4434
[Stage1] epoch 66 batch 450/560 loss=0.4205
[Stage1] epoch 66 batch 500/560 loss=0.4102
[Stage1] epoch 66 batch 550/560 loss=0.4036
[Stage1][66/150] train=0.4341 valDice(vessel)=0.4780 valClDice(vessel)=0.6881
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 67 batch 1/560 loss=0.3986
[Stage1] epoch 67 batch 50/560 loss=0.4072
[Stage1] epoch 67 batch 100/560 loss=0.4146
[Stage1] epoch 67 batch 150/560 loss=0.4467
[Stage1] epoch 67 batch 200/560 loss=0.4207
[Stage1] epoch 67 batch 250/560 loss=0.4319
[Stage1] epoch 67 batch 300/560 loss=0.4069
[Stage1] epoch 67 batch 350/560 loss=0.4366
[Stage1] epoch 67 batch 400/560 loss=0.4090
[Stage1] epoch 67 batch 450/560 loss=0.4009
[Stage1] epoch 67 batch 500/560 loss=0.4122
[Stage1] epoch 67 batch 550/560 loss=0.4585
[Stage1][67/150] train=0.4324 valDice(vessel)=0.4347 valClDice(vessel)=0.6849
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 68 batch 1/560 loss=0.4219
[Stage1] epoch 68 batch 50/560 loss=0.4076
[Stage1] epoch 68 batch 100/560 loss=0.4135
[Stage1] epoch 68 batch 150/560 loss=0.4544
[Stage1] epoch 68 batch 200/560 loss=0.4194
[Stage1] epoch 68 batch 250/560 loss=0.4089
[Stage1] epoch 68 batch 300/560 loss=0.4125
[Stage1] epoch 68 batch 350/560 loss=0.4256
[Stage1] epoch 68 batch 400/560 loss=0.4244
[Stage1] epoch 68 batch 450/560 loss=0.4309
[Stage1] epoch 68 batch 500/560 loss=0.4190
[Stage1] epoch 68 batch 550/560 loss=0.4658
[Stage1][68/150] train=0.4320 valDice(vessel)=0.4538 valClDice(vessel)=0.6488
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 69 batch 1/560 loss=0.4227
[Stage1] epoch 69 batch 50/560 loss=0.4329
[Stage1] epoch 69 batch 100/560 loss=0.4223
[Stage1] epoch 69 batch 150/560 loss=0.4259
[Stage1] epoch 69 batch 200/560 loss=0.4017
[Stage1] epoch 69 batch 250/560 loss=0.4234
[Stage1] epoch 69 batch 300/560 loss=0.4326
[Stage1] epoch 69 batch 350/560 loss=0.4510
[Stage1] epoch 69 batch 400/560 loss=0.4105
[Stage1] epoch 69 batch 450/560 loss=0.4028
[Stage1] epoch 69 batch 500/560 loss=0.4006
[Stage1] epoch 69 batch 550/560 loss=0.4351
[Stage1][69/150] train=0.4306 valDice(vessel)=0.4674 valClDice(vessel)=0.6559
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 70 batch 1/560 loss=0.4243
[Stage1] epoch 70 batch 50/560 loss=0.4223
[Stage1] epoch 70 batch 100/560 loss=0.4153
[Stage1] epoch 70 batch 150/560 loss=0.4025
[Stage1] epoch 70 batch 200/560 loss=0.4404
[Stage1] epoch 70 batch 250/560 loss=0.4096
[Stage1] epoch 70 batch 300/560 loss=0.4667
[Stage1] epoch 70 batch 350/560 loss=0.4297
[Stage1] epoch 70 batch 400/560 loss=0.5033
[Stage1] epoch 70 batch 450/560 loss=0.4421
[Stage1] epoch 70 batch 500/560 loss=0.4250
[Stage1] epoch 70 batch 550/560 loss=0.4183
[Stage1][70/150] train=0.4318 valDice(vessel)=0.4584 valClDice(vessel)=0.6995
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 71 batch 1/560 loss=0.4137
[Stage1] epoch 71 batch 50/560 loss=0.4310
[Stage1] epoch 71 batch 100/560 loss=0.4323
[Stage1] epoch 71 batch 150/560 loss=0.4684
[Stage1] epoch 71 batch 200/560 loss=0.4261
[Stage1] epoch 71 batch 250/560 loss=0.4043
[Stage1] epoch 71 batch 300/560 loss=0.4279
[Stage1] epoch 71 batch 350/560 loss=0.4212
[Stage1] epoch 71 batch 400/560 loss=0.4479
[Stage1] epoch 71 batch 450/560 loss=0.4207
[Stage1] epoch 71 batch 500/560 loss=0.4050
[Stage1] epoch 71 batch 550/560 loss=0.4595
[Stage1][71/150] train=0.4304 valDice(vessel)=0.4835 valClDice(vessel)=0.6796
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 72 batch 1/560 loss=0.4193
[Stage1] epoch 72 batch 50/560 loss=0.4187
[Stage1] epoch 72 batch 100/560 loss=0.4049
[Stage1] epoch 72 batch 150/560 loss=0.4283
[Stage1] epoch 72 batch 200/560 loss=0.4233
[Stage1] epoch 72 batch 250/560 loss=0.4632
[Stage1] epoch 72 batch 300/560 loss=0.4178
[Stage1] epoch 72 batch 350/560 loss=0.4268
[Stage1] epoch 72 batch 400/560 loss=0.4456
[Stage1] epoch 72 batch 450/560 loss=0.4374
[Stage1] epoch 72 batch 500/560 loss=0.4127
[Stage1] epoch 72 batch 550/560 loss=0.4159
[Stage1][72/150] train=0.4306 valDice(vessel)=0.5134 valClDice(vessel)=0.7696
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 73 batch 1/560 loss=0.4726
[Stage1] epoch 73 batch 50/560 loss=0.4153
[Stage1] epoch 73 batch 100/560 loss=0.4259
[Stage1] epoch 73 batch 150/560 loss=0.3990
[Stage1] epoch 73 batch 200/560 loss=0.4498
[Stage1] epoch 73 batch 250/560 loss=0.4070
[Stage1] epoch 73 batch 300/560 loss=0.4114
[Stage1] epoch 73 batch 350/560 loss=0.3951
[Stage1] epoch 73 batch 400/560 loss=0.4436
[Stage1] epoch 73 batch 450/560 loss=0.4113
[Stage1] epoch 73 batch 500/560 loss=0.4241
[Stage1] epoch 73 batch 550/560 loss=0.4045
[Stage1][73/150] train=0.4298 valDice(vessel)=0.4825 valClDice(vessel)=0.7258
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 74 batch 1/560 loss=0.4079
[Stage1] epoch 74 batch 50/560 loss=0.3862
[Stage1] epoch 74 batch 100/560 loss=0.3978
[Stage1] epoch 74 batch 150/560 loss=0.4119
[Stage1] epoch 74 batch 200/560 loss=0.4796
[Stage1] epoch 74 batch 250/560 loss=0.4220
[Stage1] epoch 74 batch 300/560 loss=0.4116
[Stage1] epoch 74 batch 350/560 loss=0.4142
[Stage1] epoch 74 batch 400/560 loss=0.4124
[Stage1] epoch 74 batch 450/560 loss=0.4024
[Stage1] epoch 74 batch 500/560 loss=0.4095
[Stage1] epoch 74 batch 550/560 loss=0.4070
[Stage1][74/150] train=0.4300 valDice(vessel)=0.5156 valClDice(vessel)=0.7089
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 75 batch 1/560 loss=0.4101
[Stage1] epoch 75 batch 50/560 loss=0.4468
[Stage1] epoch 75 batch 100/560 loss=0.4115
[Stage1] epoch 75 batch 150/560 loss=0.3964
[Stage1] epoch 75 batch 200/560 loss=0.3983
[Stage1] epoch 75 batch 250/560 loss=0.4416
[Stage1] epoch 75 batch 300/560 loss=0.4090
[Stage1] epoch 75 batch 350/560 loss=0.4103
[Stage1] epoch 75 batch 400/560 loss=0.4020
[Stage1] epoch 75 batch 450/560 loss=0.4075
[Stage1] epoch 75 batch 500/560 loss=0.4641
[Stage1] epoch 75 batch 550/560 loss=0.4525
[Stage1][75/150] train=0.4300 valDice(vessel)=0.4750 valClDice(vessel)=0.7144
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 76 batch 1/560 loss=0.4002
[Stage1] epoch 76 batch 50/560 loss=0.3969
[Stage1] epoch 76 batch 100/560 loss=0.4350
[Stage1] epoch 76 batch 150/560 loss=0.4485
[Stage1] epoch 76 batch 200/560 loss=0.4416
[Stage1] epoch 76 batch 250/560 loss=0.4261
[Stage1] epoch 76 batch 300/560 loss=0.4367
[Stage1] epoch 76 batch 350/560 loss=0.4236
[Stage1] epoch 76 batch 400/560 loss=0.4269
[Stage1] epoch 76 batch 450/560 loss=0.4539
[Stage1] epoch 76 batch 500/560 loss=0.4156
[Stage1] epoch 76 batch 550/560 loss=0.4216
[Stage1][76/150] train=0.4302 valDice(vessel)=0.4893 valClDice(vessel)=0.6900
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 77 batch 1/560 loss=0.4333
[Stage1] epoch 77 batch 50/560 loss=0.4556
[Stage1] epoch 77 batch 100/560 loss=0.3948
[Stage1] epoch 77 batch 150/560 loss=0.4527
[Stage1] epoch 77 batch 200/560 loss=0.4149
[Stage1] epoch 77 batch 250/560 loss=0.4402
[Stage1] epoch 77 batch 300/560 loss=0.4260
[Stage1] epoch 77 batch 350/560 loss=0.4369
[Stage1] epoch 77 batch 400/560 loss=0.4004
[Stage1] epoch 77 batch 450/560 loss=0.4334
[Stage1] epoch 77 batch 500/560 loss=0.4587
[Stage1] epoch 77 batch 550/560 loss=0.4405
[Stage1][77/150] train=0.4297 valDice(vessel)=0.4749 valClDice(vessel)=0.6564
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 78 batch 1/560 loss=0.4089
[Stage1] epoch 78 batch 50/560 loss=0.4098
[Stage1] epoch 78 batch 100/560 loss=0.4034
[Stage1] epoch 78 batch 150/560 loss=0.4175
[Stage1] epoch 78 batch 200/560 loss=0.4842
[Stage1] epoch 78 batch 250/560 loss=0.4344
[Stage1] epoch 78 batch 300/560 loss=0.4122
[Stage1] epoch 78 batch 350/560 loss=0.4106
[Stage1] epoch 78 batch 400/560 loss=0.4392
[Stage1] epoch 78 batch 450/560 loss=0.4290
[Stage1] epoch 78 batch 500/560 loss=0.4080
[Stage1] epoch 78 batch 550/560 loss=0.4088
[Stage1][78/150] train=0.4286 valDice(vessel)=0.5075 valClDice(vessel)=0.7449
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 79 batch 1/560 loss=0.3905
[Stage1] epoch 79 batch 50/560 loss=0.4583
[Stage1] epoch 79 batch 100/560 loss=0.4094
[Stage1] epoch 79 batch 150/560 loss=0.4155
[Stage1] epoch 79 batch 200/560 loss=0.4167
[Stage1] epoch 79 batch 250/560 loss=0.4291
[Stage1] epoch 79 batch 300/560 loss=0.4392
[Stage1] epoch 79 batch 350/560 loss=0.4020
[Stage1] epoch 79 batch 400/560 loss=0.4173
[Stage1] epoch 79 batch 450/560 loss=0.4284
[Stage1] epoch 79 batch 500/560 loss=0.4084
[Stage1] epoch 79 batch 550/560 loss=0.4455
[Stage1][79/150] train=0.4283 valDice(vessel)=0.4732 valClDice(vessel)=0.6522
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 80 batch 1/560 loss=0.4134
[Stage1] epoch 80 batch 50/560 loss=0.4264
[Stage1] epoch 80 batch 100/560 loss=0.4177
[Stage1] epoch 80 batch 150/560 loss=0.4698
[Stage1] epoch 80 batch 200/560 loss=0.4584
[Stage1] epoch 80 batch 250/560 loss=0.4150
[Stage1] epoch 80 batch 300/560 loss=0.4109
[Stage1] epoch 80 batch 350/560 loss=0.4195
[Stage1] epoch 80 batch 400/560 loss=0.4261
[Stage1] epoch 80 batch 450/560 loss=0.4165
[Stage1] epoch 80 batch 500/560 loss=0.4407
[Stage1] epoch 80 batch 550/560 loss=0.4029
[Stage1][80/150] train=0.4291 valDice(vessel)=0.4467 valClDice(vessel)=0.6527
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 81 batch 1/560 loss=0.4021
[Stage1] epoch 81 batch 50/560 loss=0.4053
[Stage1] epoch 81 batch 100/560 loss=0.4163
[Stage1] epoch 81 batch 150/560 loss=0.4167
[Stage1] epoch 81 batch 200/560 loss=0.4167
[Stage1] epoch 81 batch 250/560 loss=0.8144
[Stage1] epoch 81 batch 300/560 loss=0.4210
[Stage1] epoch 81 batch 350/560 loss=0.4075
[Stage1] epoch 81 batch 400/560 loss=0.4240
[Stage1] epoch 81 batch 450/560 loss=0.3893
[Stage1] epoch 81 batch 500/560 loss=0.4141
[Stage1] epoch 81 batch 550/560 loss=0.4097
[Stage1][81/150] train=0.4301 valDice(vessel)=0.4437 valClDice(vessel)=0.6301
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 82 batch 1/560 loss=0.4449
[Stage1] epoch 82 batch 50/560 loss=0.4027
[Stage1] epoch 82 batch 100/560 loss=0.4131
[Stage1] epoch 82 batch 150/560 loss=0.4111
[Stage1] epoch 82 batch 200/560 loss=0.7778
[Stage1] epoch 82 batch 250/560 loss=0.3924
[Stage1] epoch 82 batch 300/560 loss=0.4164
[Stage1] epoch 82 batch 350/560 loss=0.4009
[Stage1] epoch 82 batch 400/560 loss=0.4106
[Stage1] epoch 82 batch 450/560 loss=0.4549
[Stage1] epoch 82 batch 500/560 loss=0.4159
[Stage1] epoch 82 batch 550/560 loss=0.4094
[Stage1][82/150] train=0.4312 valDice(vessel)=0.4828 valClDice(vessel)=0.7488
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 83 batch 1/560 loss=0.4553
[Stage1] epoch 83 batch 50/560 loss=0.4154
[Stage1] epoch 83 batch 100/560 loss=0.4340
[Stage1] epoch 83 batch 150/560 loss=0.4019
[Stage1] epoch 83 batch 200/560 loss=0.4661
[Stage1] epoch 83 batch 250/560 loss=0.4079
[Stage1] epoch 83 batch 300/560 loss=0.4206
[Stage1] epoch 83 batch 350/560 loss=0.4357
[Stage1] epoch 83 batch 400/560 loss=0.4435
[Stage1] epoch 83 batch 450/560 loss=0.4153
[Stage1] epoch 83 batch 500/560 loss=0.4323
[Stage1] epoch 83 batch 550/560 loss=0.4155
[Stage1][83/150] train=0.4289 valDice(vessel)=0.4592 valClDice(vessel)=0.6767
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 84 batch 1/560 loss=0.4217
[Stage1] epoch 84 batch 50/560 loss=0.4376
[Stage1] epoch 84 batch 100/560 loss=0.3941
[Stage1] epoch 84 batch 150/560 loss=0.4278
[Stage1] epoch 84 batch 200/560 loss=0.4058
[Stage1] epoch 84 batch 250/560 loss=0.4232
[Stage1] epoch 84 batch 300/560 loss=0.4133
[Stage1] epoch 84 batch 350/560 loss=0.4255
[Stage1] epoch 84 batch 400/560 loss=0.3937
[Stage1] epoch 84 batch 450/560 loss=0.4310
[Stage1] epoch 84 batch 500/560 loss=0.4219
[Stage1] epoch 84 batch 550/560 loss=0.4178
[Stage1][84/150] train=0.4282 valDice(vessel)=0.4713 valClDice(vessel)=0.7205
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 85 batch 1/560 loss=0.4289
[Stage1] epoch 85 batch 50/560 loss=0.4166
[Stage1] epoch 85 batch 100/560 loss=0.4290
[Stage1] epoch 85 batch 150/560 loss=0.4137
[Stage1] epoch 85 batch 200/560 loss=0.4309
[Stage1] epoch 85 batch 250/560 loss=0.4265
[Stage1] epoch 85 batch 300/560 loss=0.3936
[Stage1] epoch 85 batch 350/560 loss=0.4850
[Stage1] epoch 85 batch 400/560 loss=0.4043
[Stage1] epoch 85 batch 450/560 loss=0.4282
[Stage1] epoch 85 batch 500/560 loss=0.4225
[Stage1] epoch 85 batch 550/560 loss=0.4352
[Stage1][85/150] train=0.4292 valDice(vessel)=0.4324 valClDice(vessel)=0.6523
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 86 batch 1/560 loss=0.4241
[Stage1] epoch 86 batch 50/560 loss=0.4115
[Stage1] epoch 86 batch 100/560 loss=0.4209
[Stage1] epoch 86 batch 150/560 loss=0.4146
[Stage1] epoch 86 batch 200/560 loss=0.4168
[Stage1] epoch 86 batch 250/560 loss=0.4143
[Stage1] epoch 86 batch 300/560 loss=0.4105
[Stage1] epoch 86 batch 350/560 loss=0.4210
[Stage1] epoch 86 batch 400/560 loss=0.4145
[Stage1] epoch 86 batch 450/560 loss=0.4486
[Stage1] epoch 86 batch 500/560 loss=0.4062
[Stage1] epoch 86 batch 550/560 loss=0.4075
[Stage1][86/150] train=0.4284 valDice(vessel)=0.4574 valClDice(vessel)=0.6656
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 87 batch 1/560 loss=0.3981
[Stage1] epoch 87 batch 50/560 loss=0.4189
[Stage1] epoch 87 batch 100/560 loss=0.4022
[Stage1] epoch 87 batch 150/560 loss=0.4718
[Stage1] epoch 87 batch 200/560 loss=0.4194
[Stage1] epoch 87 batch 250/560 loss=0.4072
[Stage1] epoch 87 batch 300/560 loss=0.3980
[Stage1] epoch 87 batch 350/560 loss=0.4045
[Stage1] epoch 87 batch 400/560 loss=0.4509
[Stage1] epoch 87 batch 450/560 loss=0.4000
[Stage1] epoch 87 batch 500/560 loss=0.4075
[Stage1] epoch 87 batch 550/560 loss=0.4239
[Stage1][87/150] train=0.4271 valDice(vessel)=0.4844 valClDice(vessel)=0.7402
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 88 batch 1/560 loss=0.4162
[Stage1] epoch 88 batch 50/560 loss=0.3901
[Stage1] epoch 88 batch 100/560 loss=0.4226
[Stage1] epoch 88 batch 150/560 loss=0.4152
[Stage1] epoch 88 batch 200/560 loss=0.4691
[Stage1] epoch 88 batch 250/560 loss=0.4048
[Stage1] epoch 88 batch 300/560 loss=0.4034
[Stage1] epoch 88 batch 350/560 loss=0.3965
[Stage1] epoch 88 batch 400/560 loss=0.4042
[Stage1] epoch 88 batch 450/560 loss=0.4545
[Stage1] epoch 88 batch 500/560 loss=0.4148
[Stage1] epoch 88 batch 550/560 loss=0.4548
[Stage1][88/150] train=0.4266 valDice(vessel)=0.4538 valClDice(vessel)=0.6528
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 89 batch 1/560 loss=0.4217
[Stage1] epoch 89 batch 50/560 loss=0.4074
[Stage1] epoch 89 batch 100/560 loss=0.4159
[Stage1] epoch 89 batch 150/560 loss=0.8292
[Stage1] epoch 89 batch 200/560 loss=0.3980
[Stage1] epoch 89 batch 250/560 loss=0.4152
[Stage1] epoch 89 batch 300/560 loss=0.4066
[Stage1] epoch 89 batch 350/560 loss=0.4008
[Stage1] epoch 89 batch 400/560 loss=0.4135
[Stage1] epoch 89 batch 450/560 loss=0.4318
[Stage1] epoch 89 batch 500/560 loss=0.8578
[Stage1] epoch 89 batch 550/560 loss=0.4510
[Stage1][89/150] train=0.4286 valDice(vessel)=0.4568 valClDice(vessel)=0.6682
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 90 batch 1/560 loss=0.3863
[Stage1] epoch 90 batch 50/560 loss=0.4186
[Stage1] epoch 90 batch 100/560 loss=0.4203
[Stage1] epoch 90 batch 150/560 loss=0.3947
[Stage1] epoch 90 batch 200/560 loss=0.4156
[Stage1] epoch 90 batch 250/560 loss=0.4140
[Stage1] epoch 90 batch 300/560 loss=0.4022
[Stage1] epoch 90 batch 350/560 loss=0.4141
[Stage1] epoch 90 batch 400/560 loss=0.4315
[Stage1] epoch 90 batch 450/560 loss=0.4014
[Stage1] epoch 90 batch 500/560 loss=0.4035
[Stage1] epoch 90 batch 550/560 loss=0.4033
[Stage1][90/150] train=0.4277 valDice(vessel)=0.4594 valClDice(vessel)=0.6645
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 91 batch 1/560 loss=0.8099
[Stage1] epoch 91 batch 50/560 loss=0.4228
[Stage1] epoch 91 batch 100/560 loss=0.4231
[Stage1] epoch 91 batch 150/560 loss=0.4314
[Stage1] epoch 91 batch 200/560 loss=0.4338
[Stage1] epoch 91 batch 250/560 loss=0.4373
[Stage1] epoch 91 batch 300/560 loss=0.4069
[Stage1] epoch 91 batch 350/560 loss=0.4136
[Stage1] epoch 91 batch 400/560 loss=0.4565
[Stage1] epoch 91 batch 450/560 loss=0.4460
[Stage1] epoch 91 batch 500/560 loss=0.4456
[Stage1] epoch 91 batch 550/560 loss=0.4391
[Stage1][91/150] train=0.4275 valDice(vessel)=0.4964 valClDice(vessel)=0.7022
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 92 batch 1/560 loss=0.4112
[Stage1] epoch 92 batch 50/560 loss=0.4221
[Stage1] epoch 92 batch 100/560 loss=0.3989
[Stage1] epoch 92 batch 150/560 loss=0.3971
[Stage1] epoch 92 batch 200/560 loss=0.4768
[Stage1] epoch 92 batch 250/560 loss=0.3931
[Stage1] epoch 92 batch 300/560 loss=0.4300
[Stage1] epoch 92 batch 350/560 loss=0.3870
[Stage1] epoch 92 batch 400/560 loss=0.4105
[Stage1] epoch 92 batch 450/560 loss=0.4126
[Stage1] epoch 92 batch 500/560 loss=0.4269
[Stage1] epoch 92 batch 550/560 loss=0.4089
[Stage1][92/150] train=0.4272 valDice(vessel)=0.4763 valClDice(vessel)=0.7062
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 93 batch 1/560 loss=0.4256
[Stage1] epoch 93 batch 50/560 loss=0.4223
[Stage1] epoch 93 batch 100/560 loss=0.4313
[Stage1] epoch 93 batch 150/560 loss=0.4653
[Stage1] epoch 93 batch 200/560 loss=0.4171
[Stage1] epoch 93 batch 250/560 loss=0.4266
[Stage1] epoch 93 batch 300/560 loss=0.3954
[Stage1] epoch 93 batch 350/560 loss=0.4088
[Stage1] epoch 93 batch 400/560 loss=0.4152
[Stage1] epoch 93 batch 450/560 loss=0.4067
[Stage1] epoch 93 batch 500/560 loss=0.4048
[Stage1] epoch 93 batch 550/560 loss=0.4189
[Stage1][93/150] train=0.4268 valDice(vessel)=0.4549 valClDice(vessel)=0.6875
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 94 batch 1/560 loss=0.4153
[Stage1] epoch 94 batch 50/560 loss=0.4585
[Stage1] epoch 94 batch 100/560 loss=0.4366
[Stage1] epoch 94 batch 150/560 loss=0.4461
[Stage1] epoch 94 batch 200/560 loss=0.4091
[Stage1] epoch 94 batch 250/560 loss=0.4447
[Stage1] epoch 94 batch 300/560 loss=0.4185
[Stage1] epoch 94 batch 350/560 loss=0.4152
[Stage1] epoch 94 batch 400/560 loss=0.3985
[Stage1] epoch 94 batch 450/560 loss=0.4113
[Stage1] epoch 94 batch 500/560 loss=0.4553
[Stage1] epoch 94 batch 550/560 loss=0.4160
[Stage1][94/150] train=0.4264 valDice(vessel)=0.4856 valClDice(vessel)=0.7107
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 95 batch 1/560 loss=0.4452
[Stage1] epoch 95 batch 50/560 loss=0.4517
[Stage1] epoch 95 batch 100/560 loss=0.3921
[Stage1] epoch 95 batch 150/560 loss=0.4080
[Stage1] epoch 95 batch 200/560 loss=0.4380
[Stage1] epoch 95 batch 250/560 loss=0.4122
[Stage1] epoch 95 batch 300/560 loss=0.4301
[Stage1] epoch 95 batch 350/560 loss=0.4172
[Stage1] epoch 95 batch 400/560 loss=0.4546
[Stage1] epoch 95 batch 450/560 loss=0.4190
[Stage1] epoch 95 batch 500/560 loss=0.4531
[Stage1] epoch 95 batch 550/560 loss=0.4212
[Stage1][95/150] train=0.4264 valDice(vessel)=0.4925 valClDice(vessel)=0.7019
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 96 batch 1/560 loss=0.4140
[Stage1] epoch 96 batch 50/560 loss=0.4804
[Stage1] epoch 96 batch 100/560 loss=0.4051
[Stage1] epoch 96 batch 150/560 loss=0.4222
[Stage1] epoch 96 batch 200/560 loss=0.4120
[Stage1] epoch 96 batch 250/560 loss=0.3988
[Stage1] epoch 96 batch 300/560 loss=0.8325
[Stage1] epoch 96 batch 350/560 loss=0.4662
[Stage1] epoch 96 batch 400/560 loss=0.3970
[Stage1] epoch 96 batch 450/560 loss=0.7587
[Stage1] epoch 96 batch 500/560 loss=0.4117
[Stage1] epoch 96 batch 550/560 loss=0.4313
[Stage1][96/150] train=0.4268 valDice(vessel)=0.3963 valClDice(vessel)=0.5431
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 97 batch 1/560 loss=0.3964
[Stage1] epoch 97 batch 50/560 loss=0.4082
[Stage1] epoch 97 batch 100/560 loss=0.3978
[Stage1] epoch 97 batch 150/560 loss=0.4537
[Stage1] epoch 97 batch 200/560 loss=0.4589
[Stage1] epoch 97 batch 250/560 loss=0.4049
[Stage1] epoch 97 batch 300/560 loss=0.7795
[Stage1] epoch 97 batch 350/560 loss=0.4500
[Stage1] epoch 97 batch 400/560 loss=0.4226
[Stage1] epoch 97 batch 450/560 loss=0.4388
[Stage1] epoch 97 batch 500/560 loss=0.4255
[Stage1] epoch 97 batch 550/560 loss=0.4164
[Stage1][97/150] train=0.4274 valDice(vessel)=0.4512 valClDice(vessel)=0.6548
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 98 batch 1/560 loss=0.4384
[Stage1] epoch 98 batch 50/560 loss=0.3966
[Stage1] epoch 98 batch 100/560 loss=0.4087
[Stage1] epoch 98 batch 150/560 loss=0.4162
[Stage1] epoch 98 batch 200/560 loss=0.4098
[Stage1] epoch 98 batch 250/560 loss=0.4831
[Stage1] epoch 98 batch 300/560 loss=0.4161
[Stage1] epoch 98 batch 350/560 loss=0.4338
[Stage1] epoch 98 batch 400/560 loss=0.4058
[Stage1] epoch 98 batch 450/560 loss=0.4006
[Stage1] epoch 98 batch 500/560 loss=0.4110
[Stage1] epoch 98 batch 550/560 loss=0.3960
[Stage1][98/150] train=0.4266 valDice(vessel)=0.4817 valClDice(vessel)=0.7004
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 99 batch 1/560 loss=0.4522
[Stage1] epoch 99 batch 50/560 loss=0.4337
[Stage1] epoch 99 batch 100/560 loss=0.4064
[Stage1] epoch 99 batch 150/560 loss=0.4106
[Stage1] epoch 99 batch 200/560 loss=0.3990
[Stage1] epoch 99 batch 250/560 loss=0.4324
[Stage1] epoch 99 batch 300/560 loss=0.3968
[Stage1] epoch 99 batch 350/560 loss=0.4015
[Stage1] epoch 99 batch 400/560 loss=0.4188
[Stage1] epoch 99 batch 450/560 loss=0.4217
[Stage1] epoch 99 batch 500/560 loss=0.3958
[Stage1] epoch 99 batch 550/560 loss=0.3903
[Stage1][99/150] train=0.4254 valDice(vessel)=0.4929 valClDice(vessel)=0.6855
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 100 batch 1/560 loss=0.4103
[Stage1] epoch 100 batch 50/560 loss=0.3986
[Stage1] epoch 100 batch 100/560 loss=0.4293
[Stage1] epoch 100 batch 150/560 loss=0.4403
[Stage1] epoch 100 batch 200/560 loss=0.4047
[Stage1] epoch 100 batch 250/560 loss=0.3945
[Stage1] epoch 100 batch 300/560 loss=0.4342
[Stage1] epoch 100 batch 350/560 loss=0.4805
[Stage1] epoch 100 batch 400/560 loss=0.4038
[Stage1] epoch 100 batch 450/560 loss=0.3997
[Stage1] epoch 100 batch 500/560 loss=0.4557
[Stage1] epoch 100 batch 550/560 loss=0.4519
[Stage1][100/150] train=0.4270 valDice(vessel)=0.5056 valClDice(vessel)=0.7561
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 101 batch 1/560 loss=0.4126
[Stage1] epoch 101 batch 50/560 loss=0.3977
[Stage1] epoch 101 batch 100/560 loss=0.4038
[Stage1] epoch 101 batch 150/560 loss=0.3977
[Stage1] epoch 101 batch 200/560 loss=0.4142
[Stage1] epoch 101 batch 250/560 loss=0.4386
[Stage1] epoch 101 batch 300/560 loss=0.4089
[Stage1] epoch 101 batch 350/560 loss=0.4117
[Stage1] epoch 101 batch 400/560 loss=0.4191
[Stage1] epoch 101 batch 450/560 loss=0.4038
[Stage1] epoch 101 batch 500/560 loss=0.4130
[Stage1] epoch 101 batch 550/560 loss=0.4407
[Stage1][101/150] train=0.4268 valDice(vessel)=0.5162 valClDice(vessel)=0.7636
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 102 batch 1/560 loss=0.4315
[Stage1] epoch 102 batch 50/560 loss=0.4129
[Stage1] epoch 102 batch 100/560 loss=0.4003
[Stage1] epoch 102 batch 150/560 loss=0.4285
[Stage1] epoch 102 batch 200/560 loss=0.4037
[Stage1] epoch 102 batch 250/560 loss=0.4802
[Stage1] epoch 102 batch 300/560 loss=0.4069
[Stage1] epoch 102 batch 350/560 loss=0.3980
[Stage1] epoch 102 batch 400/560 loss=0.4217
[Stage1] epoch 102 batch 450/560 loss=0.4039
[Stage1] epoch 102 batch 500/560 loss=0.3908
[Stage1] epoch 102 batch 550/560 loss=0.7277
[Stage1][102/150] train=0.4260 valDice(vessel)=0.5157 valClDice(vessel)=0.7790
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 103 batch 1/560 loss=0.4288
[Stage1] epoch 103 batch 50/560 loss=0.3923
[Stage1] epoch 103 batch 100/560 loss=0.4531
[Stage1] epoch 103 batch 150/560 loss=0.4193
[Stage1] epoch 103 batch 200/560 loss=0.4119
[Stage1] epoch 103 batch 250/560 loss=0.4318
[Stage1] epoch 103 batch 300/560 loss=0.4068
[Stage1] epoch 103 batch 350/560 loss=0.4138
[Stage1] epoch 103 batch 400/560 loss=0.8091
[Stage1] epoch 103 batch 450/560 loss=0.4242
[Stage1] epoch 103 batch 500/560 loss=0.4219
[Stage1] epoch 103 batch 550/560 loss=0.3953
[Stage1][103/150] train=0.4272 valDice(vessel)=0.4809 valClDice(vessel)=0.6863
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 104 batch 1/560 loss=0.4233
[Stage1] epoch 104 batch 50/560 loss=0.4573
[Stage1] epoch 104 batch 100/560 loss=0.4100
[Stage1] epoch 104 batch 150/560 loss=0.4667
[Stage1] epoch 104 batch 200/560 loss=0.4294
[Stage1] epoch 104 batch 250/560 loss=0.4301
[Stage1] epoch 104 batch 300/560 loss=0.4105
[Stage1] epoch 104 batch 350/560 loss=0.4477
[Stage1] epoch 104 batch 400/560 loss=0.4069
[Stage1] epoch 104 batch 450/560 loss=0.4146
[Stage1] epoch 104 batch 500/560 loss=0.4141
[Stage1] epoch 104 batch 550/560 loss=0.4076
[Stage1][104/150] train=0.4256 valDice(vessel)=0.4840 valClDice(vessel)=0.7102
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 105 batch 1/560 loss=0.4039
[Stage1] epoch 105 batch 50/560 loss=0.4123
[Stage1] epoch 105 batch 100/560 loss=0.4327
[Stage1] epoch 105 batch 150/560 loss=0.3991
[Stage1] epoch 105 batch 200/560 loss=0.4284
[Stage1] epoch 105 batch 250/560 loss=0.4054
[Stage1] epoch 105 batch 300/560 loss=0.4713
[Stage1] epoch 105 batch 350/560 loss=0.4090
[Stage1] epoch 105 batch 400/560 loss=0.4081
[Stage1] epoch 105 batch 450/560 loss=0.4987
[Stage1] epoch 105 batch 500/560 loss=0.4185
[Stage1] epoch 105 batch 550/560 loss=0.4108
[Stage1][105/150] train=0.4275 valDice(vessel)=0.4659 valClDice(vessel)=0.7106
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 106 batch 1/560 loss=0.4101
[Stage1] epoch 106 batch 50/560 loss=0.4087
[Stage1] epoch 106 batch 100/560 loss=0.3906
[Stage1] epoch 106 batch 150/560 loss=0.4596
[Stage1] epoch 106 batch 200/560 loss=0.4101
[Stage1] epoch 106 batch 250/560 loss=0.4348
[Stage1] epoch 106 batch 300/560 loss=0.4023
[Stage1] epoch 106 batch 350/560 loss=0.4318
[Stage1] epoch 106 batch 400/560 loss=0.4044
[Stage1] epoch 106 batch 450/560 loss=0.4763
[Stage1] epoch 106 batch 500/560 loss=0.4078
[Stage1] epoch 106 batch 550/560 loss=0.4059
[Stage1][106/150] train=0.4255 valDice(vessel)=0.4638 valClDice(vessel)=0.6746
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 107 batch 1/560 loss=0.5345
[Stage1] epoch 107 batch 50/560 loss=0.4496
[Stage1] epoch 107 batch 100/560 loss=0.4210
[Stage1] epoch 107 batch 150/560 loss=0.4203
[Stage1] epoch 107 batch 200/560 loss=0.4056
[Stage1] epoch 107 batch 250/560 loss=0.4251
[Stage1] epoch 107 batch 300/560 loss=0.4228
[Stage1] epoch 107 batch 350/560 loss=0.4435
[Stage1] epoch 107 batch 400/560 loss=0.4523
[Stage1] epoch 107 batch 450/560 loss=0.4294
[Stage1] epoch 107 batch 500/560 loss=0.4120
[Stage1] epoch 107 batch 550/560 loss=0.4225
[Stage1][107/150] train=0.4251 valDice(vessel)=0.4616 valClDice(vessel)=0.6502
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 108 batch 1/560 loss=0.4094
[Stage1] epoch 108 batch 50/560 loss=0.4272
[Stage1] epoch 108 batch 100/560 loss=0.4048
[Stage1] epoch 108 batch 150/560 loss=0.4149
[Stage1] epoch 108 batch 200/560 loss=0.4260
[Stage1] epoch 108 batch 250/560 loss=0.4361
[Stage1] epoch 108 batch 300/560 loss=0.4112
[Stage1] epoch 108 batch 350/560 loss=0.4088
[Stage1] epoch 108 batch 400/560 loss=0.4153
[Stage1] epoch 108 batch 450/560 loss=0.4124
[Stage1] epoch 108 batch 500/560 loss=0.4139
[Stage1] epoch 108 batch 550/560 loss=0.4125
[Stage1][108/150] train=0.4259 valDice(vessel)=0.4595 valClDice(vessel)=0.7079
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 109 batch 1/560 loss=0.7668
[Stage1] epoch 109 batch 50/560 loss=0.4053
[Stage1] epoch 109 batch 100/560 loss=0.4277
[Stage1] epoch 109 batch 150/560 loss=0.4086
[Stage1] epoch 109 batch 200/560 loss=0.3953
[Stage1] epoch 109 batch 250/560 loss=0.4523
[Stage1] epoch 109 batch 300/560 loss=0.3981
[Stage1] epoch 109 batch 350/560 loss=0.4444
[Stage1] epoch 109 batch 400/560 loss=0.4057
[Stage1] epoch 109 batch 450/560 loss=0.4026
[Stage1] epoch 109 batch 500/560 loss=0.4108
[Stage1] epoch 109 batch 550/560 loss=0.4225
[Stage1][109/150] train=0.4242 valDice(vessel)=0.4657 valClDice(vessel)=0.6776
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 110 batch 1/560 loss=0.4256
[Stage1] epoch 110 batch 50/560 loss=0.4247
[Stage1] epoch 110 batch 100/560 loss=0.3893
[Stage1] epoch 110 batch 150/560 loss=0.4157
[Stage1] epoch 110 batch 200/560 loss=0.4072
[Stage1] epoch 110 batch 250/560 loss=0.4328
[Stage1] epoch 110 batch 300/560 loss=0.4164
[Stage1] epoch 110 batch 350/560 loss=0.4283
[Stage1] epoch 110 batch 400/560 loss=0.3984
[Stage1] epoch 110 batch 450/560 loss=0.4163
[Stage1] epoch 110 batch 500/560 loss=0.3990
[Stage1] epoch 110 batch 550/560 loss=0.4382
[Stage1][110/150] train=0.4248 valDice(vessel)=0.4866 valClDice(vessel)=0.7032
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 111 batch 1/560 loss=0.4131
[Stage1] epoch 111 batch 50/560 loss=0.4069
[Stage1] epoch 111 batch 100/560 loss=0.4348
[Stage1] epoch 111 batch 150/560 loss=0.4503
[Stage1] epoch 111 batch 200/560 loss=0.4357
[Stage1] epoch 111 batch 250/560 loss=0.4714
[Stage1] epoch 111 batch 300/560 loss=0.4083
[Stage1] epoch 111 batch 350/560 loss=0.4283
[Stage1] epoch 111 batch 400/560 loss=0.4275
[Stage1] epoch 111 batch 450/560 loss=0.3906
[Stage1] epoch 111 batch 500/560 loss=0.4074
[Stage1] epoch 111 batch 550/560 loss=0.4163
[Stage1][111/150] train=0.4244 valDice(vessel)=0.4738 valClDice(vessel)=0.7077
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 112 batch 1/560 loss=0.4373
[Stage1] epoch 112 batch 50/560 loss=0.4245
[Stage1] epoch 112 batch 100/560 loss=0.4168
[Stage1] epoch 112 batch 150/560 loss=0.4193
[Stage1] epoch 112 batch 200/560 loss=0.4205
[Stage1] epoch 112 batch 250/560 loss=0.4029
[Stage1] epoch 112 batch 300/560 loss=0.4155
[Stage1] epoch 112 batch 350/560 loss=0.4189
[Stage1] epoch 112 batch 400/560 loss=0.4068
[Stage1] epoch 112 batch 450/560 loss=0.4131
[Stage1] epoch 112 batch 500/560 loss=0.4168
[Stage1] epoch 112 batch 550/560 loss=0.4382
[Stage1][112/150] train=0.4257 valDice(vessel)=0.4322 valClDice(vessel)=0.6194
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 113 batch 1/560 loss=0.4490
[Stage1] epoch 113 batch 50/560 loss=0.4277
[Stage1] epoch 113 batch 100/560 loss=0.4522
[Stage1] epoch 113 batch 150/560 loss=0.8586
[Stage1] epoch 113 batch 200/560 loss=0.4015
[Stage1] epoch 113 batch 250/560 loss=0.7452
[Stage1] epoch 113 batch 300/560 loss=0.4134
[Stage1] epoch 113 batch 350/560 loss=0.3958
[Stage1] epoch 113 batch 400/560 loss=0.4461
[Stage1] epoch 113 batch 450/560 loss=0.4286
[Stage1] epoch 113 batch 500/560 loss=0.4516
[Stage1] epoch 113 batch 550/560 loss=0.4559
[Stage1][113/150] train=0.4234 valDice(vessel)=0.4660 valClDice(vessel)=0.6454
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 114 batch 1/560 loss=0.3958
[Stage1] epoch 114 batch 50/560 loss=0.3989
[Stage1] epoch 114 batch 100/560 loss=0.4233
[Stage1] epoch 114 batch 150/560 loss=0.4086
[Stage1] epoch 114 batch 200/560 loss=0.4083
[Stage1] epoch 114 batch 250/560 loss=0.7561
[Stage1] epoch 114 batch 300/560 loss=0.4227
[Stage1] epoch 114 batch 350/560 loss=0.4076
[Stage1] epoch 114 batch 400/560 loss=0.3974
[Stage1] epoch 114 batch 450/560 loss=0.3956
[Stage1] epoch 114 batch 500/560 loss=0.3997
[Stage1] epoch 114 batch 550/560 loss=0.4324
[Stage1][114/150] train=0.4246 valDice(vessel)=0.4378 valClDice(vessel)=0.6332
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 115 batch 1/560 loss=0.4029
[Stage1] epoch 115 batch 50/560 loss=0.4231
[Stage1] epoch 115 batch 100/560 loss=0.4139
[Stage1] epoch 115 batch 150/560 loss=0.4210
[Stage1] epoch 115 batch 200/560 loss=0.4250
[Stage1] epoch 115 batch 250/560 loss=0.4363
[Stage1] epoch 115 batch 300/560 loss=0.4672
[Stage1] epoch 115 batch 350/560 loss=0.4057
[Stage1] epoch 115 batch 400/560 loss=0.4084
[Stage1] epoch 115 batch 450/560 loss=0.4117
[Stage1] epoch 115 batch 500/560 loss=0.4249
[Stage1] epoch 115 batch 550/560 loss=0.4129
[Stage1][115/150] train=0.4238 valDice(vessel)=0.4666 valClDice(vessel)=0.6829
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 116 batch 1/560 loss=0.4353
[Stage1] epoch 116 batch 50/560 loss=0.3919
[Stage1] epoch 116 batch 100/560 loss=0.4681
[Stage1] epoch 116 batch 150/560 loss=0.4075
[Stage1] epoch 116 batch 200/560 loss=0.4659
[Stage1] epoch 116 batch 250/560 loss=0.4030
[Stage1] epoch 116 batch 300/560 loss=0.4077
[Stage1] epoch 116 batch 350/560 loss=0.3986
[Stage1] epoch 116 batch 400/560 loss=0.4632
[Stage1] epoch 116 batch 450/560 loss=0.4151
[Stage1] epoch 116 batch 500/560 loss=0.3945
[Stage1] epoch 116 batch 550/560 loss=0.4056
[Stage1][116/150] train=0.4252 valDice(vessel)=0.4690 valClDice(vessel)=0.7007
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 117 batch 1/560 loss=0.4404
[Stage1] epoch 117 batch 50/560 loss=0.3974
[Stage1] epoch 117 batch 100/560 loss=0.4049
[Stage1] epoch 117 batch 150/560 loss=0.4068
[Stage1] epoch 117 batch 200/560 loss=0.4644
[Stage1] epoch 117 batch 250/560 loss=0.4487
[Stage1] epoch 117 batch 300/560 loss=0.4029
[Stage1] epoch 117 batch 350/560 loss=0.4001
[Stage1] epoch 117 batch 400/560 loss=0.4141
[Stage1] epoch 117 batch 450/560 loss=0.4075
[Stage1] epoch 117 batch 500/560 loss=0.4184
[Stage1] epoch 117 batch 550/560 loss=0.4674
[Stage1][117/150] train=0.4255 valDice(vessel)=0.4921 valClDice(vessel)=0.7026
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 118 batch 1/560 loss=0.4016
[Stage1] epoch 118 batch 50/560 loss=0.4096
[Stage1] epoch 118 batch 100/560 loss=0.4051
[Stage1] epoch 118 batch 150/560 loss=0.3906
[Stage1] epoch 118 batch 200/560 loss=0.4073
[Stage1] epoch 118 batch 250/560 loss=0.4093
[Stage1] epoch 118 batch 300/560 loss=0.4259
[Stage1] epoch 118 batch 350/560 loss=0.4021
[Stage1] epoch 118 batch 400/560 loss=0.4136
[Stage1] epoch 118 batch 450/560 loss=0.4039
[Stage1] epoch 118 batch 500/560 loss=0.4109
[Stage1] epoch 118 batch 550/560 loss=0.3967
[Stage1][118/150] train=0.4245 valDice(vessel)=0.4663 valClDice(vessel)=0.6887
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 119 batch 1/560 loss=0.4110
[Stage1] epoch 119 batch 50/560 loss=0.4336
[Stage1] epoch 119 batch 100/560 loss=0.3890
[Stage1] epoch 119 batch 150/560 loss=0.3848
[Stage1] epoch 119 batch 200/560 loss=0.4621
[Stage1] epoch 119 batch 250/560 loss=0.4105
[Stage1] epoch 119 batch 300/560 loss=0.4328
[Stage1] epoch 119 batch 350/560 loss=0.4103
[Stage1] epoch 119 batch 400/560 loss=0.4111
[Stage1] epoch 119 batch 450/560 loss=0.4247
[Stage1] epoch 119 batch 500/560 loss=0.4074
[Stage1] epoch 119 batch 550/560 loss=0.4036
[Stage1][119/150] train=0.4240 valDice(vessel)=0.4642 valClDice(vessel)=0.6872
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 120 batch 1/560 loss=0.3782
[Stage1] epoch 120 batch 50/560 loss=0.4243
[Stage1] epoch 120 batch 100/560 loss=0.4842
[Stage1] epoch 120 batch 150/560 loss=0.4017
[Stage1] epoch 120 batch 200/560 loss=0.4619
[Stage1] epoch 120 batch 250/560 loss=0.3950
[Stage1] epoch 120 batch 300/560 loss=0.3965
[Stage1] epoch 120 batch 350/560 loss=0.4252
[Stage1] epoch 120 batch 400/560 loss=0.4033
[Stage1] epoch 120 batch 450/560 loss=0.4162
[Stage1] epoch 120 batch 500/560 loss=0.4252
[Stage1] epoch 120 batch 550/560 loss=0.4590
[Stage1][120/150] train=0.4224 valDice(vessel)=0.4856 valClDice(vessel)=0.7091
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 121 batch 1/560 loss=0.4461
[Stage1] epoch 121 batch 50/560 loss=0.4064
[Stage1] epoch 121 batch 100/560 loss=0.3924
[Stage1] epoch 121 batch 150/560 loss=0.4042
[Stage1] epoch 121 batch 200/560 loss=0.4000
[Stage1] epoch 121 batch 250/560 loss=0.4462
[Stage1] epoch 121 batch 300/560 loss=0.4146
[Stage1] epoch 121 batch 350/560 loss=0.7741
[Stage1] epoch 121 batch 400/560 loss=0.4034
[Stage1] epoch 121 batch 450/560 loss=0.4057
[Stage1] epoch 121 batch 500/560 loss=0.4085
[Stage1] epoch 121 batch 550/560 loss=0.4181
[Stage1][121/150] train=0.4234 valDice(vessel)=0.4977 valClDice(vessel)=0.7511
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 122 batch 1/560 loss=0.4008
[Stage1] epoch 122 batch 50/560 loss=0.4046
[Stage1] epoch 122 batch 100/560 loss=0.4649
[Stage1] epoch 122 batch 150/560 loss=0.4382
[Stage1] epoch 122 batch 200/560 loss=0.4390
[Stage1] epoch 122 batch 250/560 loss=0.3955
[Stage1] epoch 122 batch 300/560 loss=0.4120
[Stage1] epoch 122 batch 350/560 loss=0.4548
[Stage1] epoch 122 batch 400/560 loss=0.4643
[Stage1] epoch 122 batch 450/560 loss=0.3961
[Stage1] epoch 122 batch 500/560 loss=0.4081
[Stage1] epoch 122 batch 550/560 loss=0.3949
[Stage1][122/150] train=0.4261 valDice(vessel)=0.4675 valClDice(vessel)=0.7188
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 123 batch 1/560 loss=0.4025
[Stage1] epoch 123 batch 50/560 loss=0.4149
[Stage1] epoch 123 batch 100/560 loss=0.4106
[Stage1] epoch 123 batch 150/560 loss=0.4022
[Stage1] epoch 123 batch 200/560 loss=0.4282
[Stage1] epoch 123 batch 250/560 loss=0.4513
[Stage1] epoch 123 batch 300/560 loss=0.3940
[Stage1] epoch 123 batch 350/560 loss=0.4321
[Stage1] epoch 123 batch 400/560 loss=0.3955
[Stage1] epoch 123 batch 450/560 loss=0.3941
[Stage1] epoch 123 batch 500/560 loss=0.3951
[Stage1] epoch 123 batch 550/560 loss=0.4669
[Stage1][123/150] train=0.4232 valDice(vessel)=0.4615 valClDice(vessel)=0.6981
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 124 batch 1/560 loss=0.4253
[Stage1] epoch 124 batch 50/560 loss=0.4040
[Stage1] epoch 124 batch 100/560 loss=0.4070
[Stage1] epoch 124 batch 150/560 loss=0.4229
[Stage1] epoch 124 batch 200/560 loss=0.3947
[Stage1] epoch 124 batch 250/560 loss=0.3978
[Stage1] epoch 124 batch 300/560 loss=0.3903
[Stage1] epoch 124 batch 350/560 loss=0.4471
[Stage1] epoch 124 batch 400/560 loss=0.4368
[Stage1] epoch 124 batch 450/560 loss=0.4682
[Stage1] epoch 124 batch 500/560 loss=0.4038
[Stage1] epoch 124 batch 550/560 loss=0.4048
[Stage1][124/150] train=0.4230 valDice(vessel)=0.4769 valClDice(vessel)=0.6937
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 125 batch 1/560 loss=0.4091
[Stage1] epoch 125 batch 50/560 loss=0.3981
[Stage1] epoch 125 batch 100/560 loss=0.4140
[Stage1] epoch 125 batch 150/560 loss=0.3938
[Stage1] epoch 125 batch 200/560 loss=0.3936
[Stage1] epoch 125 batch 250/560 loss=0.4199
[Stage1] epoch 125 batch 300/560 loss=0.4275
[Stage1] epoch 125 batch 350/560 loss=0.3963
[Stage1] epoch 125 batch 400/560 loss=0.4370
[Stage1] epoch 125 batch 450/560 loss=0.4150
[Stage1] epoch 125 batch 500/560 loss=0.4033
[Stage1] epoch 125 batch 550/560 loss=0.4101
[Stage1][125/150] train=0.4238 valDice(vessel)=0.4876 valClDice(vessel)=0.7085
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 126 batch 1/560 loss=0.4207
[Stage1] epoch 126 batch 50/560 loss=0.4020
[Stage1] epoch 126 batch 100/560 loss=0.3956
[Stage1] epoch 126 batch 150/560 loss=0.4443
[Stage1] epoch 126 batch 200/560 loss=0.3991
[Stage1] epoch 126 batch 250/560 loss=0.4304
[Stage1] epoch 126 batch 300/560 loss=0.4291
[Stage1] epoch 126 batch 350/560 loss=0.3943
[Stage1] epoch 126 batch 400/560 loss=0.3917
[Stage1] epoch 126 batch 450/560 loss=0.4358
[Stage1] epoch 126 batch 500/560 loss=0.4627
[Stage1] epoch 126 batch 550/560 loss=0.3916
[Stage1][126/150] train=0.4235 valDice(vessel)=0.4687 valClDice(vessel)=0.7006
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 127 batch 1/560 loss=0.3926
[Stage1] epoch 127 batch 50/560 loss=0.4495
[Stage1] epoch 127 batch 100/560 loss=0.3876
[Stage1] epoch 127 batch 150/560 loss=0.4320
[Stage1] epoch 127 batch 200/560 loss=0.4516
[Stage1] epoch 127 batch 250/560 loss=0.4046
[Stage1] epoch 127 batch 300/560 loss=0.4273
[Stage1] epoch 127 batch 350/560 loss=0.4085
[Stage1] epoch 127 batch 400/560 loss=0.4122
[Stage1] epoch 127 batch 450/560 loss=0.4174
[Stage1] epoch 127 batch 500/560 loss=0.3996
[Stage1] epoch 127 batch 550/560 loss=0.4103
[Stage1][127/150] train=0.4227 valDice(vessel)=0.4762 valClDice(vessel)=0.6823
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 128 batch 1/560 loss=0.4054
[Stage1] epoch 128 batch 50/560 loss=0.4026
[Stage1] epoch 128 batch 100/560 loss=0.4483
[Stage1] epoch 128 batch 150/560 loss=0.4195
[Stage1] epoch 128 batch 200/560 loss=0.4077
[Stage1] epoch 128 batch 250/560 loss=0.4016
[Stage1] epoch 128 batch 300/560 loss=0.4174
[Stage1] epoch 128 batch 350/560 loss=0.4463
[Stage1] epoch 128 batch 400/560 loss=0.3917
[Stage1] epoch 128 batch 450/560 loss=0.3891
[Stage1] epoch 128 batch 500/560 loss=0.4004
[Stage1] epoch 128 batch 550/560 loss=0.4450
[Stage1][128/150] train=0.4212 valDice(vessel)=0.4839 valClDice(vessel)=0.7303
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 129 batch 1/560 loss=0.4527
[Stage1] epoch 129 batch 50/560 loss=0.4225
[Stage1] epoch 129 batch 100/560 loss=0.4154
[Stage1] epoch 129 batch 150/560 loss=0.4599
[Stage1] epoch 129 batch 200/560 loss=0.4118
[Stage1] epoch 129 batch 250/560 loss=0.4411
[Stage1] epoch 129 batch 300/560 loss=0.4289
[Stage1] epoch 129 batch 350/560 loss=0.4043
[Stage1] epoch 129 batch 400/560 loss=0.4165
[Stage1] epoch 129 batch 450/560 loss=0.4316
[Stage1] epoch 129 batch 500/560 loss=0.4087
[Stage1] epoch 129 batch 550/560 loss=0.4142
[Stage1][129/150] train=0.4225 valDice(vessel)=0.4633 valClDice(vessel)=0.6699
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 130 batch 1/560 loss=0.4152
[Stage1] epoch 130 batch 50/560 loss=0.4053
[Stage1] epoch 130 batch 100/560 loss=0.4264
[Stage1] epoch 130 batch 150/560 loss=0.4130
[Stage1] epoch 130 batch 200/560 loss=0.4083
[Stage1] epoch 130 batch 250/560 loss=0.4005
[Stage1] epoch 130 batch 300/560 loss=0.4021
[Stage1] epoch 130 batch 350/560 loss=0.4082
[Stage1] epoch 130 batch 400/560 loss=0.4221
[Stage1] epoch 130 batch 450/560 loss=0.4245
[Stage1] epoch 130 batch 500/560 loss=0.4206
[Stage1] epoch 130 batch 550/560 loss=0.4157
[Stage1][130/150] train=0.4218 valDice(vessel)=0.4663 valClDice(vessel)=0.6731
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 131 batch 1/560 loss=0.4424
[Stage1] epoch 131 batch 50/560 loss=0.4455
[Stage1] epoch 131 batch 100/560 loss=0.3936
[Stage1] epoch 131 batch 150/560 loss=0.4517
[Stage1] epoch 131 batch 200/560 loss=0.3937
[Stage1] epoch 131 batch 250/560 loss=0.3946
[Stage1] epoch 131 batch 300/560 loss=0.4024
[Stage1] epoch 131 batch 350/560 loss=0.4059
[Stage1] epoch 131 batch 400/560 loss=0.4072
[Stage1] epoch 131 batch 450/560 loss=0.4115
[Stage1] epoch 131 batch 500/560 loss=0.3978
[Stage1] epoch 131 batch 550/560 loss=0.3990
[Stage1][131/150] train=0.4218 valDice(vessel)=0.4709 valClDice(vessel)=0.6949
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 132 batch 1/560 loss=0.4351
[Stage1] epoch 132 batch 50/560 loss=0.4392
[Stage1] epoch 132 batch 100/560 loss=0.4106
[Stage1] epoch 132 batch 150/560 loss=0.4323
[Stage1] epoch 132 batch 200/560 loss=0.4837
[Stage1] epoch 132 batch 250/560 loss=0.4078
[Stage1] epoch 132 batch 300/560 loss=0.4223
[Stage1] epoch 132 batch 350/560 loss=0.4014
[Stage1] epoch 132 batch 400/560 loss=0.4833
[Stage1] epoch 132 batch 450/560 loss=0.4247
[Stage1] epoch 132 batch 500/560 loss=0.4205
[Stage1] epoch 132 batch 550/560 loss=0.4171
[Stage1][132/150] train=0.4220 valDice(vessel)=0.4183 valClDice(vessel)=0.6422
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 133 batch 1/560 loss=0.4023
[Stage1] epoch 133 batch 50/560 loss=0.4285
[Stage1] epoch 133 batch 100/560 loss=0.4257
[Stage1] epoch 133 batch 150/560 loss=0.3866
[Stage1] epoch 133 batch 200/560 loss=0.4501
[Stage1] epoch 133 batch 250/560 loss=0.4229
[Stage1] epoch 133 batch 300/560 loss=0.4135
[Stage1] epoch 133 batch 350/560 loss=0.4285
[Stage1] epoch 133 batch 400/560 loss=0.4496
[Stage1] epoch 133 batch 450/560 loss=0.4031
[Stage1] epoch 133 batch 500/560 loss=0.4198
[Stage1] epoch 133 batch 550/560 loss=0.4074
[Stage1][133/150] train=0.4225 valDice(vessel)=0.4558 valClDice(vessel)=0.6817
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 134 batch 1/560 loss=0.7237
[Stage1] epoch 134 batch 50/560 loss=0.4053
[Stage1] epoch 134 batch 100/560 loss=0.3910
[Stage1] epoch 134 batch 150/560 loss=0.3972
[Stage1] epoch 134 batch 200/560 loss=0.3998
[Stage1] epoch 134 batch 250/560 loss=0.4531
[Stage1] epoch 134 batch 300/560 loss=0.4025
[Stage1] epoch 134 batch 350/560 loss=0.4197
[Stage1] epoch 134 batch 400/560 loss=0.7196
[Stage1] epoch 134 batch 450/560 loss=0.4143
[Stage1] epoch 134 batch 500/560 loss=0.3976
[Stage1] epoch 134 batch 550/560 loss=0.3999
[Stage1][134/150] train=0.4214 valDice(vessel)=0.4718 valClDice(vessel)=0.6828
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 135 batch 1/560 loss=0.4135
[Stage1] epoch 135 batch 50/560 loss=0.4253
[Stage1] epoch 135 batch 100/560 loss=0.4278
[Stage1] epoch 135 batch 150/560 loss=0.3907
[Stage1] epoch 135 batch 200/560 loss=0.4093
[Stage1] epoch 135 batch 250/560 loss=0.4258
[Stage1] epoch 135 batch 300/560 loss=0.4030
[Stage1] epoch 135 batch 350/560 loss=0.4278
[Stage1] epoch 135 batch 400/560 loss=0.4081
[Stage1] epoch 135 batch 450/560 loss=0.4041
[Stage1] epoch 135 batch 500/560 loss=0.4114
[Stage1] epoch 135 batch 550/560 loss=0.4400
[Stage1][135/150] train=0.4206 valDice(vessel)=0.5107 valClDice(vessel)=0.7497
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 136 batch 1/560 loss=0.4396
[Stage1] epoch 136 batch 50/560 loss=0.3973
[Stage1] epoch 136 batch 100/560 loss=0.4369
[Stage1] epoch 136 batch 150/560 loss=0.4775
[Stage1] epoch 136 batch 200/560 loss=0.3976
[Stage1] epoch 136 batch 250/560 loss=0.4093
[Stage1] epoch 136 batch 300/560 loss=0.4811
[Stage1] epoch 136 batch 350/560 loss=0.3987
[Stage1] epoch 136 batch 400/560 loss=0.4015
[Stage1] epoch 136 batch 450/560 loss=0.4063
[Stage1] epoch 136 batch 500/560 loss=0.4083
[Stage1] epoch 136 batch 550/560 loss=0.4050
[Stage1][136/150] train=0.4219 valDice(vessel)=0.4565 valClDice(vessel)=0.6628
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 137 batch 1/560 loss=0.4267
[Stage1] epoch 137 batch 50/560 loss=0.4332
[Stage1] epoch 137 batch 100/560 loss=0.4187
[Stage1] epoch 137 batch 150/560 loss=0.4359
[Stage1] epoch 137 batch 200/560 loss=0.3964
[Stage1] epoch 137 batch 250/560 loss=0.4015
[Stage1] epoch 137 batch 300/560 loss=0.3976
[Stage1] epoch 137 batch 350/560 loss=0.6371
[Stage1] epoch 137 batch 400/560 loss=0.3980
[Stage1] epoch 137 batch 450/560 loss=0.4138
[Stage1] epoch 137 batch 500/560 loss=0.4325
[Stage1] epoch 137 batch 550/560 loss=0.5030
[Stage1][137/150] train=0.4202 valDice(vessel)=0.4525 valClDice(vessel)=0.6652
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 138 batch 1/560 loss=0.4078
[Stage1] epoch 138 batch 50/560 loss=0.4527
[Stage1] epoch 138 batch 100/560 loss=0.4229
[Stage1] epoch 138 batch 150/560 loss=0.4006
[Stage1] epoch 138 batch 200/560 loss=0.4352
[Stage1] epoch 138 batch 250/560 loss=0.4039
[Stage1] epoch 138 batch 300/560 loss=0.4133
[Stage1] epoch 138 batch 350/560 loss=0.4290
[Stage1] epoch 138 batch 400/560 loss=0.4013
[Stage1] epoch 138 batch 450/560 loss=0.4431
[Stage1] epoch 138 batch 500/560 loss=0.3963
[Stage1] epoch 138 batch 550/560 loss=0.4057
[Stage1][138/150] train=0.4223 valDice(vessel)=0.4440 valClDice(vessel)=0.6869
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 139 batch 1/560 loss=0.4591
[Stage1] epoch 139 batch 50/560 loss=0.4412
[Stage1] epoch 139 batch 100/560 loss=0.3915
[Stage1] epoch 139 batch 150/560 loss=0.4282
[Stage1] epoch 139 batch 200/560 loss=0.4204
[Stage1] epoch 139 batch 250/560 loss=0.3832
[Stage1] epoch 139 batch 300/560 loss=0.3946
[Stage1] epoch 139 batch 350/560 loss=0.3862
[Stage1] epoch 139 batch 400/560 loss=0.4050
[Stage1] epoch 139 batch 450/560 loss=0.4117
[Stage1] epoch 139 batch 500/560 loss=0.4187
[Stage1] epoch 139 batch 550/560 loss=0.4047
[Stage1][139/150] train=0.4215 valDice(vessel)=0.4808 valClDice(vessel)=0.7339
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 140 batch 1/560 loss=0.4056
[Stage1] epoch 140 batch 50/560 loss=0.3885
[Stage1] epoch 140 batch 100/560 loss=0.3873
[Stage1] epoch 140 batch 150/560 loss=0.4412
[Stage1] epoch 140 batch 200/560 loss=0.3918
[Stage1] epoch 140 batch 250/560 loss=0.4475
[Stage1] epoch 140 batch 300/560 loss=0.4120
[Stage1] epoch 140 batch 350/560 loss=0.4584
[Stage1] epoch 140 batch 400/560 loss=0.8334
[Stage1] epoch 140 batch 450/560 loss=0.7811
[Stage1] epoch 140 batch 500/560 loss=0.4036
[Stage1] epoch 140 batch 550/560 loss=0.4062
[Stage1][140/150] train=0.4207 valDice(vessel)=0.4407 valClDice(vessel)=0.6831
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 141 batch 1/560 loss=0.4498
[Stage1] epoch 141 batch 50/560 loss=0.4165
[Stage1] epoch 141 batch 100/560 loss=0.4074
[Stage1] epoch 141 batch 150/560 loss=0.4145
[Stage1] epoch 141 batch 200/560 loss=0.3963
[Stage1] epoch 141 batch 250/560 loss=0.4085
[Stage1] epoch 141 batch 300/560 loss=0.4439
[Stage1] epoch 141 batch 350/560 loss=0.4178
[Stage1] epoch 141 batch 400/560 loss=0.3955
[Stage1] epoch 141 batch 450/560 loss=0.3880
[Stage1] epoch 141 batch 500/560 loss=0.4339
[Stage1] epoch 141 batch 550/560 loss=0.4209
[Stage1][141/150] train=0.4212 valDice(vessel)=0.4695 valClDice(vessel)=0.6930
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 142 batch 1/560 loss=0.3961
[Stage1] epoch 142 batch 50/560 loss=0.4210
[Stage1] epoch 142 batch 100/560 loss=0.4210
[Stage1] epoch 142 batch 150/560 loss=0.4283
[Stage1] epoch 142 batch 200/560 loss=0.3982
[Stage1] epoch 142 batch 250/560 loss=0.4142
[Stage1] epoch 142 batch 300/560 loss=0.4145
[Stage1] epoch 142 batch 350/560 loss=0.3962
[Stage1] epoch 142 batch 400/560 loss=0.4067
[Stage1] epoch 142 batch 450/560 loss=0.3789
[Stage1] epoch 142 batch 500/560 loss=0.4298
[Stage1] epoch 142 batch 550/560 loss=0.4373
[Stage1][142/150] train=0.4225 valDice(vessel)=0.4475 valClDice(vessel)=0.6527
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 143 batch 1/560 loss=0.4114
[Stage1] epoch 143 batch 50/560 loss=0.4455
[Stage1] epoch 143 batch 100/560 loss=0.3897
[Stage1] epoch 143 batch 150/560 loss=0.4316
[Stage1] epoch 143 batch 200/560 loss=0.4252
[Stage1] epoch 143 batch 250/560 loss=0.4201
[Stage1] epoch 143 batch 300/560 loss=0.4364
[Stage1] epoch 143 batch 350/560 loss=0.4037
[Stage1] epoch 143 batch 400/560 loss=0.4006
[Stage1] epoch 143 batch 450/560 loss=0.3903
[Stage1] epoch 143 batch 500/560 loss=0.4229
[Stage1] epoch 143 batch 550/560 loss=0.3988
[Stage1][143/150] train=0.4228 valDice(vessel)=0.4754 valClDice(vessel)=0.6769
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 144 batch 1/560 loss=0.7542
[Stage1] epoch 144 batch 50/560 loss=0.3889
[Stage1] epoch 144 batch 100/560 loss=0.4266
[Stage1] epoch 144 batch 150/560 loss=0.4031
[Stage1] epoch 144 batch 200/560 loss=0.4020
[Stage1] epoch 144 batch 250/560 loss=0.3931
[Stage1] epoch 144 batch 300/560 loss=0.4218
[Stage1] epoch 144 batch 350/560 loss=0.4100
[Stage1] epoch 144 batch 400/560 loss=0.4245
[Stage1] epoch 144 batch 450/560 loss=0.4173
[Stage1] epoch 144 batch 500/560 loss=0.4315
[Stage1] epoch 144 batch 550/560 loss=0.3996
[Stage1][144/150] train=0.4207 valDice(vessel)=0.4759 valClDice(vessel)=0.6970
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 145 batch 1/560 loss=0.4027
[Stage1] epoch 145 batch 50/560 loss=0.4021
[Stage1] epoch 145 batch 100/560 loss=0.4417
[Stage1] epoch 145 batch 150/560 loss=0.4096
[Stage1] epoch 145 batch 200/560 loss=0.4272
[Stage1] epoch 145 batch 250/560 loss=0.4003
[Stage1] epoch 145 batch 300/560 loss=0.4236
[Stage1] epoch 145 batch 350/560 loss=0.4014
[Stage1] epoch 145 batch 400/560 loss=0.4032
[Stage1] epoch 145 batch 450/560 loss=0.4262
[Stage1] epoch 145 batch 500/560 loss=0.4047
[Stage1] epoch 145 batch 550/560 loss=0.4014
[Stage1][145/150] train=0.4206 valDice(vessel)=0.4595 valClDice(vessel)=0.6698
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 146 batch 1/560 loss=0.4286
[Stage1] epoch 146 batch 50/560 loss=0.4352
[Stage1] epoch 146 batch 100/560 loss=0.4172
[Stage1] epoch 146 batch 150/560 loss=0.4405
[Stage1] epoch 146 batch 200/560 loss=0.4069
[Stage1] epoch 146 batch 250/560 loss=0.4090
[Stage1] epoch 146 batch 300/560 loss=0.4028
[Stage1] epoch 146 batch 350/560 loss=0.4060
[Stage1] epoch 146 batch 400/560 loss=0.3862
[Stage1] epoch 146 batch 450/560 loss=0.3981
[Stage1] epoch 146 batch 500/560 loss=0.4094
[Stage1] epoch 146 batch 550/560 loss=0.4072
[Stage1][146/150] train=0.4198 valDice(vessel)=0.4744 valClDice(vessel)=0.7310
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 147 batch 1/560 loss=0.4235
[Stage1] epoch 147 batch 50/560 loss=0.4286
[Stage1] epoch 147 batch 100/560 loss=0.3974
[Stage1] epoch 147 batch 150/560 loss=0.4220
[Stage1] epoch 147 batch 200/560 loss=0.3991
[Stage1] epoch 147 batch 250/560 loss=0.4194
[Stage1] epoch 147 batch 300/560 loss=0.4184
[Stage1] epoch 147 batch 350/560 loss=0.3920
[Stage1] epoch 147 batch 400/560 loss=0.4416
[Stage1] epoch 147 batch 450/560 loss=0.3832
[Stage1] epoch 147 batch 500/560 loss=0.4441
[Stage1] epoch 147 batch 550/560 loss=0.3980
[Stage1][147/150] train=0.4194 valDice(vessel)=0.5253 valClDice(vessel)=0.7485
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 148 batch 1/560 loss=0.4225
[Stage1] epoch 148 batch 50/560 loss=0.4188
[Stage1] epoch 148 batch 100/560 loss=0.4580
[Stage1] epoch 148 batch 150/560 loss=0.4423
[Stage1] epoch 148 batch 200/560 loss=0.4360
[Stage1] epoch 148 batch 250/560 loss=0.4176
[Stage1] epoch 148 batch 300/560 loss=0.3893
[Stage1] epoch 148 batch 350/560 loss=0.4043
[Stage1] epoch 148 batch 400/560 loss=0.4038
[Stage1] epoch 148 batch 450/560 loss=0.4149
[Stage1] epoch 148 batch 500/560 loss=0.4073
[Stage1] epoch 148 batch 550/560 loss=0.4083
[Stage1][148/150] train=0.4186 valDice(vessel)=0.4445 valClDice(vessel)=0.6313
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 149 batch 1/560 loss=0.4137
[Stage1] epoch 149 batch 50/560 loss=0.3896
[Stage1] epoch 149 batch 100/560 loss=0.4197
[Stage1] epoch 149 batch 150/560 loss=0.4601
[Stage1] epoch 149 batch 200/560 loss=0.4028
[Stage1] epoch 149 batch 250/560 loss=0.4273
[Stage1] epoch 149 batch 300/560 loss=0.3967
[Stage1] epoch 149 batch 350/560 loss=0.4178
[Stage1] epoch 149 batch 400/560 loss=0.4160
[Stage1] epoch 149 batch 450/560 loss=0.4240
[Stage1] epoch 149 batch 500/560 loss=0.4265
[Stage1] epoch 149 batch 550/560 loss=0.4393
[Stage1][149/150] train=0.4210 valDice(vessel)=0.5021 valClDice(vessel)=0.7189
[Stage1] DEBUG train patch: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
[Stage1] epoch 150 batch 1/560 loss=0.4115
[Stage1] epoch 150 batch 50/560 loss=0.4535
[Stage1] epoch 150 batch 100/560 loss=0.4476
[Stage1] epoch 150 batch 150/560 loss=0.4293
[Stage1] epoch 150 batch 200/560 loss=0.7811
[Stage1] epoch 150 batch 250/560 loss=0.4486
[Stage1] epoch 150 batch 300/560 loss=0.4030
[Stage1] epoch 150 batch 350/560 loss=0.3909
[Stage1] epoch 150 batch 400/560 loss=0.4115
[Stage1] epoch 150 batch 450/560 loss=0.4104
[Stage1] epoch 150 batch 500/560 loss=0.3970
[Stage1] epoch 150 batch 550/560 loss=0.4043
[Stage1][150/150] train=0.4202 valDice(vessel)=0.5131 valClDice(vessel)=0.7371

losses.py for CompositeLoss import

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class DiceCELoss3D(nn.Module):
    def __init__(self, num_classes=3, class_weights=None, smooth=1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        if class_weights is not None:
            self.register_buffer("w", torch.tensor(class_weights, dtype=torch.float32))
        else:
            self.w = None

    def forward(self, logits, target):
        # logits: (B,C,D,H,W), target: (B,D,H,W) with values in {0..C-1}
        ce = F.cross_entropy(logits, target, weight=self.w)
        probs = F.softmax(logits, dim=1)

        with torch.no_grad():
            onehot = torch.zeros_like(probs).scatter_(1, target.unsqueeze(1), 1.0)

        dims = (0,2,3,4)
        intersect = (probs * onehot).sum(dim=dims)
        pred_sum  = probs.sum(dim=dims)
        targ_sum  = onehot.sum(dim=dims)

        dice_per_class = (2*intersect + self.smooth) / (pred_sum + targ_sum + self.smooth)
        dice_loss = 1 - dice_per_class.mean()
        return ce + dice_loss


def soft_skeletonize_3d(x, iters=8):
    # x: (B,1,D,H,W) in [0,1], differentiable approx using max-pool morphological thinning
    for _ in range(iters):
        eroded = 1.0 - F.max_pool3d(1.0 - x, kernel_size=3, stride=1, padding=1)
        opened = F.max_pool3d(eroded, kernel_size=3, stride=1, padding=1)
        contour = F.relu(opened - eroded)  # soft contour
        x = F.relu(x - contour)
    return x

class SoftClDiceLoss(nn.Module):
    """
    Soft clDice from Shit et al. (CVPR'21) – differentiable centerline overlap.
    Computes union-of-vessels by taking 1 - p_bg and 1 - y_bg and compares skeleta.
    """
    def __init__(self, iters=8, eps=1e-6):
        super().__init__()
        self.iters = iters
        self.eps = eps

    def forward(self, probs, target_onehot):
        # probs: (B,C,D,H,W) softmax; target_onehot: same shape onehot
        # Build vessel union channel: 1 - background
        p_v = 1.0 - probs[:, :1, ...]  # assumes class 0 = background
        y_v = 1.0 - target_onehot[:, :1, ...]

        p_skel = soft_skeletonize_3d(p_v.clamp(0,1), iters=self.iters)
        y_skel = soft_skeletonize_3d(y_v, iters=self.iters)

        tprec = (p_skel * y_v).sum() / (p_skel.sum() + self.eps)
        tsens = (y_skel * p_v).sum() / (y_skel.sum() + self.eps)
        cldice = (2*tprec*tsens) / (tprec + tsens + self.eps)
        return 1.0 - cldice


class CompositeLoss(nn.Module):
    def __init__(self, num_classes=3, class_weights=None, soft_cldice_weight=0.0, soft_cldice_iters=8):
        super().__init__()
        self.dicece = DiceCELoss3D(num_classes, class_weights)
        self.cl = SoftClDiceLoss(iters=soft_cldice_iters)
        self.w_cl = soft_cldice_weight

    def forward(self, logits, target):
        base = self.dicece(logits, target)
        if self.w_cl <= 0:
            return base
        probs = F.softmax(logits, dim=1)
        with torch.no_grad():
            onehot = torch.zeros_like(probs).scatter_(1, target.unsqueeze(1), 1.0)
        return base + self.w_cl * self.cl(probs, onehot)

Step 2 of training: Perform the 3 stages of training of the 3-class segmentation heads (Stage 3 focuses on just artery and vein classes)

In [ ]:
import os, random, argparse, json, pathlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from monai.inferers import sliding_window_inference
import multiprocessing as mp
from multiprocessing import Pool, cpu_count
from torch.utils.data import Dataset


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

'''
def random_multi_crop_3d(
    img,
    lab,
    roi_size,
    num_samples,
    fg_prob: float = 0.5,
):
    """
    Randomly crop 3D patches from volumes, with optional vessel-biased sampling.

    img: (B, C, D, H, W)
    lab: (B, D, H, W)
    roi_size: [pD, pH, pW]
    num_samples: patches per volume
    fg_prob: probability a given patch is forced to contain foreground (label > 0)
    """
    B, C, D, H, W = img.shape
    pD, pH, pW = roi_size
    assert pD <= D and pH <= H and pW <= W, (
        f"Patch size {roi_size} is larger than volume {(D, H, W)}"
    )

    device = img.device
    img_p = torch.empty((B * num_samples, C, pD, pH, pW),
                        dtype=img.dtype, device=device)
    lab_p = torch.empty((B * num_samples, pD, pH, pW),
                        dtype=lab.dtype, device=lab.device)

    out_idx = 0
    for b in range(B):
        # foreground voxel indices for this volume (vessel union)
        fg_mask = (lab[b] > 0)
        fg_idx = torch.nonzero(fg_mask, as_tuple=False)  # (N_fg, 3) [z,y,x]

        for _ in range(num_samples):
            use_fg = (fg_idx.numel() > 0) and (torch.rand(1, device=device) < fg_prob)

            if use_fg:
                # pick a random foreground voxel as (approximate) patch center
                rand_idx = torch.randint(fg_idx.shape[0], (1,), device=device).item()
                zc, yc, xc = fg_idx[rand_idx].tolist()  # now zc,yc,xc are ints

                # compute start coords so patch stays in bounds
                z = max(0, min(zc - pD // 2, D - pD))
                y = max(0, min(yc - pH // 2, H - pH))
                x = max(0, min(xc - pW // 2, W - pW))
            else:
                # uniform random crop
                z = torch.randint(0, D - pD + 1, (1,), device=device).item()
                y = torch.randint(0, H - pH + 1, (1,), device=device).item()
                x = torch.randint(0, W - pW + 1, (1,), device=device).item()

            img_p[out_idx] = img[b, :, z:z+pD, y:y+pH, x:x+pW]
            lab_p[out_idx] = lab[b, z:z+pD, y:y+pH, x:x+pW]
            out_idx += 1

    return img_p, lab_p
'''


def _compute_cldice_worker(p_np, g_np, cldice_metric):
    """
    Worker for multiprocessing: computes clDice for one case.
    p_np, g_np are downsampled boolean numpy arrays (D, H, W).
    """
    return float(cldice_metric(p_np, g_np))


def freeze_backbone(model):
    """
    Freeze encoder/decoder; keep classification heads trainable:
      - output_block (3-class A/V head)
      - deep supervision heads when present
      - vessel_head (1-channel vessel head)
    """
    for name, p in model.named_parameters():
        if (
            "output_block" in name
            or "deep_supervision_heads" in name
            or "vessel_head" in name
        ):
            p.requires_grad = True
        else:
            p.requires_grad = False


def unfreeze_encoder_tail(model, n_stages=2):
    """
    Stage 2: keep heads trainable; optionally unfreeze last decoder stage.
    """
    # Heads
    if hasattr(model, "output_block"):
        for p in model.output_block.parameters():
            p.requires_grad = True
    if hasattr(model, "deep_supervision_heads"):
        for p in model.deep_supervision_heads.parameters():
            p.requires_grad = True
    if hasattr(model, "vessel_head"):
        for p in model.vessel_head.parameters():
            p.requires_grad = True

    # Optionally: last decoder level
    if hasattr(model, "decoder"):
        for p in model.decoder[-1].parameters():
            p.requires_grad = True


def av_head_loss(
    logits,
    lab,
    class_weights_av=None,
    focal_gamma: float = 0.0,
    use_tversky: bool = False,
    tversky_alpha: float = 0.5,
    tversky_beta: float = 0.5,
):
    """
    Extra artery/vein classification loss on vessel voxels only.

    logits: (B, 3, D, H, W), channels [bg, artery, vein]
    lab:    (B, D, H, W) with values {0,1,2}
    """

    # Only compute on vessel voxels
    mask = lab > 0  # (B, D, H, W)
    if not mask.any():
        return logits.new_tensor(0.0)

    # Restrict logits to artery/vein channels
    logits_av = logits[:, 1:3, ...]  # (B, 2, D, H, W)

    # (B,2,D,H,W) -> (B,D,H,W,2) -> (N,2)
    logits_flat = logits_av.permute(0, 2, 3, 4, 1)[mask]  # (N, 2)

    # Targets: map {artery=1, vein=2} -> {0,1}
    targets_flat = (lab[mask] - 1).long()                 # (N,)

    # A/V class weights
    weight = None
    if class_weights_av is not None:
        w = torch.as_tensor(
            class_weights_av, dtype=torch.float32, device=logits_flat.device
        )
        if w.numel() == 2:
            weight = w

    # Focal cross-entropy term
    if focal_gamma > 1e-6:
        log_probs = F.log_softmax(logits_flat, dim=1)  # (N,2)
        probs = log_probs.exp()
        pt = probs.gather(1, targets_flat.unsqueeze(1)).squeeze(1)  # (N,)

        ce_per = -log_probs.gather(1, targets_flat.unsqueeze(1)).squeeze(1)
        focal = (1.0 - pt) ** focal_gamma * ce_per
        if weight is not None:
            focal = focal * weight[targets_flat]
        ce = focal.mean()
    else:
        ce = F.cross_entropy(logits_flat, targets_flat, weight=weight)

    # Dice / Tversky term
    probs_flat = F.softmax(logits_flat, dim=1)  # (N,2)
    target_onehot = torch.zeros_like(probs_flat)
    target_onehot.scatter_(1, targets_flat.unsqueeze(1), 1.0)

    if use_tversky:
        TP = (probs_flat * target_onehot).sum(dim=0)
        FP = (probs_flat * (1 - target_onehot)).sum(dim=0)
        FN = ((1 - probs_flat) * target_onehot).sum(dim=0)
        tversky = TP / (TP + tversky_alpha * FP + tversky_beta * FN + 1e-5)
        dice_term = tversky
    else:
        intersection = (probs_flat * target_onehot).sum(dim=0)
        denom = probs_flat.sum(dim=0) + target_onehot.sum(dim=0) + 1e-5
        dice_term = 2.0 * intersection / denom

    dice_loss = 1.0 - dice_term.mean()

    return ce + dice_loss


def one_epoch_av_refine(
    model,
    loader,
    opt,
    scaler,
    device,
    amp=True,
    class_weights_av=None,
):
    """
    Stage 3: train only av_refine_head on top of a frozen backbone.

    - Uses GT vessel mask (lab > 0) to restrict loss to vessel voxels.
    - Predicts artery vs vein conditionally inside vessel regions.
    """
    # Backbone in eval mode (no BN/dropout changes)
    model.eval()
    # But we still want av_refine_head to be trainable
    if hasattr(model, "av_refine_head"):
        model.av_refine_head.train()

    running = []

    for i, batch in enumerate(loader, start=1):
        img = batch["image"].to(device)
        lab = batch["label"].to(device).long()
        opt.zero_grad(set_to_none=True)

        with autocast(enabled=amp):
            # Get base logits; do NOT backprop through backbone
            with torch.no_grad():
                logits_base = model(img)          # (B,3,D,H,W)

            logits_base = logits_base.detach()

            vessel_mask = lab > 0                # (B,D,H,W)
            if not vessel_mask.any():
                # no vessels in this batch; skip
                continue

            # AV refine head: map 3-channel logits -> 2 classes (art/vein)
            av_logits = model.av_refine_head(logits_base)       # (B,2,D,H,W)

            # Flatten over vessel voxels only
            av_logits_flat = av_logits.permute(0, 2, 3, 4, 1)[vessel_mask]  # (N,2)
            targets_flat = (lab[vessel_mask] - 1).long()                    # {0,1}

            weight = None
            if class_weights_av is not None:
                w = torch.as_tensor(
                    class_weights_av,
                    dtype=torch.float32,
                    device=av_logits_flat.device,
                )
                if w.numel() == 2:
                    weight = w

            loss = F.cross_entropy(av_logits_flat, targets_flat, weight=weight)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running.append(loss.item())

        if i % 50 == 0 or i == 1:
            print(f"  [train_av_refine] batch {i}/{len(loader)} loss={loss.item():.4f}")

    return float(np.mean(running)) if running else 0.0


def one_epoch(
    model,
    loader,
    loss_fn,
    opt,
    scaler,
    device,
    amp=True,
    cldice_loss_fn=None,
    cldice_weight: float = 0.0,        # Union-of-vessels
    patch_size=None,
    samples_per_volume: int = 1,
    av_head_weight: float = 0.0,
    av_class_weights=None,
    cldice_weight_art: float = 0.0,
    cldice_weight_vein: float = 0.0,
    vessel_consistency_weight: float = 0.0,
    av_focal_gamma: float = 0.0,
    av_use_tversky: bool = False,
    av_tversky_alpha: float = 0.5,
    av_tversky_beta: float = 0.5,
):

    model.train()
    running = []

    for i, batch in enumerate(loader, start=1):
        img, lab = batch["image"].to(device), batch["label"].to(device).long()
        opt.zero_grad(set_to_none=True)

        # Patch-based training
        if patch_size is not None and samples_per_volume > 0:
            img, lab = random_multi_crop_3d(
                img,
                lab,
                roi_size=patch_size,
                num_samples=samples_per_volume,
                fg_prob=0.5,  # 50% of patches vessel-focused
            )

        with autocast(enabled=amp):
            logits = model(img)

            # Ensure labels are in [0, num_classes-1]
            n_classes = logits.shape[1]
            invalid = (lab < 0) | (lab >= n_classes)
            if invalid.any():
                lab = lab.clone()
                lab[invalid] = 0

            base_loss = loss_fn(logits, lab)
            loss = base_loss

            # Shared probabilities
            probs = F.softmax(logits, dim=1)

            # Vessel head / union-of-vessels clDice
            if cldice_loss_fn is not None and cldice_weight > 0.0:
                vessel_lab = (lab > 0).long()                 # (B*,D,H,W)
                vessel_gt  = vessel_lab.unsqueeze(1).float()  # (B*,1,D,H,W)

                if hasattr(model, "vessel_head"):
                    vessel_logits = model.vessel_head(logits)
                    vessel_probs  = torch.sigmoid(vessel_logits)
                else:
                    vessel_probs = probs[:, 1:3].sum(dim=1, keepdim=True)

                vessel_probs = vessel_probs.clamp(0.0, 1.0)
                cl_loss_union = cldice_loss_fn(vessel_gt, vessel_probs)
                loss = loss + cldice_weight * cl_loss_union

            # Per-class clDice for artery and vein
            if cldice_loss_fn is not None and (cldice_weight_art > 0.0 or cldice_weight_vein > 0.0):
                # Artery = class 1
                if cldice_weight_art > 0.0:
                    gt_art = (lab == 1).float().unsqueeze(1)
                    if gt_art.sum() > 0:
                        pr_art = probs[:, 1:2, ...]
                        cl_art = cldice_loss_fn(gt_art, pr_art)
                        loss = loss + cldice_weight_art * cl_art

                # Vein = class 2
                if cldice_weight_vein > 0.0:
                    gt_vein = (lab == 2).float().unsqueeze(1)
                    if gt_vein.sum() > 0:
                        pr_vein = probs[:, 2:3, ...]
                        cl_vein = cldice_loss_fn(gt_vein, pr_vein)
                        loss = loss + cldice_weight_vein * cl_vein

            # Extra A/V classification loss (on vessel voxels)
            if av_head_weight > 0.0:
                av_loss = av_head_loss(
                    logits,
                    lab,
                    class_weights_av=av_class_weights,
                    focal_gamma=av_focal_gamma,
                    use_tversky=av_use_tversky,
                    tversky_alpha=av_tversky_alpha,
                    tversky_beta=av_tversky_beta,
                )
                loss = loss + av_head_weight * av_loss

            # Vessel_head vs A/V union
            if vessel_consistency_weight > 0.0 and hasattr(model, "vessel_head"):
                union_av = probs[:, 1:3, ...].sum(dim=1, keepdim=True)  # (B*,1,...)
                vessel_logits = model.vessel_head(logits)
                vessel_probs  = torch.sigmoid(vessel_logits)

                cons_loss = F.mse_loss(vessel_probs, union_av)
                loss = loss + vessel_consistency_weight * cons_loss

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running.append(loss.item())

        if i == 1:
            # One-time debug: make sure patches are not full volume
            print("DEBUG train patch shape:", img.shape, lab.shape, flush=True)

        if i % 50 == 0 or i == 1:
            print(f"  [train] batch {i}/{len(loader)}  loss={loss.item():.4f}")

    return float(np.mean(running))


@torch.no_grad()
def eval_epoch(
    model,
    loader,
    device,
    cldice_metric=None,
    patch_size=None,
    num_metric_workers: int = 0,
    use_av_refine: bool = False,
):
    """
    Evaluation on full volumes via sliding-window inference.

    Returns:
      - mean Dice (union A∪V)
      - mean hard clDice (union A∪V)
      - mean Dice (artery, class=1)
      - mean hard clDice (artery)
      - mean Dice (vein, class=2)
      - mean hard clDice (vein)
    """
    model.eval()

    # Dice accumulators
    dice_union_scores = []
    dice_art_scores = []
    dice_vein_scores = []

    # clDice accumulators
    cldice_union_scores = []
    cldice_art_scores = []
    cldice_vein_scores = []

    # Downsampled masks for clDice
    cldice_cases_union = []
    cldice_cases_art = []
    cldice_cases_vein = []

    ds_factor = 4  # Spatial downsample factor for clDice masks

    for batch in loader:
        img, lab = batch["image"].to(device), batch["label"].to(device).long()

        if patch_size is not None:
            logits = sliding_window_inference(
                img,
                roi_size=patch_size,
                sw_batch_size=2,
                predictor=model,
            )
        else:
            logits = model(img)

        if use_av_refine and hasattr(model, "av_refine_head"):
            # Base probs
            base_probs = F.softmax(logits, dim=1)        # (B,3,D,H,W)
            p_bg = base_probs[:, 0:1, ...]
            p_union = base_probs[:, 1:3, ...].sum(dim=1, keepdim=True).clamp(0.0, 1.0)

            # Conditional A/V prediction from refine head
            av_logits = model.av_refine_head(logits)     # (B,2,D,H,W)
            av_probs = F.softmax(av_logits, dim=1)
            p_art_cond = av_probs[:, 0:1, ...]
            p_vein_cond = av_probs[:, 1:2, ...]

            p_art = p_union * p_art_cond
            p_vein = p_union * p_vein_cond

            # Renormalize to keep sum ~1
            denom = p_bg + p_art + p_vein + 1e-8
            probs = torch.cat(
                [p_bg / denom, p_art / denom, p_vein / denom],
                dim=1,
            )
        else:
            probs = F.softmax(logits, dim=1)

        pred = probs.argmax(1)  # (B, D, H, W)

        B = pred.shape[0]
        for b in range(B):
            pb = pred[b]
            gb = lab[b]

            # UNION OF VESSELS (A∪V)
            p_union = pb > 0
            g_union = gb > 0

            inter_u = (p_union & g_union).sum().float()
            denom_u = p_union.sum().float() + g_union.sum().float()
            if denom_u > 0:
                dice_u = (2.0 * inter_u) / (denom_u + 1e-5)
                dice_union_scores.append(dice_u.item())

            if cldice_metric is not None:
                p_u_small = p_union[::ds_factor, ::ds_factor, ::ds_factor]
                g_u_small = g_union[::ds_factor, ::ds_factor, ::ds_factor]
                cldice_cases_union.append(
                    (
                        p_u_small.cpu().numpy().astype(bool),
                        g_u_small.cpu().numpy().astype(bool),
                    )
                )

            # ARTERY (class = 1)
            g_art = gb == 1
            if g_art.any():  # Only evaluate arteries if GT has some artery voxels
                p_art = pb == 1

                inter_a = (p_art & g_art).sum().float()
                denom_a = p_art.sum().float() + g_art.sum().float()
                if denom_a > 0:
                    dice_a = (2.0 * inter_a) / (denom_a + 1e-5)
                    dice_art_scores.append(dice_a.item())

                    if cldice_metric is not None:
                        p_a_small = p_art[::ds_factor, ::ds_factor, ::ds_factor]
                        g_a_small = g_art[::ds_factor, ::ds_factor, ::ds_factor]
                        cldice_cases_art.append(
                            (
                                p_a_small.cpu().numpy().astype(bool),
                                g_a_small.cpu().numpy().astype(bool),
                            )
                        )

            # VEIN (class = 2)
            g_vein = gb == 2
            if g_vein.any():  # Only evaluate veins if GT has some vein voxels
                p_vein = pb == 2

                inter_v = (p_vein & g_vein).sum().float()
                denom_v = p_vein.sum().float() + g_vein.sum().float()
                if denom_v > 0:
                    dice_v = (2.0 * inter_v) / (denom_v + 1e-5)
                    dice_vein_scores.append(dice_v.item())

                    if cldice_metric is not None:
                        p_v_small = p_vein[::ds_factor, ::ds_factor, ::ds_factor]
                        g_v_small = g_vein[::ds_factor, ::ds_factor, ::ds_factor]
                        cldice_cases_vein.append(
                            (
                                p_v_small.cpu().numpy().astype(bool),
                                g_v_small.cpu().numpy().astype(bool),
                            )
                        )

    # Helper to compute clDice for a list of (pred, gt) cases
    def _compute_cldice_list(cases):
        if cldice_metric is None or not cases:
            return []
        if num_metric_workers > 0:
            print(
                f"[eval_epoch] clDice over {len(cases)} cases "
                f"with {num_metric_workers} workers",
                flush=True,
            )
            with mp.Pool(processes=num_metric_workers) as pool:
                return pool.starmap(
                    _compute_cldice_worker,
                    [(p_np, g_np, cldice_metric) for (p_np, g_np) in cases],
                )
        else:
            return [
                _compute_cldice_worker(p_np, g_np, cldice_metric)
                for (p_np, g_np) in cases
            ]

    # Compute clDice for union, artery, vein
    cldice_union_scores.extend(_compute_cldice_list(cldice_cases_union))
    cldice_art_scores.extend(_compute_cldice_list(cldice_cases_art))
    cldice_vein_scores.extend(_compute_cldice_list(cldice_cases_vein))

    # Means (fall back to 0.0 if no samples)
    mean_dice_union = float(np.mean(dice_union_scores)) if dice_union_scores else 0.0
    mean_dice_art = float(np.mean(dice_art_scores)) if dice_art_scores else 0.0
    mean_dice_vein = float(np.mean(dice_vein_scores)) if dice_vein_scores else 0.0

    mean_cldice_union = (
        float(np.mean(cldice_union_scores)) if cldice_union_scores else 0.0
    )
    mean_cldice_art = (
        float(np.mean(cldice_art_scores)) if cldice_art_scores else 0.0
    )
    mean_cldice_vein = (
        float(np.mean(cldice_vein_scores)) if cldice_vein_scores else 0.0
    )

    return (
        mean_dice_union,
        mean_cldice_union,
        mean_dice_art,
        mean_cldice_art,
        mean_dice_vein,
        mean_cldice_vein,
    )



def make_items_from_dirs(image_dir, label_dir):
    image_dir = pathlib.Path(image_dir)
    label_dir = pathlib.Path(label_dir)
    items = []

    for img_path in sorted(image_dir.glob("*.nii*")):
        img_name = img_path.name

        # Handle pattern: image_###.nii.gz -> label_###.nii.gz
        if img_name.startswith("image_"):
            lbl_name = "label_" + img_name[len("image_"):]
        else:
            # Fallback: same filename if matching names
            lbl_name = img_name

        lab_path = label_dir / lbl_name
        if not lab_path.exists():
            print(f"WARNING: no label for {img_name}, expected {lab_path}")
            continue

        items.append((str(img_path), str(lab_path)))

    if not items:
        raise RuntimeError(
            f"No image/label pairs found in {image_dir} and {label_dir}. "
            f"Check that filenames follow image_### / label_### pattern."
        )

    return items


def make_loader(kind, cfg, train=True):
    if kind == "train":
        items = make_items_from_dirs(
            cfg["data"]["train_images"],
            cfg["data"]["train_labels"],
        )
    else:
        items = make_items_from_dirs(
            cfg["data"]["val_images"],
            cfg["data"]["val_labels"],
        )

    ds = NiftiVolume(items, cfg, train=train)
    aug = make_aug_transforms(cfg, train=train)
    ds.set_transform(aug)

    batch_size = cfg["optim"]["batch_size"] if train else 1

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=train,
        num_workers=cfg["data"].get("num_workers", 8),
        pin_memory=True,
    )


def main(cfg):
    history = {
    "epoch": [],
    "stage": [],
    "train_loss": [],
    # Union-of-vessels (A∪V)
    "val_dice": [],
    "val_clDice": [],
    # Class-wise metrics
    "val_dice_artery": [],
    "val_clDice_artery": [],
    "val_dice_vein": [],
    "val_clDice_vein": [],
    }

    val_cldice_workers = cfg["optim"].get("val_cldice_workers", 0)
    print("val_cldice_workers =", val_cldice_workers, flush=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    set_seed(cfg["seed"])

    # NiftiVolume handles patch sampling for training.
    train_patch_size = None          # no extra random_multi_crop_3d
    eval_patch_size  = cfg["data"]["patch_size"]  # use for sliding window
    samples_per_volume = 1

    # Data
    train_loader = make_loader("train", cfg, train=True)
    val_loader   = make_loader("val", cfg, train=False)

    # Debug: grab one batch to make sure loader works
    first_batch = next(iter(train_loader))
    print("DEBUG: first_batch image shape:", first_batch["image"].shape)
    print("DEBUG: first_batch label shape:", first_batch["label"].shape)

    # Model
    model = build_model(
        num_classes=cfg["model"]["num_classes"],
        dropout=cfg["model"].get("dropout", 0.0),
    )

    # Add AV refine head (2-class) on top of 3-class logits
    if not hasattr(model, "av_refine_head"):
        model.av_refine_head = nn.Conv3d(
            cfg["model"]["num_classes"],  # In_channels = 3
            2,                             # Artery / vein
            kernel_size=1,
        )

    # Load VesselFM backbone weights, but ignore mismatched head
    pre_ckpt = cfg["model"].get("pretrain_ckpt", None)
    if pre_ckpt:
        print(f"Loading pre-trained VesselFM weights from {pre_ckpt}")
        ckpt = torch.load(pre_ckpt, map_location="cpu")
        state = ckpt.get("state_dict", ckpt)

        model_state = model.state_dict()
        filtered_state = {}

        for k, v in state.items():
            if k in model_state and v.shape == model_state[k].shape:
                filtered_state[k] = v
            else:
                # This will include the 1-channel output_block head params
                print(f"Skipping {k}: ckpt {tuple(v.shape)} vs model {tuple(model_state.get(k, torch.empty(0)).shape)}")

        # Now load only compatible weights
        missing, unexpected = model.load_state_dict(filtered_state, strict=False)
        print(f"Loaded pre-trained backbone with {len(missing)} missing and {len(unexpected)} unexpected keys")

    model.to(device)

    # Extra A/V classification head loss hyperparams
    av_head_weight = cfg["loss"].get("av_head_weight", 1.0)
    av_class_weights = cfg["loss"].get("av_class_weights", [1.0, 1.0])

    # Focal + Tversky for A/V head
    av_focal_gamma = cfg["loss"].get("av_focal_gamma", 0.0)
    av_use_tversky = cfg["loss"].get("av_use_tversky", False)
    av_tversky_alpha = cfg["loss"].get("av_tversky_alpha", 0.5)
    av_tversky_beta  = cfg["loss"].get("av_tversky_beta", 0.5)

    # Loss (Dice+CE etc.)
    loss_fn = CompositeLoss(
        num_classes=cfg["model"]["num_classes"],
        class_weights=cfg["loss"]["class_weights"],
        soft_cldice_weight=0.0,
        soft_cldice_iters=cfg["loss"]["soft_cldice_iters"],
    ).to(device)

    # clDice weights per stage (from YAML)
    use_soft = cfg["loss"].get("use_soft_cldice", False)
    cldice_weight_s1 = cfg["loss"].get("soft_cldice_weight_stage1", 0.0) if use_soft else 0.0
    cldice_weight_s2 = cfg["loss"].get("soft_cldice_weight_stage2", 0.0) if use_soft else 0.0

    # Per-class clDice weights + consistency
    cldice_weight_art   = cfg["loss"].get("soft_cldice_weight_art", 0.0)
    cldice_weight_vein  = cfg["loss"].get("soft_cldice_weight_vein", 0.0)
    vessel_consistency_weight = cfg["loss"].get("vessel_consistency_weight", 0.0)

    cldice_loss_fn = None
    if (cldice_weight_s1 > 0.0) or (cldice_weight_s2 > 0.0):
        cldice_loss_fn = SoftCLDiceLoss(
            iter_=cfg["loss"]["soft_cldice_iters"], smooth=1.0
        ).to(device)


    # Stage 1: freeze backbone, train head and decoder
    freeze_backbone(model)
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["optim"]["lr_stage1"],
        weight_decay=cfg["optim"]["weight_decay"],
    )
    scaler = GradScaler(enabled=cfg["optim"]["amp"])

    best_cl = -1.0
    for epoch in range(cfg["optim"]["epochs_stage1"]):
        tr = one_epoch(
            model,
            train_loader,
            loss_fn,
            opt,
            scaler,
            device,
            amp=cfg["optim"]["amp"],
            cldice_loss_fn=cldice_loss_fn,
            cldice_weight=cldice_weight_s1,
            patch_size=train_patch_size,
            samples_per_volume=samples_per_volume,
            av_head_weight=av_head_weight,
            av_class_weights=av_class_weights,
            cldice_weight_art=cldice_weight_art,
            cldice_weight_vein=cldice_weight_vein,
            vessel_consistency_weight=vessel_consistency_weight,
            av_focal_gamma=av_focal_gamma,
            av_use_tversky=av_use_tversky,
            av_tversky_alpha=av_tversky_alpha,
            av_tversky_beta=av_tversky_beta,
        )

        (
            va_dice_union,
            va_cldice_union,
            va_dice_art,
            va_cldice_art,
            va_dice_vein,
            va_cldice_vein,
        ) = eval_epoch(
            model,
            val_loader,
            device,
            cldice_metric=hard_cldice,
            patch_size=eval_patch_size,
            num_metric_workers=val_cldice_workers,
        )

        history["epoch"].append(epoch + 1)
        history["stage"].append("S1")
        history["train_loss"].append(tr)

        # union (keep old column names)
        history["val_dice"].append(va_dice_union)
        history["val_clDice"].append(va_cldice_union)

        # NEW: per-class
        history["val_dice_artery"].append(va_dice_art)
        history["val_clDice_artery"].append(va_cldice_art)
        history["val_dice_vein"].append(va_dice_vein)
        history["val_clDice_vein"].append(va_cldice_vein)

        # still pick best model by union clDice (A∪V)
        if va_cldice_union > best_cl:
            best_cl = va_cldice_union
            torch.save(
                model.state_dict(),
                f"checkpoints/{cfg['experiment']}_best_cldice.pt",
            )

        print(
            f"[S1][{epoch+1}/{cfg['optim']['epochs_stage1']}] "
            f"loss={tr:.4f} "
            f"valDice(A∪V)={va_dice_union:.4f} valClDice(A∪V)={va_cldice_union:.4f} "
            f"valDice(art)={va_dice_art:.4f} valClDice(art)={va_cldice_art:.4f} "
            f"valDice(vein)={va_dice_vein:.4f} valClDice(vein)={va_cldice_vein:.4f}"
        )

    # Stage 2: continue training head with lower LR
    unfreeze_encoder_tail(model, n_stages=2)
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["optim"]["lr_stage2"],
        weight_decay=cfg["optim"]["weight_decay"],
    )

    for epoch in range(cfg["optim"]["epochs_stage2"]):
        tr = one_epoch(
            model,
            train_loader,
            loss_fn,
            opt,
            scaler,
            device,
            amp=cfg["optim"]["amp"],
            cldice_loss_fn=cldice_loss_fn,
            cldice_weight=cldice_weight_s2,
            patch_size=train_patch_size,
            samples_per_volume=samples_per_volume,
            av_head_weight=av_head_weight,
            av_class_weights=av_class_weights,
            cldice_weight_art=cldice_weight_art,
            cldice_weight_vein=cldice_weight_vein,
            vessel_consistency_weight=vessel_consistency_weight,
            av_focal_gamma=av_focal_gamma,
            av_use_tversky=av_use_tversky,
            av_tversky_alpha=av_tversky_alpha,
            av_tversky_beta=av_tversky_beta,
        )
        (
            va_dice_union,
            va_cldice_union,
            va_dice_art,
            va_cldice_art,
            va_dice_vein,
            va_cldice_vein,
        ) = eval_epoch(
            model,
            val_loader,
            device,
            cldice_metric=hard_cldice,
            patch_size=eval_patch_size,
            num_metric_workers=val_cldice_workers,
        )

        history["epoch"].append(epoch + 1)
        history["stage"].append("S2")
        history["train_loss"].append(tr)

        history["val_dice"].append(va_dice_union)
        history["val_clDice"].append(va_cldice_union)
        history["val_dice_artery"].append(va_dice_art)
        history["val_clDice_artery"].append(va_cldice_art)
        history["val_dice_vein"].append(va_dice_vein)
        history["val_clDice_vein"].append(va_cldice_vein)

        if va_cldice_union > best_cl:
            best_cl = va_cldice_union
            torch.save(
                model.state_dict(),
                f"checkpoints/{cfg['experiment']}_best_cldice.pt",
            )

        print(
            f"[S2][{epoch+1}/{cfg['optim']['epochs_stage2']}] "
            f"loss={tr:.4f} "
            f"valDice(A∪V)={va_dice_union:.4f} valClDice(A∪V)={va_cldice_union:.4f} "
            f"valDice(art)={va_dice_art:.4f} valClDice(art)={va_cldice_art:.4f} "
            f"valDice(vein)={va_dice_vein:.4f} valClDice(vein)={va_cldice_vein:.4f}"
        )

    # Stage 3: freeze backbone, train only av_refine_head
    lr_av_refine = cfg["optim"].get("lr_av_refine", 0.0)
    epochs_av_refine = cfg["optim"].get("epochs_av_refine", 0)

    if epochs_av_refine > 0 and lr_av_refine > 0.0:
        print("\n=== Stage 3: AV refine head training ===", flush=True)

        # Freeze everything except av_refine_head
        for name, p in model.named_parameters():
            if "av_refine_head" in name:
                p.requires_grad = True
            else:
                p.requires_grad = False

        opt_av = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr_av_refine,
            weight_decay=cfg["optim"]["weight_decay"],
        )

        class_weights_refine = cfg["loss"].get(
            "av_refine_class_weights",
            cfg["loss"].get("av_class_weights", [1.0, 1.0]),
        )

        for epoch in range(epochs_av_refine):
            tr = one_epoch_av_refine(
                model,
                train_loader,
                opt_av,
                scaler,
                device,
                amp=cfg["optim"]["amp"],
                class_weights_av=class_weights_refine,
            )

            (
                va_dice_union,
                va_cldice_union,
                va_dice_art,
                va_cldice_art,
                va_dice_vein,
                va_cldice_vein,
            ) = eval_epoch(
                model,
                val_loader,
                device,
                cldice_metric=hard_cldice,
                patch_size=eval_patch_size,
                num_metric_workers=val_cldice_workers,
                use_av_refine=True,
            )

            history["epoch"].append(epoch + 1)
            history["stage"].append("S3")
            history["train_loss"].append(tr)
            history["val_dice"].append(va_dice_union)
            history["val_clDice"].append(va_cldice_union)
            history["val_dice_artery"].append(va_dice_art)
            history["val_clDice_artery"].append(va_cldice_art)
            history["val_dice_vein"].append(va_dice_vein)
            history["val_clDice_vein"].append(va_cldice_vein)

            if va_cldice_union > best_cl:
                best_cl = va_cldice_union
                torch.save(
                    model.state_dict(),
                    f"checkpoints/{cfg['experiment']}_best_cldice.pt",
                )

            print(
                f"[S3][{epoch+1}/{epochs_av_refine}] "
                f"loss={tr:.4f} "
                f"valDice(A∪V)={va_dice_union:.4f} valClDice(A∪V)={va_cldice_union:.4f} "
                f"valDice(art)={va_dice_art:.4f} valClDice(art)={va_cldice_art:.4f} "
                f"valDice(vein)={va_dice_vein:.4f} valClDice(vein)={va_cldice_vein:.4f}"
            )

    try:
        import pandas as pd

        os.makedirs("checkpoints", exist_ok=True)
        pd.DataFrame(history).to_csv(f"checkpoints/{cfg['experiment']}_training_curve.csv", index=False)
    except Exception as e:
        print("Could not save training curve CSV:", e)

In [ ]:
import os
import sys
import yaml

# Point to repo root
repo_root = "/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/VesselFMAdaptationMethod/vesselFM-main/vesselFM-main"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from vesselfm.seg.train_av import main

config_path = os.path.join(repo_root, "configs", "av_ct.yaml")
with open(config_path) as f:
    cfg = yaml.safe_load(f)

main(cfg)

val_cldice_workers = 11
DEBUG: first_batch image shape: torch.Size([2, 1, 96, 96, 96])
DEBUG: first_batch label shape: torch.Size([2, 96, 96, 96])
Loading pre-trained VesselFM weights from /projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/VesselFMAdaptationMethod/vesselFM-main/vesselFM-main/checkpoints/vessel_ct_vessel_best.pt
Loaded pre-trained backbone with 2 missing and 0 unexpected keys
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=11.9369
  [train] batch 50/560  loss=9.8356
  [train] batch 100/560  loss=10.3510
  [train] batch 150/560  loss=9.4378
  [train] batch 200/560  loss=8.7589
  [train] batch 250/560  loss=9.1517
  [train] batch 300/560  loss=8.2694
  [train] batch 350/560  loss=8.6005
  [train] batch 400/560  loss=8.8251
  [train] batch 450/560  loss=7.7012
  [train] batch 500/560  loss=7.5458
  [train] batch 550/560  loss=7.3708
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][1/60] loss=8.7904 valDice(A∪V)=0.0118 valClDice(A∪V)=0.0225 valDice(art)=0.0011 valClDice(art)=0.0020 valDice(vein)=0.0328 valClDice(vein)=0.0744
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=7.0954
  [train] batch 50/560  loss=6.7396
  [train] batch 100/560  loss=6.8517
  [train] batch 150/560  loss=6.5536
  [train] batch 200/560  loss=6.3753
  [train] batch 250/560  loss=6.1330
  [train] batch 300/560  loss=5.5790
  [train] batch 350/560  loss=5.2328
  [train] batch 400/560  loss=4.9355
  [train] batch 450/560  loss=4.7542
  [train] batch 500/560  loss=4.3985
  [train] batch 550/560  loss=4.3312
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][2/60] loss=5.7546 valDice(A∪V)=0.0116 valClDice(A∪V)=0.0234 valDice(art)=0.0024 valClDice(art)=0.0047 valDice(vein)=0.0272 valClDice(vein)=0.0673
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=4.2967
  [train] batch 50/560  loss=4.0533
  [train] batch 100/560  loss=3.9480
  [train] batch 150/560  loss=3.7784
  [train] batch 200/560  loss=3.4211
  [train] batch 250/560  loss=3.2891
  [train] batch 300/560  loss=3.1914
  [train] batch 350/560  loss=2.9455
  [train] batch 400/560  loss=2.7496
  [train] batch 450/560  loss=2.7894
  [train] batch 500/560  loss=2.4822
  [train] batch 550/560  loss=2.5488
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][3/60] loss=3.2438 valDice(A∪V)=0.1165 valClDice(A∪V)=0.1736 valDice(art)=0.0263 valClDice(art)=0.0394 valDice(vein)=0.2517 valClDice(vein)=0.2646
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=2.4123
  [train] batch 50/560  loss=2.3825
  [train] batch 100/560  loss=2.3351
  [train] batch 150/560  loss=2.2663
  [train] batch 200/560  loss=2.1811
  [train] batch 250/560  loss=2.1500
  [train] batch 300/560  loss=2.1479
  [train] batch 350/560  loss=2.0952
  [train] batch 400/560  loss=2.1106
  [train] batch 450/560  loss=2.0043
  [train] batch 500/560  loss=2.0257
  [train] batch 550/560  loss=2.0128
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][4/60] loss=2.1874 valDice(A∪V)=0.2735 valClDice(A∪V)=0.3109 valDice(art)=0.0790 valClDice(art)=0.0821 valDice(vein)=0.2981 valClDice(vein)=0.3017
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.9476
  [train] batch 50/560  loss=2.0284
  [train] batch 100/560  loss=1.9252
  [train] batch 150/560  loss=1.9591
  [train] batch 200/560  loss=2.0396
  [train] batch 250/560  loss=1.9050
  [train] batch 300/560  loss=1.9351
  [train] batch 350/560  loss=1.9087
  [train] batch 400/560  loss=1.9175
  [train] batch 450/560  loss=1.8161
  [train] batch 500/560  loss=1.9861
  [train] batch 550/560  loss=1.8539
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][5/60] loss=1.9318 valDice(A∪V)=0.3373 valClDice(A∪V)=0.3669 valDice(art)=0.1153 valClDice(art)=0.1065 valDice(vein)=0.2933 valClDice(vein)=0.2998
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.8289
  [train] batch 50/560  loss=1.8180
  [train] batch 100/560  loss=1.7942
  [train] batch 150/560  loss=1.9347
  [train] batch 200/560  loss=1.8258
  [train] batch 250/560  loss=1.8434
  [train] batch 300/560  loss=1.7162
  [train] batch 350/560  loss=1.8569
  [train] batch 400/560  loss=1.7726
  [train] batch 450/560  loss=1.9911
  [train] batch 500/560  loss=1.7902
  [train] batch 550/560  loss=1.8129
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][6/60] loss=1.8270 valDice(A∪V)=0.3695 valClDice(A∪V)=0.3933 valDice(art)=0.1488 valClDice(art)=0.1232 valDice(vein)=0.2748 valClDice(vein)=0.2907
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7809
  [train] batch 50/560  loss=1.6279
  [train] batch 100/560  loss=1.8116
  [train] batch 150/560  loss=1.7901
  [train] batch 200/560  loss=1.7410
  [train] batch 250/560  loss=1.7356
  [train] batch 300/560  loss=1.6333
  [train] batch 350/560  loss=1.7824
  [train] batch 400/560  loss=2.4089
  [train] batch 450/560  loss=1.7907
  [train] batch 500/560  loss=1.7623
  [train] batch 550/560  loss=1.7038
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][7/60] loss=1.7608 valDice(A∪V)=0.4008 valClDice(A∪V)=0.4220 valDice(art)=0.1718 valClDice(art)=0.1191 valDice(vein)=0.2751 valClDice(vein)=0.3076
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6174
  [train] batch 50/560  loss=1.6978
  [train] batch 100/560  loss=1.7819
  [train] batch 150/560  loss=1.6692
  [train] batch 200/560  loss=1.6366
  [train] batch 250/560  loss=1.6753
  [train] batch 300/560  loss=1.6843
  [train] batch 350/560  loss=1.7044
  [train] batch 400/560  loss=1.7687
  [train] batch 450/560  loss=1.6906
  [train] batch 500/560  loss=1.7750
  [train] batch 550/560  loss=1.7225
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][8/60] loss=1.7018 valDice(A∪V)=0.4230 valClDice(A∪V)=0.4420 valDice(art)=0.1962 valClDice(art)=0.1363 valDice(vein)=0.2655 valClDice(vein)=0.3012
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7264
  [train] batch 50/560  loss=1.6618
  [train] batch 100/560  loss=1.6429
  [train] batch 150/560  loss=1.6655
  [train] batch 200/560  loss=1.6460
  [train] batch 250/560  loss=1.7532
  [train] batch 300/560  loss=1.5845
  [train] batch 350/560  loss=1.6123
  [train] batch 400/560  loss=1.6190
  [train] batch 450/560  loss=1.6268
  [train] batch 500/560  loss=1.7252
  [train] batch 550/560  loss=1.6890
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][9/60] loss=1.6660 valDice(A∪V)=0.4425 valClDice(A∪V)=0.4594 valDice(art)=0.2157 valClDice(art)=0.1554 valDice(vein)=0.2673 valClDice(vein)=0.3035
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6871
  [train] batch 50/560  loss=1.6061
  [train] batch 100/560  loss=1.6499
  [train] batch 150/560  loss=1.7513
  [train] batch 200/560  loss=1.6898
  [train] batch 250/560  loss=1.6614
  [train] batch 300/560  loss=1.5383
  [train] batch 350/560  loss=1.6175
  [train] batch 400/560  loss=1.6250
  [train] batch 450/560  loss=1.7047
  [train] batch 500/560  loss=1.7057
  [train] batch 550/560  loss=1.6100
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][10/60] loss=1.6513 valDice(A∪V)=0.4549 valClDice(A∪V)=0.4694 valDice(art)=0.2344 valClDice(art)=0.1588 valDice(vein)=0.2658 valClDice(vein)=0.3078
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6115
  [train] batch 50/560  loss=1.5169
  [train] batch 100/560  loss=1.6360
  [train] batch 150/560  loss=1.5654
  [train] batch 200/560  loss=1.6732
  [train] batch 250/560  loss=1.7196
  [train] batch 300/560  loss=1.6050
  [train] batch 350/560  loss=1.5767
  [train] batch 400/560  loss=1.5626
  [train] batch 450/560  loss=1.6842
  [train] batch 500/560  loss=1.6456
  [train] batch 550/560  loss=1.5861
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][11/60] loss=1.6323 valDice(A∪V)=0.4753 valClDice(A∪V)=0.4871 valDice(art)=0.2726 valClDice(art)=0.2076 valDice(vein)=0.2571 valClDice(vein)=0.2915
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5940
  [train] batch 50/560  loss=1.5480
  [train] batch 100/560  loss=1.5242
  [train] batch 150/560  loss=1.5501
  [train] batch 200/560  loss=1.6601
  [train] batch 250/560  loss=1.5458
  [train] batch 300/560  loss=1.6226
  [train] batch 350/560  loss=1.6045
  [train] batch 400/560  loss=1.6633
  [train] batch 450/560  loss=1.6603
  [train] batch 500/560  loss=1.6315
  [train] batch 550/560  loss=1.6101
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][12/60] loss=1.6229 valDice(A∪V)=0.4826 valClDice(A∪V)=0.4927 valDice(art)=0.2953 valClDice(art)=0.2503 valDice(vein)=0.2444 valClDice(vein)=0.2648
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5799
  [train] batch 50/560  loss=1.5989
  [train] batch 100/560  loss=1.5386
  [train] batch 150/560  loss=1.6246
  [train] batch 200/560  loss=1.5188
  [train] batch 250/560  loss=1.5694
  [train] batch 300/560  loss=1.5098
  [train] batch 350/560  loss=1.6045
  [train] batch 400/560  loss=1.7251
  [train] batch 450/560  loss=1.5925
  [train] batch 500/560  loss=1.5570
  [train] batch 550/560  loss=1.4722
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][13/60] loss=1.6126 valDice(A∪V)=0.4978 valClDice(A∪V)=0.5063 valDice(art)=0.2882 valClDice(art)=0.2028 valDice(vein)=0.2714 valClDice(vein)=0.3147
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5818
  [train] batch 50/560  loss=1.6236
  [train] batch 100/560  loss=1.6478
  [train] batch 150/560  loss=1.6520
  [train] batch 200/560  loss=1.7045
  [train] batch 250/560  loss=1.6288
  [train] batch 300/560  loss=1.4364
  [train] batch 350/560  loss=1.6937
  [train] batch 400/560  loss=1.5210
  [train] batch 450/560  loss=1.6530
  [train] batch 500/560  loss=1.5178
  [train] batch 550/560  loss=1.5369
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][14/60] loss=1.6079 valDice(A∪V)=0.5073 valClDice(A∪V)=0.5145 valDice(art)=0.2974 valClDice(art)=0.2383 valDice(vein)=0.2710 valClDice(vein)=0.3023
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5853
  [train] batch 50/560  loss=1.6317
  [train] batch 100/560  loss=1.5881
  [train] batch 150/560  loss=1.6233
  [train] batch 200/560  loss=1.6070
  [train] batch 250/560  loss=1.6250
  [train] batch 300/560  loss=1.4911
  [train] batch 350/560  loss=1.6152
  [train] batch 400/560  loss=1.5012
  [train] batch 450/560  loss=1.5474
  [train] batch 500/560  loss=1.6186
  [train] batch 550/560  loss=2.3079
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][15/60] loss=1.5985 valDice(A∪V)=0.5159 valClDice(A∪V)=0.5238 valDice(art)=0.3173 valClDice(art)=0.2659 valDice(vein)=0.2656 valClDice(vein)=0.2908
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5433
  [train] batch 50/560  loss=1.5706
  [train] batch 100/560  loss=1.6481
  [train] batch 150/560  loss=1.5500
  [train] batch 200/560  loss=1.7143
  [train] batch 250/560  loss=1.5504
  [train] batch 300/560  loss=1.5733
  [train] batch 350/560  loss=1.5417
  [train] batch 400/560  loss=1.5234
  [train] batch 450/560  loss=1.5121
  [train] batch 500/560  loss=1.6194
  [train] batch 550/560  loss=1.5604
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][16/60] loss=1.6005 valDice(A∪V)=0.5238 valClDice(A∪V)=0.5312 valDice(art)=0.3117 valClDice(art)=0.2413 valDice(vein)=0.2796 valClDice(vein)=0.3134
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5958
  [train] batch 50/560  loss=1.7342
  [train] batch 100/560  loss=1.6624
  [train] batch 150/560  loss=1.5850
  [train] batch 200/560  loss=1.5458
  [train] batch 250/560  loss=1.5225
  [train] batch 300/560  loss=1.6072
  [train] batch 350/560  loss=1.4975
  [train] batch 400/560  loss=1.5822
  [train] batch 450/560  loss=1.6183
  [train] batch 500/560  loss=1.5316
  [train] batch 550/560  loss=1.5733
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][17/60] loss=1.5913 valDice(A∪V)=0.5368 valClDice(A∪V)=0.5434 valDice(art)=0.3220 valClDice(art)=0.2590 valDice(vein)=0.2838 valClDice(vein)=0.3125
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6535
  [train] batch 50/560  loss=1.5830
  [train] batch 100/560  loss=1.5552
  [train] batch 150/560  loss=1.5063
  [train] batch 200/560  loss=1.6461
  [train] batch 250/560  loss=1.5871
  [train] batch 300/560  loss=1.5667
  [train] batch 350/560  loss=1.6304
  [train] batch 400/560  loss=1.5790
  [train] batch 450/560  loss=1.5878
  [train] batch 500/560  loss=1.9272
  [train] batch 550/560  loss=1.6882
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][18/60] loss=1.5883 valDice(A∪V)=0.5389 valClDice(A∪V)=0.5466 valDice(art)=0.3334 valClDice(art)=0.2827 valDice(vein)=0.2766 valClDice(vein)=0.2984
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.4672
  [train] batch 50/560  loss=1.5925
  [train] batch 100/560  loss=1.5085
  [train] batch 150/560  loss=1.5775
  [train] batch 200/560  loss=1.5680
  [train] batch 250/560  loss=1.7115
  [train] batch 300/560  loss=1.6431
  [train] batch 350/560  loss=1.5786
  [train] batch 400/560  loss=1.5363
  [train] batch 450/560  loss=1.5661
  [train] batch 500/560  loss=1.5467
  [train] batch 550/560  loss=1.5564
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][19/60] loss=1.5915 valDice(A∪V)=0.5437 valClDice(A∪V)=0.5524 valDice(art)=0.3243 valClDice(art)=0.2492 valDice(vein)=0.2910 valClDice(vein)=0.3253
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7171
  [train] batch 50/560  loss=1.4978
  [train] batch 100/560  loss=1.5904
  [train] batch 150/560  loss=1.5910
  [train] batch 200/560  loss=1.5823
  [train] batch 250/560  loss=1.6868
  [train] batch 300/560  loss=1.5259
  [train] batch 350/560  loss=1.5735
  [train] batch 400/560  loss=1.6202
  [train] batch 450/560  loss=1.5619
  [train] batch 500/560  loss=1.5190
  [train] batch 550/560  loss=1.5648
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][20/60] loss=1.5899 valDice(A∪V)=0.5559 valClDice(A∪V)=0.5640 valDice(art)=0.3384 valClDice(art)=0.2762 valDice(vein)=0.2913 valClDice(vein)=0.3179
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5612
  [train] batch 50/560  loss=1.6193
  [train] batch 100/560  loss=2.2951
  [train] batch 150/560  loss=1.4411
  [train] batch 200/560  loss=1.5106
  [train] batch 250/560  loss=1.6109
  [train] batch 300/560  loss=1.5291
  [train] batch 350/560  loss=1.6393
  [train] batch 400/560  loss=1.5462
  [train] batch 450/560  loss=1.5362
  [train] batch 500/560  loss=1.5033
  [train] batch 550/560  loss=1.6459
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][21/60] loss=1.5862 valDice(A∪V)=0.5582 valClDice(A∪V)=0.5680 valDice(art)=0.3434 valClDice(art)=0.2662 valDice(vein)=0.2928 valClDice(vein)=0.3238
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5458
  [train] batch 50/560  loss=1.7055
  [train] batch 100/560  loss=1.5404
  [train] batch 150/560  loss=1.4689
  [train] batch 200/560  loss=1.6274
  [train] batch 250/560  loss=1.5816
  [train] batch 300/560  loss=1.4657
  [train] batch 350/560  loss=1.5834
  [train] batch 400/560  loss=1.4780
  [train] batch 450/560  loss=1.5575
  [train] batch 500/560  loss=1.5805
  [train] batch 550/560  loss=1.6392
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][22/60] loss=1.5845 valDice(A∪V)=0.5644 valClDice(A∪V)=0.5741 valDice(art)=0.3498 valClDice(art)=0.2747 valDice(vein)=0.2941 valClDice(vein)=0.3232
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5286
  [train] batch 50/560  loss=1.6386
  [train] batch 100/560  loss=1.5232
  [train] batch 150/560  loss=1.5533
  [train] batch 200/560  loss=1.7496
  [train] batch 250/560  loss=1.5607
  [train] batch 300/560  loss=1.5625
  [train] batch 350/560  loss=1.4850
  [train] batch 400/560  loss=1.6286
  [train] batch 450/560  loss=1.5384
  [train] batch 500/560  loss=1.5635
  [train] batch 550/560  loss=1.5427
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][23/60] loss=1.5840 valDice(A∪V)=0.5678 valClDice(A∪V)=0.5784 valDice(art)=0.3425 valClDice(art)=0.2626 valDice(vein)=0.3018 valClDice(vein)=0.3327
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.4857
  [train] batch 50/560  loss=1.6180
  [train] batch 100/560  loss=1.5518
  [train] batch 150/560  loss=1.6815
  [train] batch 200/560  loss=1.5678
  [train] batch 250/560  loss=1.6207
  [train] batch 300/560  loss=1.5653
  [train] batch 350/560  loss=1.6151
  [train] batch 400/560  loss=1.6148
  [train] batch 450/560  loss=1.5296
  [train] batch 500/560  loss=1.6525
  [train] batch 550/560  loss=1.5792
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][24/60] loss=1.5814 valDice(A∪V)=0.5733 valClDice(A∪V)=0.5847 valDice(art)=0.3650 valClDice(art)=0.3050 valDice(vein)=0.2895 valClDice(vein)=0.3076
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5064
  [train] batch 50/560  loss=1.6113
  [train] batch 100/560  loss=1.5786
  [train] batch 150/560  loss=1.5609
  [train] batch 200/560  loss=1.5528
  [train] batch 250/560  loss=2.1821
  [train] batch 300/560  loss=1.5439
  [train] batch 350/560  loss=1.4973
  [train] batch 400/560  loss=1.5777
  [train] batch 450/560  loss=1.5897
  [train] batch 500/560  loss=1.7145
  [train] batch 550/560  loss=1.5036
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][25/60] loss=1.5825 valDice(A∪V)=0.5713 valClDice(A∪V)=0.5837 valDice(art)=0.3355 valClDice(art)=0.2245 valDice(vein)=0.3107 valClDice(vein)=0.3520
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.4887
  [train] batch 50/560  loss=2.0752
  [train] batch 100/560  loss=1.6462
  [train] batch 150/560  loss=1.4999
  [train] batch 200/560  loss=1.5828
  [train] batch 250/560  loss=1.4890
  [train] batch 300/560  loss=1.6046
  [train] batch 350/560  loss=1.6116
  [train] batch 400/560  loss=1.6739
  [train] batch 450/560  loss=1.5850
  [train] batch 500/560  loss=1.4802
  [train] batch 550/560  loss=1.6489
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][26/60] loss=1.5831 valDice(A∪V)=0.5802 valClDice(A∪V)=0.5920 valDice(art)=0.3587 valClDice(art)=0.2959 valDice(vein)=0.3010 valClDice(vein)=0.3224
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5280
  [train] batch 50/560  loss=1.5300
  [train] batch 100/560  loss=1.5937
  [train] batch 150/560  loss=1.6516
  [train] batch 200/560  loss=1.5929
  [train] batch 250/560  loss=1.6169
  [train] batch 300/560  loss=1.4985
  [train] batch 350/560  loss=1.6852
  [train] batch 400/560  loss=1.6487
  [train] batch 450/560  loss=1.5746
  [train] batch 500/560  loss=1.6294
  [train] batch 550/560  loss=1.5373
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][27/60] loss=1.5826 valDice(A∪V)=0.5825 valClDice(A∪V)=0.5949 valDice(art)=0.3538 valClDice(art)=0.2766 valDice(vein)=0.3078 valClDice(vein)=0.3358
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6042
  [train] batch 50/560  loss=1.4840
  [train] batch 100/560  loss=1.5699
  [train] batch 150/560  loss=1.5536
  [train] batch 200/560  loss=1.4968
  [train] batch 250/560  loss=2.5251
  [train] batch 300/560  loss=1.5598
  [train] batch 350/560  loss=1.6099
  [train] batch 400/560  loss=1.5892
  [train] batch 450/560  loss=1.6207
  [train] batch 500/560  loss=1.5704
  [train] batch 550/560  loss=1.5874
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][28/60] loss=1.5793 valDice(A∪V)=0.5855 valClDice(A∪V)=0.5979 valDice(art)=0.3873 valClDice(art)=0.3458 valDice(vein)=0.2785 valClDice(vein)=0.2751
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5485
  [train] batch 50/560  loss=1.6644
  [train] batch 100/560  loss=1.5579
  [train] batch 150/560  loss=1.5454
  [train] batch 200/560  loss=1.5111
  [train] batch 250/560  loss=1.4941
  [train] batch 300/560  loss=1.5741
  [train] batch 350/560  loss=1.5970
  [train] batch 400/560  loss=1.6653
  [train] batch 450/560  loss=1.5460
  [train] batch 500/560  loss=1.5488
  [train] batch 550/560  loss=1.6809
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][29/60] loss=1.5772 valDice(A∪V)=0.5890 valClDice(A∪V)=0.6024 valDice(art)=0.3812 valClDice(art)=0.2970 valDice(vein)=0.2950 valClDice(vein)=0.3178
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6154
  [train] batch 50/560  loss=1.5914
  [train] batch 100/560  loss=1.4890
  [train] batch 150/560  loss=1.6379
  [train] batch 200/560  loss=1.5933
  [train] batch 250/560  loss=1.6365
  [train] batch 300/560  loss=1.5951
  [train] batch 350/560  loss=1.5197
  [train] batch 400/560  loss=1.6998
  [train] batch 450/560  loss=1.4681
  [train] batch 500/560  loss=1.5843
  [train] batch 550/560  loss=1.4793
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][30/60] loss=1.5798 valDice(A∪V)=0.5885 valClDice(A∪V)=0.6019 valDice(art)=0.3737 valClDice(art)=0.3046 valDice(vein)=0.2996 valClDice(vein)=0.3191
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6258
  [train] batch 50/560  loss=1.5661
  [train] batch 100/560  loss=1.6027
  [train] batch 150/560  loss=1.5547
  [train] batch 200/560  loss=1.5906
  [train] batch 250/560  loss=1.4758
  [train] batch 300/560  loss=1.6169
  [train] batch 350/560  loss=1.4579
  [train] batch 400/560  loss=1.5531
  [train] batch 450/560  loss=1.6431
  [train] batch 500/560  loss=1.5304
  [train] batch 550/560  loss=1.5992
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][31/60] loss=1.5808 valDice(A∪V)=0.5924 valClDice(A∪V)=0.6070 valDice(art)=0.3690 valClDice(art)=0.2857 valDice(vein)=0.3070 valClDice(vein)=0.3335
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5144
  [train] batch 50/560  loss=1.5769
  [train] batch 100/560  loss=1.6187
  [train] batch 150/560  loss=1.4989
  [train] batch 200/560  loss=1.5118
  [train] batch 250/560  loss=1.6020
  [train] batch 300/560  loss=1.6405
  [train] batch 350/560  loss=1.6723
  [train] batch 400/560  loss=1.5519
  [train] batch 450/560  loss=1.6302
  [train] batch 500/560  loss=1.5881
  [train] batch 550/560  loss=1.5641
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][32/60] loss=1.5759 valDice(A∪V)=0.5938 valClDice(A∪V)=0.6081 valDice(art)=0.3718 valClDice(art)=0.3006 valDice(vein)=0.3040 valClDice(vein)=0.3234
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5337
  [train] batch 50/560  loss=1.5927
  [train] batch 100/560  loss=1.4749
  [train] batch 150/560  loss=1.6450
  [train] batch 200/560  loss=1.6153
  [train] batch 250/560  loss=1.4652
  [train] batch 300/560  loss=1.5016
  [train] batch 350/560  loss=1.5814
  [train] batch 400/560  loss=1.5898
  [train] batch 450/560  loss=1.6713
  [train] batch 500/560  loss=1.5593
  [train] batch 550/560  loss=1.6145
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][33/60] loss=1.5800 valDice(A∪V)=0.5915 valClDice(A∪V)=0.6063 valDice(art)=0.3662 valClDice(art)=0.3077 valDice(vein)=0.3046 valClDice(vein)=0.3212
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6865
  [train] batch 50/560  loss=1.6607
  [train] batch 100/560  loss=1.4516
  [train] batch 150/560  loss=1.6159
  [train] batch 200/560  loss=1.5730
  [train] batch 250/560  loss=1.5476
  [train] batch 300/560  loss=1.5772
  [train] batch 350/560  loss=1.6197
  [train] batch 400/560  loss=1.5145
  [train] batch 450/560  loss=1.4631
  [train] batch 500/560  loss=1.6255
  [train] batch 550/560  loss=1.4745
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][34/60] loss=1.5783 valDice(A∪V)=0.5940 valClDice(A∪V)=0.6086 valDice(art)=0.3740 valClDice(art)=0.3069 valDice(vein)=0.3037 valClDice(vein)=0.3212
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5127
  [train] batch 50/560  loss=1.6716
  [train] batch 100/560  loss=1.5599
  [train] batch 150/560  loss=1.7119
  [train] batch 200/560  loss=1.5859
  [train] batch 250/560  loss=1.5433
  [train] batch 300/560  loss=1.5844
  [train] batch 350/560  loss=1.6515
  [train] batch 400/560  loss=1.5733
  [train] batch 450/560  loss=1.6073
  [train] batch 500/560  loss=1.5842
  [train] batch 550/560  loss=1.5596
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][35/60] loss=1.5820 valDice(A∪V)=0.5934 valClDice(A∪V)=0.6080 valDice(art)=0.3438 valClDice(art)=0.2364 valDice(vein)=0.3244 valClDice(vein)=0.3629
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5929
  [train] batch 50/560  loss=1.6587
  [train] batch 100/560  loss=1.6253
  [train] batch 150/560  loss=1.6135
  [train] batch 200/560  loss=1.7225
  [train] batch 250/560  loss=1.4786
  [train] batch 300/560  loss=1.7377
  [train] batch 350/560  loss=1.6547
  [train] batch 400/560  loss=1.6157
  [train] batch 450/560  loss=1.5130
  [train] batch 500/560  loss=1.5863
  [train] batch 550/560  loss=1.5917
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][36/60] loss=1.5787 valDice(A∪V)=0.5948 valClDice(A∪V)=0.6095 valDice(art)=0.3804 valClDice(art)=0.3353 valDice(vein)=0.2963 valClDice(vein)=0.3018
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5060
  [train] batch 50/560  loss=1.4636
  [train] batch 100/560  loss=1.4722
  [train] batch 150/560  loss=1.5972
  [train] batch 200/560  loss=1.5770
  [train] batch 250/560  loss=1.6052
  [train] batch 300/560  loss=1.7144
  [train] batch 350/560  loss=1.6221
  [train] batch 400/560  loss=1.5853
  [train] batch 450/560  loss=1.5760
  [train] batch 500/560  loss=1.6474
  [train] batch 550/560  loss=1.7189
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][37/60] loss=1.5798 valDice(A∪V)=0.5926 valClDice(A∪V)=0.6068 valDice(art)=0.3771 valClDice(art)=0.2841 valDice(vein)=0.3033 valClDice(vein)=0.3292
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.4665
  [train] batch 50/560  loss=1.7061
  [train] batch 100/560  loss=1.5733
  [train] batch 150/560  loss=1.6027
  [train] batch 200/560  loss=1.5437
  [train] batch 250/560  loss=1.5788
  [train] batch 300/560  loss=1.5489
  [train] batch 350/560  loss=1.7115
  [train] batch 400/560  loss=1.4801
  [train] batch 450/560  loss=1.5124
  [train] batch 500/560  loss=1.5923
  [train] batch 550/560  loss=1.5293
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][38/60] loss=1.5773 valDice(A∪V)=0.6001 valClDice(A∪V)=0.6157 valDice(art)=0.3940 valClDice(art)=0.3383 valDice(vein)=0.2874 valClDice(vein)=0.2860
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5379
  [train] batch 50/560  loss=1.6229
  [train] batch 100/560  loss=1.6624
  [train] batch 150/560  loss=1.5867
  [train] batch 200/560  loss=1.5174
  [train] batch 250/560  loss=1.5266
  [train] batch 300/560  loss=1.5163
  [train] batch 350/560  loss=1.6310
  [train] batch 400/560  loss=2.3110
  [train] batch 450/560  loss=1.5916
  [train] batch 500/560  loss=1.4870
  [train] batch 550/560  loss=1.5518
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][39/60] loss=1.5761 valDice(A∪V)=0.5964 valClDice(A∪V)=0.6117 valDice(art)=0.3798 valClDice(art)=0.3214 valDice(vein)=0.2992 valClDice(vein)=0.3090
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.4344
  [train] batch 50/560  loss=1.5160
  [train] batch 100/560  loss=1.6000
  [train] batch 150/560  loss=1.6119
  [train] batch 200/560  loss=1.6557
  [train] batch 250/560  loss=1.5546
  [train] batch 300/560  loss=1.5857
  [train] batch 350/560  loss=1.6549
  [train] batch 400/560  loss=1.6354
  [train] batch 450/560  loss=1.5666
  [train] batch 500/560  loss=1.6191
  [train] batch 550/560  loss=1.5078
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][40/60] loss=1.5735 valDice(A∪V)=0.5976 valClDice(A∪V)=0.6128 valDice(art)=0.3882 valClDice(art)=0.3238 valDice(vein)=0.2937 valClDice(vein)=0.3016
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5689
  [train] batch 50/560  loss=1.5651
  [train] batch 100/560  loss=1.5510
  [train] batch 150/560  loss=1.4545
  [train] batch 200/560  loss=1.5541
  [train] batch 250/560  loss=1.5900
  [train] batch 300/560  loss=1.5183
  [train] batch 350/560  loss=1.6765
  [train] batch 400/560  loss=1.5680
  [train] batch 450/560  loss=1.5462
  [train] batch 500/560  loss=1.5242
  [train] batch 550/560  loss=1.7192
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][41/60] loss=1.5732 valDice(A∪V)=0.5982 valClDice(A∪V)=0.6123 valDice(art)=0.3782 valClDice(art)=0.2973 valDice(vein)=0.3059 valClDice(vein)=0.3258
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5846
  [train] batch 50/560  loss=1.5385
  [train] batch 100/560  loss=1.4622
  [train] batch 150/560  loss=1.6381
  [train] batch 200/560  loss=1.6359
  [train] batch 250/560  loss=1.5896
  [train] batch 300/560  loss=1.5424
  [train] batch 350/560  loss=1.6320
  [train] batch 400/560  loss=1.5451
  [train] batch 450/560  loss=1.5895
  [train] batch 500/560  loss=1.5863
  [train] batch 550/560  loss=1.5316
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][42/60] loss=1.5746 valDice(A∪V)=0.6012 valClDice(A∪V)=0.6159 valDice(art)=0.3754 valClDice(art)=0.2851 valDice(vein)=0.3114 valClDice(vein)=0.3369
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5349
  [train] batch 50/560  loss=1.5562
  [train] batch 100/560  loss=1.5714
  [train] batch 150/560  loss=1.6781
  [train] batch 200/560  loss=1.4451
  [train] batch 250/560  loss=1.5430
  [train] batch 300/560  loss=1.6219
  [train] batch 350/560  loss=1.4637
  [train] batch 400/560  loss=1.6375
  [train] batch 450/560  loss=1.5657
  [train] batch 500/560  loss=1.5377
  [train] batch 550/560  loss=1.5593
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][43/60] loss=1.5733 valDice(A∪V)=0.5973 valClDice(A∪V)=0.6111 valDice(art)=0.3723 valClDice(art)=0.2938 valDice(vein)=0.3088 valClDice(vein)=0.3286
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5469
  [train] batch 50/560  loss=1.5200
  [train] batch 100/560  loss=1.4907
  [train] batch 150/560  loss=1.5582
  [train] batch 200/560  loss=1.5555
  [train] batch 250/560  loss=1.6112
  [train] batch 300/560  loss=1.5757
  [train] batch 350/560  loss=1.5846
  [train] batch 400/560  loss=1.5072
  [train] batch 450/560  loss=1.5528
  [train] batch 500/560  loss=1.4649
  [train] batch 550/560  loss=1.5223
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][44/60] loss=1.5747 valDice(A∪V)=0.5981 valClDice(A∪V)=0.6118 valDice(art)=0.3577 valClDice(art)=0.2565 valDice(vein)=0.3210 valClDice(vein)=0.3546
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5738
  [train] batch 50/560  loss=1.5797
  [train] batch 100/560  loss=1.6102
  [train] batch 150/560  loss=1.5126
  [train] batch 200/560  loss=1.5249
  [train] batch 250/560  loss=1.4633
  [train] batch 300/560  loss=1.6164
  [train] batch 350/560  loss=1.5341
  [train] batch 400/560  loss=1.6137
  [train] batch 450/560  loss=1.4843
  [train] batch 500/560  loss=1.6490
  [train] batch 550/560  loss=1.5098
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][45/60] loss=1.5745 valDice(A∪V)=0.6009 valClDice(A∪V)=0.6149 valDice(art)=0.3659 valClDice(art)=0.2696 valDice(vein)=0.3183 valClDice(vein)=0.3486
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5834
  [train] batch 50/560  loss=1.5362
  [train] batch 100/560  loss=1.5869
  [train] batch 150/560  loss=1.5130
  [train] batch 200/560  loss=1.5636
  [train] batch 250/560  loss=1.5218
  [train] batch 300/560  loss=1.4844
  [train] batch 350/560  loss=1.6306
  [train] batch 400/560  loss=1.5317
  [train] batch 450/560  loss=1.4837
  [train] batch 500/560  loss=1.5009
  [train] batch 550/560  loss=1.4759
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][46/60] loss=1.5728 valDice(A∪V)=0.6009 valClDice(A∪V)=0.6144 valDice(art)=0.3862 valClDice(art)=0.3104 valDice(vein)=0.3018 valClDice(vein)=0.3149
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6044
  [train] batch 50/560  loss=1.5493
  [train] batch 100/560  loss=1.5682
  [train] batch 150/560  loss=1.5658
  [train] batch 200/560  loss=1.4984
  [train] batch 250/560  loss=1.6822
  [train] batch 300/560  loss=1.7132
  [train] batch 350/560  loss=1.7287
  [train] batch 400/560  loss=1.5522
  [train] batch 450/560  loss=1.5107
  [train] batch 500/560  loss=1.5814
  [train] batch 550/560  loss=1.5823
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][47/60] loss=1.5784 valDice(A∪V)=0.6004 valClDice(A∪V)=0.6142 valDice(art)=0.3861 valClDice(art)=0.3177 valDice(vein)=0.3010 valClDice(vein)=0.3119
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5551
  [train] batch 50/560  loss=1.6238
  [train] batch 100/560  loss=1.5750
  [train] batch 150/560  loss=1.4686
  [train] batch 200/560  loss=1.5189
  [train] batch 250/560  loss=1.5404
  [train] batch 300/560  loss=1.5528
  [train] batch 350/560  loss=1.4608
  [train] batch 400/560  loss=1.5542
  [train] batch 450/560  loss=1.5323
  [train] batch 500/560  loss=1.6741
  [train] batch 550/560  loss=1.4958
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][48/60] loss=1.5778 valDice(A∪V)=0.6029 valClDice(A∪V)=0.6163 valDice(art)=0.3978 valClDice(art)=0.3453 valDice(vein)=0.2867 valClDice(vein)=0.2782
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5035
  [train] batch 50/560  loss=1.5647
  [train] batch 100/560  loss=1.5290
  [train] batch 150/560  loss=1.4726
  [train] batch 200/560  loss=1.5159
  [train] batch 250/560  loss=1.6211
  [train] batch 300/560  loss=1.4979
  [train] batch 350/560  loss=1.5871
  [train] batch 400/560  loss=1.5806
  [train] batch 450/560  loss=1.5107
  [train] batch 500/560  loss=1.5226
  [train] batch 550/560  loss=1.5968
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][49/60] loss=1.5716 valDice(A∪V)=0.6030 valClDice(A∪V)=0.6159 valDice(art)=0.3803 valClDice(art)=0.2933 valDice(vein)=0.3092 valClDice(vein)=0.3299
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5439
  [train] batch 50/560  loss=1.5157
  [train] batch 100/560  loss=1.5434
  [train] batch 150/560  loss=1.5505
  [train] batch 200/560  loss=1.5373
  [train] batch 250/560  loss=1.6315
  [train] batch 300/560  loss=1.5220
  [train] batch 350/560  loss=1.5637
  [train] batch 400/560  loss=1.5440
  [train] batch 450/560  loss=1.5915
  [train] batch 500/560  loss=1.5089
  [train] batch 550/560  loss=1.5859
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][50/60] loss=1.5700 valDice(A∪V)=0.6015 valClDice(A∪V)=0.6146 valDice(art)=0.3964 valClDice(art)=0.3415 valDice(vein)=0.2877 valClDice(vein)=0.2820
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6050
  [train] batch 50/560  loss=1.5112
  [train] batch 100/560  loss=1.5328
  [train] batch 150/560  loss=1.5160
  [train] batch 200/560  loss=1.5369
  [train] batch 250/560  loss=1.5096
  [train] batch 300/560  loss=1.5894
  [train] batch 350/560  loss=1.5965
  [train] batch 400/560  loss=1.5239
  [train] batch 450/560  loss=1.5590
  [train] batch 500/560  loss=1.5836
  [train] batch 550/560  loss=1.5829
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][51/60] loss=1.5714 valDice(A∪V)=0.6009 valClDice(A∪V)=0.6129 valDice(art)=0.3686 valClDice(art)=0.2668 valDice(vein)=0.3172 valClDice(vein)=0.3484
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5684
  [train] batch 50/560  loss=1.6106
  [train] batch 100/560  loss=1.6054
  [train] batch 150/560  loss=1.5850
  [train] batch 200/560  loss=1.5713
  [train] batch 250/560  loss=1.5560
  [train] batch 300/560  loss=1.5928
  [train] batch 350/560  loss=1.5562
  [train] batch 400/560  loss=1.5117
  [train] batch 450/560  loss=1.4751
  [train] batch 500/560  loss=1.5921
  [train] batch 550/560  loss=1.5802
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][52/60] loss=1.5755 valDice(A∪V)=0.6038 valClDice(A∪V)=0.6158 valDice(art)=0.3938 valClDice(art)=0.3442 valDice(vein)=0.2935 valClDice(vein)=0.2899
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5600
  [train] batch 50/560  loss=1.5620
  [train] batch 100/560  loss=1.6141
  [train] batch 150/560  loss=1.5539
  [train] batch 200/560  loss=1.5732
  [train] batch 250/560  loss=1.5243
  [train] batch 300/560  loss=1.5909
  [train] batch 350/560  loss=1.5999
  [train] batch 400/560  loss=1.5693
  [train] batch 450/560  loss=1.5523
  [train] batch 500/560  loss=1.5678
  [train] batch 550/560  loss=1.6985
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][53/60] loss=1.5743 valDice(A∪V)=0.5997 valClDice(A∪V)=0.6119 valDice(art)=0.3880 valClDice(art)=0.3163 valDice(vein)=0.2990 valClDice(vein)=0.3094
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6113
  [train] batch 50/560  loss=1.6197
  [train] batch 100/560  loss=1.5015
  [train] batch 150/560  loss=1.6351
  [train] batch 200/560  loss=1.6554
  [train] batch 250/560  loss=1.6886
  [train] batch 300/560  loss=1.5670
  [train] batch 350/560  loss=1.5851
  [train] batch 400/560  loss=1.6253
  [train] batch 450/560  loss=1.4431
  [train] batch 500/560  loss=1.5498
  [train] batch 550/560  loss=1.5313
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][54/60] loss=1.5727 valDice(A∪V)=0.6036 valClDice(A∪V)=0.6158 valDice(art)=0.3931 valClDice(art)=0.3243 valDice(vein)=0.2969 valClDice(vein)=0.3034
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5352
  [train] batch 50/560  loss=1.5941
  [train] batch 100/560  loss=1.5984
  [train] batch 150/560  loss=1.8025
  [train] batch 200/560  loss=1.6305
  [train] batch 250/560  loss=1.7170
  [train] batch 300/560  loss=1.7296
  [train] batch 350/560  loss=1.5055
  [train] batch 400/560  loss=1.6440
  [train] batch 450/560  loss=1.5903
  [train] batch 500/560  loss=1.5018
  [train] batch 550/560  loss=1.5827
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][55/60] loss=1.5738 valDice(A∪V)=0.5996 valClDice(A∪V)=0.6112 valDice(art)=0.3824 valClDice(art)=0.3048 valDice(vein)=0.3038 valClDice(vein)=0.3176
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5943
  [train] batch 50/560  loss=1.6398
  [train] batch 100/560  loss=1.5289
  [train] batch 150/560  loss=1.5935
  [train] batch 200/560  loss=1.6251
  [train] batch 250/560  loss=1.4903
  [train] batch 300/560  loss=2.1965
  [train] batch 350/560  loss=1.4245
  [train] batch 400/560  loss=1.6212
  [train] batch 450/560  loss=1.5722
  [train] batch 500/560  loss=1.6086
  [train] batch 550/560  loss=1.5009
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][56/60] loss=1.5727 valDice(A∪V)=0.6045 valClDice(A∪V)=0.6154 valDice(art)=0.3894 valClDice(art)=0.3240 valDice(vein)=0.3013 valClDice(vein)=0.3077
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.8012
  [train] batch 50/560  loss=1.4917
  [train] batch 100/560  loss=1.5877
  [train] batch 150/560  loss=1.5329
  [train] batch 200/560  loss=1.6809
  [train] batch 250/560  loss=1.6159
  [train] batch 300/560  loss=1.6752
  [train] batch 350/560  loss=1.5456
  [train] batch 400/560  loss=1.4739
  [train] batch 450/560  loss=1.5768
  [train] batch 500/560  loss=1.5607
  [train] batch 550/560  loss=1.5279
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][57/60] loss=1.5726 valDice(A∪V)=0.6014 valClDice(A∪V)=0.6119 valDice(art)=0.3874 valClDice(art)=0.3198 valDice(vein)=0.3015 valClDice(vein)=0.3109
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.4535
  [train] batch 50/560  loss=1.5012
  [train] batch 100/560  loss=1.6371
  [train] batch 150/560  loss=1.6389
  [train] batch 200/560  loss=1.5406
  [train] batch 250/560  loss=1.5462
  [train] batch 300/560  loss=1.5625
  [train] batch 350/560  loss=1.4796
  [train] batch 400/560  loss=1.4876
  [train] batch 450/560  loss=2.3451
  [train] batch 500/560  loss=1.4867
  [train] batch 550/560  loss=1.6582
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][58/60] loss=1.5738 valDice(A∪V)=0.6000 valClDice(A∪V)=0.6104 valDice(art)=0.3750 valClDice(art)=0.2968 valDice(vein)=0.3104 valClDice(vein)=0.3285
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.4396
  [train] batch 50/560  loss=1.5708
  [train] batch 100/560  loss=1.6755
  [train] batch 150/560  loss=1.6230
  [train] batch 200/560  loss=1.6086
  [train] batch 250/560  loss=1.5669
  [train] batch 300/560  loss=1.6754
  [train] batch 350/560  loss=1.5146
  [train] batch 400/560  loss=1.5069
  [train] batch 450/560  loss=1.5037
  [train] batch 500/560  loss=1.5739
  [train] batch 550/560  loss=1.5781
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][59/60] loss=1.5745 valDice(A∪V)=0.5975 valClDice(A∪V)=0.6082 valDice(art)=0.3717 valClDice(art)=0.2783 valDice(vein)=0.3133 valClDice(vein)=0.3411
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6175
  [train] batch 50/560  loss=1.6656
  [train] batch 100/560  loss=1.5569
  [train] batch 150/560  loss=1.5371
  [train] batch 200/560  loss=1.5255
  [train] batch 250/560  loss=1.6653
  [train] batch 300/560  loss=1.5178
  [train] batch 350/560  loss=1.6076
  [train] batch 400/560  loss=1.5675
  [train] batch 450/560  loss=1.6857
  [train] batch 500/560  loss=1.6775
  [train] batch 550/560  loss=1.5530
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S1][60/60] loss=1.5770 valDice(A∪V)=0.5980 valClDice(A∪V)=0.6087 valDice(art)=0.3293 valClDice(art)=0.2284 valDice(vein)=0.3330 valClDice(vein)=0.3676
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6803
  [train] batch 50/560  loss=1.8238
  [train] batch 100/560  loss=1.6391
  [train] batch 150/560  loss=1.5793
  [train] batch 200/560  loss=1.6282
  [train] batch 250/560  loss=1.7085
  [train] batch 300/560  loss=1.7081
  [train] batch 350/560  loss=1.6574
  [train] batch 400/560  loss=1.5507
  [train] batch 450/560  loss=1.6540
  [train] batch 500/560  loss=1.5704
  [train] batch 550/560  loss=1.5981
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][1/50] loss=1.6302 valDice(A∪V)=0.6013 valClDice(A∪V)=0.6117 valDice(art)=0.3835 valClDice(art)=0.3236 valDice(vein)=0.3040 valClDice(vein)=0.3107
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5799
  [train] batch 50/560  loss=1.6240
  [train] batch 100/560  loss=1.5939
  [train] batch 150/560  loss=1.5221
  [train] batch 200/560  loss=1.7157
  [train] batch 250/560  loss=1.5787
  [train] batch 300/560  loss=1.5951
  [train] batch 350/560  loss=1.5375
  [train] batch 400/560  loss=1.4944
  [train] batch 450/560  loss=1.6238
  [train] batch 500/560  loss=1.6515
  [train] batch 550/560  loss=1.5046
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][2/50] loss=1.6286 valDice(A∪V)=0.6007 valClDice(A∪V)=0.6111 valDice(art)=0.3752 valClDice(art)=0.3049 valDice(vein)=0.3108 valClDice(vein)=0.3255
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5431
  [train] batch 50/560  loss=2.2505
  [train] batch 100/560  loss=1.6213
  [train] batch 150/560  loss=1.5443
  [train] batch 200/560  loss=1.5545
  [train] batch 250/560  loss=1.5341
  [train] batch 300/560  loss=1.5852
  [train] batch 350/560  loss=1.8553
  [train] batch 400/560  loss=1.7102
  [train] batch 450/560  loss=1.6541
  [train] batch 500/560  loss=1.5745
  [train] batch 550/560  loss=1.5940
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][3/50] loss=1.6237 valDice(A∪V)=0.6010 valClDice(A∪V)=0.6114 valDice(art)=0.3830 valClDice(art)=0.3124 valDice(vein)=0.3056 valClDice(vein)=0.3175
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6168
  [train] batch 50/560  loss=1.6215
  [train] batch 100/560  loss=1.5686
  [train] batch 150/560  loss=1.5623
  [train] batch 200/560  loss=1.6789
  [train] batch 250/560  loss=1.8013
  [train] batch 300/560  loss=2.5739
  [train] batch 350/560  loss=1.7631
  [train] batch 400/560  loss=1.6255
  [train] batch 450/560  loss=1.6576
  [train] batch 500/560  loss=1.5332
  [train] batch 550/560  loss=1.5696
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][4/50] loss=1.6266 valDice(A∪V)=0.6011 valClDice(A∪V)=0.6116 valDice(art)=0.3912 valClDice(art)=0.3241 valDice(vein)=0.2986 valClDice(vein)=0.3060
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7016
  [train] batch 50/560  loss=1.5566
  [train] batch 100/560  loss=1.6621
  [train] batch 150/560  loss=1.6204
  [train] batch 200/560  loss=1.8172
  [train] batch 250/560  loss=1.5704
  [train] batch 300/560  loss=1.6519
  [train] batch 350/560  loss=1.6982
  [train] batch 400/560  loss=1.5412
  [train] batch 450/560  loss=1.5752
  [train] batch 500/560  loss=1.8005
  [train] batch 550/560  loss=1.6560
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][5/50] loss=1.6264 valDice(A∪V)=0.6017 valClDice(A∪V)=0.6122 valDice(art)=0.3856 valClDice(art)=0.3073 valDice(vein)=0.3049 valClDice(vein)=0.3198
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5643
  [train] batch 50/560  loss=1.7226
  [train] batch 100/560  loss=2.1549
  [train] batch 150/560  loss=1.5982
  [train] batch 200/560  loss=1.6803
  [train] batch 250/560  loss=1.6022
  [train] batch 300/560  loss=1.6630
  [train] batch 350/560  loss=1.5394
  [train] batch 400/560  loss=1.6878
  [train] batch 450/560  loss=1.6073
  [train] batch 500/560  loss=1.6308
  [train] batch 550/560  loss=1.8224
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][6/50] loss=1.6271 valDice(A∪V)=0.6008 valClDice(A∪V)=0.6112 valDice(art)=0.3743 valClDice(art)=0.2898 valDice(vein)=0.3127 valClDice(vein)=0.3346
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5377
  [train] batch 50/560  loss=1.6425
  [train] batch 100/560  loss=1.6306
  [train] batch 150/560  loss=1.7333
  [train] batch 200/560  loss=1.5035
  [train] batch 250/560  loss=1.6038
  [train] batch 300/560  loss=1.5703
  [train] batch 350/560  loss=1.5778
  [train] batch 400/560  loss=1.6333
  [train] batch 450/560  loss=1.5541
  [train] batch 500/560  loss=1.6119
  [train] batch 550/560  loss=1.5416
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][7/50] loss=1.6290 valDice(A∪V)=0.6008 valClDice(A∪V)=0.6111 valDice(art)=0.3852 valClDice(art)=0.3118 valDice(vein)=0.3036 valClDice(vein)=0.3157
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6167
  [train] batch 50/560  loss=1.6077
  [train] batch 100/560  loss=1.7475
  [train] batch 150/560  loss=1.6557
  [train] batch 200/560  loss=1.6164
  [train] batch 250/560  loss=1.5798
  [train] batch 300/560  loss=1.7205
  [train] batch 350/560  loss=1.6188
  [train] batch 400/560  loss=1.6283
  [train] batch 450/560  loss=1.6251
  [train] batch 500/560  loss=1.7044
  [train] batch 550/560  loss=1.6352
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][8/50] loss=1.6263 valDice(A∪V)=0.6010 valClDice(A∪V)=0.6112 valDice(art)=0.3897 valClDice(art)=0.3201 valDice(vein)=0.2997 valClDice(vein)=0.3075
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5165
  [train] batch 50/560  loss=1.5944
  [train] batch 100/560  loss=1.6700
  [train] batch 150/560  loss=1.6534
  [train] batch 200/560  loss=1.6926
  [train] batch 250/560  loss=1.6153
  [train] batch 300/560  loss=1.5614
  [train] batch 350/560  loss=1.6272
  [train] batch 400/560  loss=1.5536
  [train] batch 450/560  loss=1.5901
  [train] batch 500/560  loss=1.6856
  [train] batch 550/560  loss=1.6687
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][9/50] loss=1.6278 valDice(A∪V)=0.6010 valClDice(A∪V)=0.6113 valDice(art)=0.3828 valClDice(art)=0.3075 valDice(vein)=0.3060 valClDice(vein)=0.3202
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6642
  [train] batch 50/560  loss=1.6438
  [train] batch 100/560  loss=1.6570
  [train] batch 150/560  loss=1.5939
  [train] batch 200/560  loss=1.5909
  [train] batch 250/560  loss=1.5045
  [train] batch 300/560  loss=1.7269
  [train] batch 350/560  loss=1.6951
  [train] batch 400/560  loss=1.6653
  [train] batch 450/560  loss=1.5924
  [train] batch 500/560  loss=1.7472
  [train] batch 550/560  loss=1.6092
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][10/50] loss=1.6271 valDice(A∪V)=0.6022 valClDice(A∪V)=0.6125 valDice(art)=0.3884 valClDice(art)=0.3183 valDice(vein)=0.3020 valClDice(vein)=0.3113
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5827
  [train] batch 50/560  loss=1.5351
  [train] batch 100/560  loss=1.6432
  [train] batch 150/560  loss=1.6603
  [train] batch 200/560  loss=1.6662
  [train] batch 250/560  loss=1.7819
  [train] batch 300/560  loss=1.6223
  [train] batch 350/560  loss=1.6577
  [train] batch 400/560  loss=1.5551
  [train] batch 450/560  loss=1.7050
  [train] batch 500/560  loss=1.6869
  [train] batch 550/560  loss=1.5962
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][11/50] loss=1.6287 valDice(A∪V)=0.6014 valClDice(A∪V)=0.6117 valDice(art)=0.3764 valClDice(art)=0.3024 valDice(vein)=0.3108 valClDice(vein)=0.3266
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5536
  [train] batch 50/560  loss=1.6143
  [train] batch 100/560  loss=1.6425
  [train] batch 150/560  loss=1.5700
  [train] batch 200/560  loss=1.6728
  [train] batch 250/560  loss=1.5917
  [train] batch 300/560  loss=1.5456
  [train] batch 350/560  loss=1.6442
  [train] batch 400/560  loss=1.6176
  [train] batch 450/560  loss=1.5718
  [train] batch 500/560  loss=1.6204
  [train] batch 550/560  loss=1.6578
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][12/50] loss=1.6282 valDice(A∪V)=0.6017 valClDice(A∪V)=0.6119 valDice(art)=0.3852 valClDice(art)=0.3129 valDice(vein)=0.3043 valClDice(vein)=0.3158
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7633
  [train] batch 50/560  loss=1.5340
  [train] batch 100/560  loss=1.6280
  [train] batch 150/560  loss=1.5643
  [train] batch 200/560  loss=1.5470
  [train] batch 250/560  loss=1.6867
  [train] batch 300/560  loss=1.6591
  [train] batch 350/560  loss=1.5950
  [train] batch 400/560  loss=1.5723
  [train] batch 450/560  loss=1.6439
  [train] batch 500/560  loss=1.5921
  [train] batch 550/560  loss=1.6443
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][13/50] loss=1.6282 valDice(A∪V)=0.6010 valClDice(A∪V)=0.6112 valDice(art)=0.3850 valClDice(art)=0.3106 valDice(vein)=0.3043 valClDice(vein)=0.3164
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5367
  [train] batch 50/560  loss=1.5524
  [train] batch 100/560  loss=1.7480
  [train] batch 150/560  loss=1.5693
  [train] batch 200/560  loss=1.5500
  [train] batch 250/560  loss=1.5590
  [train] batch 300/560  loss=1.5445
  [train] batch 350/560  loss=1.5860
  [train] batch 400/560  loss=1.5986
  [train] batch 450/560  loss=1.7025
  [train] batch 500/560  loss=1.5960
  [train] batch 550/560  loss=1.5621
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][14/50] loss=1.6272 valDice(A∪V)=0.6017 valClDice(A∪V)=0.6119 valDice(art)=0.3889 valClDice(art)=0.3118 valDice(vein)=0.3019 valClDice(vein)=0.3128
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6631
  [train] batch 50/560  loss=1.5829
  [train] batch 100/560  loss=1.7228
  [train] batch 150/560  loss=1.6288
  [train] batch 200/560  loss=1.5605
  [train] batch 250/560  loss=1.6854
  [train] batch 300/560  loss=1.7824
  [train] batch 350/560  loss=1.5431
  [train] batch 400/560  loss=1.6302
  [train] batch 450/560  loss=1.7419
  [train] batch 500/560  loss=1.6585
  [train] batch 550/560  loss=1.7046
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][15/50] loss=1.6280 valDice(A∪V)=0.6020 valClDice(A∪V)=0.6121 valDice(art)=0.3843 valClDice(art)=0.3040 valDice(vein)=0.3062 valClDice(vein)=0.3218
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6482
  [train] batch 50/560  loss=1.6256
  [train] batch 100/560  loss=1.6039
  [train] batch 150/560  loss=1.5865
  [train] batch 200/560  loss=1.5527
  [train] batch 250/560  loss=1.6376
  [train] batch 300/560  loss=1.5165
  [train] batch 350/560  loss=1.5629
  [train] batch 400/560  loss=1.5666
  [train] batch 450/560  loss=1.5427
  [train] batch 500/560  loss=1.6628
  [train] batch 550/560  loss=1.5962
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][16/50] loss=1.6281 valDice(A∪V)=0.6010 valClDice(A∪V)=0.6113 valDice(art)=0.3795 valClDice(art)=0.2963 valDice(vein)=0.3093 valClDice(vein)=0.3287
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6468
  [train] batch 50/560  loss=1.6366
  [train] batch 100/560  loss=1.6872
  [train] batch 150/560  loss=1.5518
  [train] batch 200/560  loss=1.6691
  [train] batch 250/560  loss=1.5795
  [train] batch 300/560  loss=1.5755
  [train] batch 350/560  loss=1.5862
  [train] batch 400/560  loss=1.6082
  [train] batch 450/560  loss=1.5644
  [train] batch 500/560  loss=1.6100
  [train] batch 550/560  loss=1.7535
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][17/50] loss=1.6262 valDice(A∪V)=0.6006 valClDice(A∪V)=0.6109 valDice(art)=0.3828 valClDice(art)=0.3068 valDice(vein)=0.3060 valClDice(vein)=0.3212
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6542
  [train] batch 50/560  loss=1.6282
  [train] batch 100/560  loss=1.6354
  [train] batch 150/560  loss=1.5531
  [train] batch 200/560  loss=1.7083
  [train] batch 250/560  loss=1.5801
  [train] batch 300/560  loss=1.6337
  [train] batch 350/560  loss=1.6622
  [train] batch 400/560  loss=1.5780
  [train] batch 450/560  loss=1.6770
  [train] batch 500/560  loss=1.5498
  [train] batch 550/560  loss=1.6377
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][18/50] loss=1.6245 valDice(A∪V)=0.6004 valClDice(A∪V)=0.6108 valDice(art)=0.3852 valClDice(art)=0.3125 valDice(vein)=0.3037 valClDice(vein)=0.3160
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6050
  [train] batch 50/560  loss=1.6186
  [train] batch 100/560  loss=1.5532
  [train] batch 150/560  loss=1.6418
  [train] batch 200/560  loss=1.5895
  [train] batch 250/560  loss=1.5486
  [train] batch 300/560  loss=1.5627
  [train] batch 350/560  loss=1.6677
  [train] batch 400/560  loss=1.5888
  [train] batch 450/560  loss=1.6318
  [train] batch 500/560  loss=1.5913
  [train] batch 550/560  loss=1.6931
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][19/50] loss=1.6256 valDice(A∪V)=0.5998 valClDice(A∪V)=0.6101 valDice(art)=0.3794 valClDice(art)=0.3044 valDice(vein)=0.3079 valClDice(vein)=0.3237
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5641
  [train] batch 50/560  loss=1.6408
  [train] batch 100/560  loss=1.6557
  [train] batch 150/560  loss=1.5396
  [train] batch 200/560  loss=1.7137
  [train] batch 250/560  loss=1.6426
  [train] batch 300/560  loss=1.6875
  [train] batch 350/560  loss=1.5789
  [train] batch 400/560  loss=1.6799
  [train] batch 450/560  loss=1.5287
  [train] batch 500/560  loss=2.5992
  [train] batch 550/560  loss=1.6197
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][20/50] loss=1.6323 valDice(A∪V)=0.6005 valClDice(A∪V)=0.6108 valDice(art)=0.3863 valClDice(art)=0.3155 valDice(vein)=0.3028 valClDice(vein)=0.3142
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7396
  [train] batch 50/560  loss=1.6179
  [train] batch 100/560  loss=1.6342
  [train] batch 150/560  loss=1.6088
  [train] batch 200/560  loss=1.5494
  [train] batch 250/560  loss=1.7084
  [train] batch 300/560  loss=1.6190
  [train] batch 350/560  loss=1.5497
  [train] batch 400/560  loss=1.5886
  [train] batch 450/560  loss=2.1990
  [train] batch 500/560  loss=1.5880
  [train] batch 550/560  loss=1.5529
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][21/50] loss=1.6265 valDice(A∪V)=0.6000 valClDice(A∪V)=0.6104 valDice(art)=0.3773 valClDice(art)=0.3002 valDice(vein)=0.3098 valClDice(vein)=0.3266
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6308
  [train] batch 50/560  loss=1.5950
  [train] batch 100/560  loss=1.5524
  [train] batch 150/560  loss=1.6751
  [train] batch 200/560  loss=1.6438
  [train] batch 250/560  loss=1.7184
  [train] batch 300/560  loss=1.4967
  [train] batch 350/560  loss=1.5789
  [train] batch 400/560  loss=1.5749
  [train] batch 450/560  loss=1.6604
  [train] batch 500/560  loss=1.5719
  [train] batch 550/560  loss=1.5802
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][22/50] loss=1.6225 valDice(A∪V)=0.6010 valClDice(A∪V)=0.6112 valDice(art)=0.3861 valClDice(art)=0.3131 valDice(vein)=0.3035 valClDice(vein)=0.3154
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6126
  [train] batch 50/560  loss=1.6385
  [train] batch 100/560  loss=1.6651
  [train] batch 150/560  loss=1.6247
  [train] batch 200/560  loss=1.5551
  [train] batch 250/560  loss=1.6547
  [train] batch 300/560  loss=1.5927
  [train] batch 350/560  loss=1.5791
  [train] batch 400/560  loss=1.6720
  [train] batch 450/560  loss=1.5586
  [train] batch 500/560  loss=1.6331
  [train] batch 550/560  loss=1.6211
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][23/50] loss=1.6307 valDice(A∪V)=0.6007 valClDice(A∪V)=0.6109 valDice(art)=0.3793 valClDice(art)=0.3022 valDice(vein)=0.3088 valClDice(vein)=0.3256
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6662
  [train] batch 50/560  loss=1.5638
  [train] batch 100/560  loss=1.6936
  [train] batch 150/560  loss=1.6825
  [train] batch 200/560  loss=1.6499
  [train] batch 250/560  loss=1.5521
  [train] batch 300/560  loss=1.5358
  [train] batch 350/560  loss=1.6408
  [train] batch 400/560  loss=1.6467
  [train] batch 450/560  loss=1.5286
  [train] batch 500/560  loss=1.6550
  [train] batch 550/560  loss=1.5397
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][24/50] loss=1.6284 valDice(A∪V)=0.6015 valClDice(A∪V)=0.6118 valDice(art)=0.3815 valClDice(art)=0.3053 valDice(vein)=0.3077 valClDice(vein)=0.3233
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5817
  [train] batch 50/560  loss=1.6154
  [train] batch 100/560  loss=1.5369
  [train] batch 150/560  loss=1.5292
  [train] batch 200/560  loss=1.5944
  [train] batch 250/560  loss=1.5374
  [train] batch 300/560  loss=1.5315
  [train] batch 350/560  loss=1.7194
  [train] batch 400/560  loss=1.5312
  [train] batch 450/560  loss=1.6062
  [train] batch 500/560  loss=1.5779
  [train] batch 550/560  loss=1.5981
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][25/50] loss=1.6302 valDice(A∪V)=0.6015 valClDice(A∪V)=0.6117 valDice(art)=0.3865 valClDice(art)=0.3121 valDice(vein)=0.3037 valClDice(vein)=0.3159
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7513
  [train] batch 50/560  loss=1.6591
  [train] batch 100/560  loss=1.6503
  [train] batch 150/560  loss=1.6425
  [train] batch 200/560  loss=1.6342
  [train] batch 250/560  loss=1.5962
  [train] batch 300/560  loss=1.5617
  [train] batch 350/560  loss=1.5312
  [train] batch 400/560  loss=1.6069
  [train] batch 450/560  loss=1.5694
  [train] batch 500/560  loss=1.6493
  [train] batch 550/560  loss=1.7204
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][26/50] loss=1.6302 valDice(A∪V)=0.6014 valClDice(A∪V)=0.6117 valDice(art)=0.3837 valClDice(art)=0.3089 valDice(vein)=0.3060 valClDice(vein)=0.3197
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6473
  [train] batch 50/560  loss=1.5750
  [train] batch 100/560  loss=1.7143
  [train] batch 150/560  loss=1.5406
  [train] batch 200/560  loss=1.7232
  [train] batch 250/560  loss=1.5655
  [train] batch 300/560  loss=1.5810
  [train] batch 350/560  loss=1.5974
  [train] batch 400/560  loss=1.6397
  [train] batch 450/560  loss=1.6806
  [train] batch 500/560  loss=1.5738
  [train] batch 550/560  loss=1.6531
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][27/50] loss=1.6243 valDice(A∪V)=0.6015 valClDice(A∪V)=0.6118 valDice(art)=0.3848 valClDice(art)=0.3083 valDice(vein)=0.3052 valClDice(vein)=0.3190
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6298
  [train] batch 50/560  loss=1.5806
  [train] batch 100/560  loss=1.6395
  [train] batch 150/560  loss=1.6234
  [train] batch 200/560  loss=1.6413
  [train] batch 250/560  loss=1.6114
  [train] batch 300/560  loss=1.5766
  [train] batch 350/560  loss=1.6251
  [train] batch 400/560  loss=1.6012
  [train] batch 450/560  loss=1.5794
  [train] batch 500/560  loss=1.5401
  [train] batch 550/560  loss=1.6419
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][28/50] loss=1.6244 valDice(A∪V)=0.6013 valClDice(A∪V)=0.6115 valDice(art)=0.3902 valClDice(art)=0.3192 valDice(vein)=0.3001 valClDice(vein)=0.3086
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5527
  [train] batch 50/560  loss=1.5539
  [train] batch 100/560  loss=1.5186
  [train] batch 150/560  loss=1.5826
  [train] batch 200/560  loss=1.6401
  [train] batch 250/560  loss=1.5547
  [train] batch 300/560  loss=1.6147
  [train] batch 350/560  loss=1.5564
  [train] batch 400/560  loss=1.6334
  [train] batch 450/560  loss=1.7213
  [train] batch 500/560  loss=1.5466
  [train] batch 550/560  loss=1.5595
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][29/50] loss=1.6298 valDice(A∪V)=0.6016 valClDice(A∪V)=0.6117 valDice(art)=0.3872 valClDice(art)=0.3128 valDice(vein)=0.3032 valClDice(vein)=0.3151
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6771
  [train] batch 50/560  loss=1.6105
  [train] batch 100/560  loss=1.5516
  [train] batch 150/560  loss=1.5229
  [train] batch 200/560  loss=1.6537
  [train] batch 250/560  loss=1.5833
  [train] batch 300/560  loss=1.7450
  [train] batch 350/560  loss=1.5650
  [train] batch 400/560  loss=1.6855
  [train] batch 450/560  loss=1.5995
  [train] batch 500/560  loss=1.6578
  [train] batch 550/560  loss=1.7969
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][30/50] loss=1.6275 valDice(A∪V)=0.6005 valClDice(A∪V)=0.6109 valDice(art)=0.3751 valClDice(art)=0.2934 valDice(vein)=0.3120 valClDice(vein)=0.3325
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5389
  [train] batch 50/560  loss=1.6653
  [train] batch 100/560  loss=1.5271
  [train] batch 150/560  loss=1.5840
  [train] batch 200/560  loss=1.6258
  [train] batch 250/560  loss=1.5620
  [train] batch 300/560  loss=1.5868
  [train] batch 350/560  loss=1.7145
  [train] batch 400/560  loss=1.7620
  [train] batch 450/560  loss=1.6197
  [train] batch 500/560  loss=1.6931
  [train] batch 550/560  loss=1.8687
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][31/50] loss=1.6236 valDice(A∪V)=0.6009 valClDice(A∪V)=0.6110 valDice(art)=0.3869 valClDice(art)=0.3175 valDice(vein)=0.3025 valClDice(vein)=0.3125
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6481
  [train] batch 50/560  loss=1.5095
  [train] batch 100/560  loss=1.6090
  [train] batch 150/560  loss=1.6676
  [train] batch 200/560  loss=1.5795
  [train] batch 250/560  loss=1.5985
  [train] batch 300/560  loss=1.6386
  [train] batch 350/560  loss=1.5762
  [train] batch 400/560  loss=1.7038
  [train] batch 450/560  loss=1.5863
  [train] batch 500/560  loss=1.6196
  [train] batch 550/560  loss=1.6112
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][32/50] loss=1.6287 valDice(A∪V)=0.6009 valClDice(A∪V)=0.6111 valDice(art)=0.3838 valClDice(art)=0.3105 valDice(vein)=0.3052 valClDice(vein)=0.3178
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6631
  [train] batch 50/560  loss=1.5996
  [train] batch 100/560  loss=1.6418
  [train] batch 150/560  loss=1.6483
  [train] batch 200/560  loss=1.6973
  [train] batch 250/560  loss=1.6272
  [train] batch 300/560  loss=1.6094
  [train] batch 350/560  loss=1.5977
  [train] batch 400/560  loss=1.5963
  [train] batch 450/560  loss=1.5679
  [train] batch 500/560  loss=1.5857
  [train] batch 550/560  loss=1.6484
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][33/50] loss=1.6273 valDice(A∪V)=0.6007 valClDice(A∪V)=0.6109 valDice(art)=0.3849 valClDice(art)=0.3095 valDice(vein)=0.3046 valClDice(vein)=0.3178
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6267
  [train] batch 50/560  loss=1.4920
  [train] batch 100/560  loss=1.5685
  [train] batch 150/560  loss=1.6027
  [train] batch 200/560  loss=1.6169
  [train] batch 250/560  loss=1.5907
  [train] batch 300/560  loss=1.5237
  [train] batch 350/560  loss=1.6292
  [train] batch 400/560  loss=1.6146
  [train] batch 450/560  loss=1.6725
  [train] batch 500/560  loss=1.5471
  [train] batch 550/560  loss=1.6472
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][34/50] loss=1.6288 valDice(A∪V)=0.6006 valClDice(A∪V)=0.6108 valDice(art)=0.3824 valClDice(art)=0.3065 valDice(vein)=0.3064 valClDice(vein)=0.3214
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6305
  [train] batch 50/560  loss=2.1798
  [train] batch 100/560  loss=1.5203
  [train] batch 150/560  loss=1.6700
  [train] batch 200/560  loss=1.5808
  [train] batch 250/560  loss=1.6358
  [train] batch 300/560  loss=1.5257
  [train] batch 350/560  loss=1.7023
  [train] batch 400/560  loss=1.6061
  [train] batch 450/560  loss=1.6502
  [train] batch 500/560  loss=1.6936
  [train] batch 550/560  loss=1.6092
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][35/50] loss=1.6249 valDice(A∪V)=0.6018 valClDice(A∪V)=0.6118 valDice(art)=0.3890 valClDice(art)=0.3181 valDice(vein)=0.3016 valClDice(vein)=0.3107
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5287
  [train] batch 50/560  loss=1.6483
  [train] batch 100/560  loss=1.6833
  [train] batch 150/560  loss=1.6277
  [train] batch 200/560  loss=1.5722
  [train] batch 250/560  loss=1.5919
  [train] batch 300/560  loss=1.5865
  [train] batch 350/560  loss=1.5458
  [train] batch 400/560  loss=1.7986
  [train] batch 450/560  loss=1.6131
  [train] batch 500/560  loss=1.6433
  [train] batch 550/560  loss=2.3083
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][36/50] loss=1.6274 valDice(A∪V)=0.6024 valClDice(A∪V)=0.6124 valDice(art)=0.3869 valClDice(art)=0.3112 valDice(vein)=0.3042 valClDice(vein)=0.3160
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7317
  [train] batch 50/560  loss=1.5879
  [train] batch 100/560  loss=1.6555
  [train] batch 150/560  loss=1.4994
  [train] batch 200/560  loss=1.8792
  [train] batch 250/560  loss=1.5641
  [train] batch 300/560  loss=1.4902
  [train] batch 350/560  loss=1.6691
  [train] batch 400/560  loss=1.7072
  [train] batch 450/560  loss=1.6007
  [train] batch 500/560  loss=1.5750
  [train] batch 550/560  loss=1.6238
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][37/50] loss=1.6319 valDice(A∪V)=0.6008 valClDice(A∪V)=0.6109 valDice(art)=0.3760 valClDice(art)=0.2922 valDice(vein)=0.3120 valClDice(vein)=0.3333
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5493
  [train] batch 50/560  loss=1.6182
  [train] batch 100/560  loss=1.5891
  [train] batch 150/560  loss=1.7420
  [train] batch 200/560  loss=1.5714
  [train] batch 250/560  loss=1.8022
  [train] batch 300/560  loss=1.5747
  [train] batch 350/560  loss=1.6138
  [train] batch 400/560  loss=1.6233
  [train] batch 450/560  loss=1.5629
  [train] batch 500/560  loss=1.5843
  [train] batch 550/560  loss=1.6772
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][38/50] loss=1.6257 valDice(A∪V)=0.6011 valClDice(A∪V)=0.6110 valDice(art)=0.3790 valClDice(art)=0.2947 valDice(vein)=0.3103 valClDice(vein)=0.3301
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6658
  [train] batch 50/560  loss=1.5809
  [train] batch 100/560  loss=1.6640
  [train] batch 150/560  loss=1.6903
  [train] batch 200/560  loss=1.6040
  [train] batch 250/560  loss=1.6027
  [train] batch 300/560  loss=1.6523
  [train] batch 350/560  loss=1.5956
  [train] batch 400/560  loss=1.6776
  [train] batch 450/560  loss=1.5744
  [train] batch 500/560  loss=1.5869
  [train] batch 550/560  loss=1.5604
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][39/50] loss=1.6273 valDice(A∪V)=0.6008 valClDice(A∪V)=0.6108 valDice(art)=0.3785 valClDice(art)=0.3001 valDice(vein)=0.3097 valClDice(vein)=0.3268
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5799
  [train] batch 50/560  loss=2.2440
  [train] batch 100/560  loss=1.6869
  [train] batch 150/560  loss=1.7069
  [train] batch 200/560  loss=1.5411
  [train] batch 250/560  loss=1.5403
  [train] batch 300/560  loss=1.6126
  [train] batch 350/560  loss=1.5333
  [train] batch 400/560  loss=1.5383
  [train] batch 450/560  loss=1.6507
  [train] batch 500/560  loss=1.6368
  [train] batch 550/560  loss=1.7210
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][40/50] loss=1.6265 valDice(A∪V)=0.6005 valClDice(A∪V)=0.6106 valDice(art)=0.3807 valClDice(art)=0.3073 valDice(vein)=0.3073 valClDice(vein)=0.3217
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5483
  [train] batch 50/560  loss=1.4780
  [train] batch 100/560  loss=1.5610
  [train] batch 150/560  loss=1.5822
  [train] batch 200/560  loss=1.5584
  [train] batch 250/560  loss=1.6528
  [train] batch 300/560  loss=1.6805
  [train] batch 350/560  loss=1.5276
  [train] batch 400/560  loss=1.6221
  [train] batch 450/560  loss=1.5572
  [train] batch 500/560  loss=1.6965
  [train] batch 550/560  loss=1.7026
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][41/50] loss=1.6276 valDice(A∪V)=0.6009 valClDice(A∪V)=0.6108 valDice(art)=0.3823 valClDice(art)=0.3071 valDice(vein)=0.3067 valClDice(vein)=0.3208
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6502
  [train] batch 50/560  loss=1.5683
  [train] batch 100/560  loss=1.6109
  [train] batch 150/560  loss=1.6032
  [train] batch 200/560  loss=1.6348
  [train] batch 250/560  loss=1.6321
  [train] batch 300/560  loss=1.6299
  [train] batch 350/560  loss=1.6246
  [train] batch 400/560  loss=1.5948
  [train] batch 450/560  loss=1.5098
  [train] batch 500/560  loss=1.5933
  [train] batch 550/560  loss=2.4758
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][42/50] loss=1.6280 valDice(A∪V)=0.6006 valClDice(A∪V)=0.6105 valDice(art)=0.3781 valClDice(art)=0.3044 valDice(vein)=0.3094 valClDice(vein)=0.3241
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5953
  [train] batch 50/560  loss=1.5722
  [train] batch 100/560  loss=1.5498
  [train] batch 150/560  loss=1.6620
  [train] batch 200/560  loss=1.6449
  [train] batch 250/560  loss=1.4873
  [train] batch 300/560  loss=1.5453
  [train] batch 350/560  loss=1.5920
  [train] batch 400/560  loss=1.4948
  [train] batch 450/560  loss=1.6229
  [train] batch 500/560  loss=1.5667
  [train] batch 550/560  loss=1.6146
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][43/50] loss=1.6237 valDice(A∪V)=0.6010 valClDice(A∪V)=0.6109 valDice(art)=0.3822 valClDice(art)=0.3069 valDice(vein)=0.3069 valClDice(vein)=0.3211
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6552
  [train] batch 50/560  loss=1.5545
  [train] batch 100/560  loss=1.6433
  [train] batch 150/560  loss=1.6347
  [train] batch 200/560  loss=1.5768
  [train] batch 250/560  loss=1.6860
  [train] batch 300/560  loss=1.6173
  [train] batch 350/560  loss=1.6781
  [train] batch 400/560  loss=1.5691
  [train] batch 450/560  loss=1.6721
  [train] batch 500/560  loss=1.6163
  [train] batch 550/560  loss=1.5619
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][44/50] loss=1.6273 valDice(A∪V)=0.6007 valClDice(A∪V)=0.6107 valDice(art)=0.3799 valClDice(art)=0.3029 valDice(vein)=0.3086 valClDice(vein)=0.3248
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5679
  [train] batch 50/560  loss=1.6614
  [train] batch 100/560  loss=1.5653
  [train] batch 150/560  loss=1.6213
  [train] batch 200/560  loss=1.5899
  [train] batch 250/560  loss=1.7122
  [train] batch 300/560  loss=1.6076
  [train] batch 350/560  loss=1.6102
  [train] batch 400/560  loss=1.5092
  [train] batch 450/560  loss=1.5936
  [train] batch 500/560  loss=1.6303
  [train] batch 550/560  loss=1.6183
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][45/50] loss=1.6278 valDice(A∪V)=0.6018 valClDice(A∪V)=0.6116 valDice(art)=0.3892 valClDice(art)=0.3165 valDice(vein)=0.3018 valClDice(vein)=0.3118
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6056
  [train] batch 50/560  loss=2.3243
  [train] batch 100/560  loss=1.5063
  [train] batch 150/560  loss=1.5588
  [train] batch 200/560  loss=1.5624
  [train] batch 250/560  loss=1.6532
  [train] batch 300/560  loss=1.7374
  [train] batch 350/560  loss=1.6383
  [train] batch 400/560  loss=1.5775
  [train] batch 450/560  loss=1.6038
  [train] batch 500/560  loss=1.5987
  [train] batch 550/560  loss=1.6618
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][46/50] loss=1.6347 valDice(A∪V)=0.6013 valClDice(A∪V)=0.6112 valDice(art)=0.3804 valClDice(art)=0.3022 valDice(vein)=0.3090 valClDice(vein)=0.3255
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5624
  [train] batch 50/560  loss=1.7691
  [train] batch 100/560  loss=1.7083
  [train] batch 150/560  loss=1.7083
  [train] batch 200/560  loss=1.4834
  [train] batch 250/560  loss=1.5434
  [train] batch 300/560  loss=1.5713
  [train] batch 350/560  loss=1.6498
  [train] batch 400/560  loss=1.5721
  [train] batch 450/560  loss=1.6954
  [train] batch 500/560  loss=1.5321
  [train] batch 550/560  loss=1.7017
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][47/50] loss=1.6317 valDice(A∪V)=0.6011 valClDice(A∪V)=0.6108 valDice(art)=0.3859 valClDice(art)=0.3113 valDice(vein)=0.3041 valClDice(vein)=0.3162
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.7560
  [train] batch 50/560  loss=1.6225
  [train] batch 100/560  loss=1.6189
  [train] batch 150/560  loss=1.6066
  [train] batch 200/560  loss=1.5922
  [train] batch 250/560  loss=1.5537
  [train] batch 300/560  loss=1.6347
  [train] batch 350/560  loss=1.6299
  [train] batch 400/560  loss=2.2520
  [train] batch 450/560  loss=1.5663
  [train] batch 500/560  loss=1.6493
  [train] batch 550/560  loss=1.6128
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][48/50] loss=1.6262 valDice(A∪V)=0.6019 valClDice(A∪V)=0.6116 valDice(art)=0.3897 valClDice(art)=0.3192 valDice(vein)=0.3014 valClDice(vein)=0.3101
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.5939
  [train] batch 50/560  loss=1.5550
  [train] batch 100/560  loss=1.6456
  [train] batch 150/560  loss=1.6648
  [train] batch 200/560  loss=1.6519
  [train] batch 250/560  loss=1.6204
  [train] batch 300/560  loss=1.4868
  [train] batch 350/560  loss=1.6021
  [train] batch 400/560  loss=1.5769
  [train] batch 450/560  loss=1.6742
  [train] batch 500/560  loss=1.6483
  [train] batch 550/560  loss=1.5736
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][49/50] loss=1.6271 valDice(A∪V)=0.6013 valClDice(A∪V)=0.6109 valDice(art)=0.3827 valClDice(art)=0.3021 valDice(vein)=0.3076 valClDice(vein)=0.3246
DEBUG train patch shape: torch.Size([2, 1, 96, 96, 96]) torch.Size([2, 96, 96, 96])
  [train] batch 1/560  loss=1.6450
  [train] batch 50/560  loss=1.5720
  [train] batch 100/560  loss=1.5600
  [train] batch 150/560  loss=1.5726
  [train] batch 200/560  loss=1.5851
  [train] batch 250/560  loss=1.5798
  [train] batch 300/560  loss=1.6385
  [train] batch 350/560  loss=2.3487
  [train] batch 400/560  loss=1.5857
  [train] batch 450/560  loss=1.6592
  [train] batch 500/560  loss=1.5475
  [train] batch 550/560  loss=1.5643
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S2][50/50] loss=1.6245 valDice(A∪V)=0.6014 valClDice(A∪V)=0.6110 valDice(art)=0.3792 valClDice(art)=0.2977 valDice(vein)=0.3103 valClDice(vein)=0.3286

=== Stage 3: AV refine head training ===
  [train_av_refine] batch 1/560 loss=2.4875
  [train_av_refine] batch 50/560 loss=1.5199
  [train_av_refine] batch 100/560 loss=1.7856
  [train_av_refine] batch 150/560 loss=1.7226
  [train_av_refine] batch 200/560 loss=1.3295
  [train_av_refine] batch 250/560 loss=2.0367
  [train_av_refine] batch 300/560 loss=1.6205
  [train_av_refine] batch 350/560 loss=1.0845
  [train_av_refine] batch 400/560 loss=1.9545
  [train_av_refine] batch 450/560 loss=1.7681
  [train_av_refine] batch 500/560 loss=1.0272
  [train_av_refine] batch 550/560 loss=2.0152
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][1/25] loss=1.9388 valDice(A∪V)=0.6130 valClDice(A∪V)=0.6197 valDice(art)=0.3608 valClDice(art)=0.3565 valDice(vein)=0.0044 valClDice(vein)=0.0003
  [train_av_refine] batch 1/560 loss=1.7930
  [train_av_refine] batch 50/560 loss=2.0974
  [train_av_refine] batch 100/560 loss=1.7253
  [train_av_refine] batch 150/560 loss=1.5683
  [train_av_refine] batch 200/560 loss=1.0660
  [train_av_refine] batch 250/560 loss=1.8744
  [train_av_refine] batch 300/560 loss=0.9691
  [train_av_refine] batch 350/560 loss=1.4976
  [train_av_refine] batch 400/560 loss=1.6978
  [train_av_refine] batch 450/560 loss=1.2742
  [train_av_refine] batch 500/560 loss=0.8212
  [train_av_refine] batch 550/560 loss=1.5431
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][2/25] loss=1.4414 valDice(A∪V)=0.5945 valClDice(A∪V)=0.6055 valDice(art)=0.4208 valClDice(art)=0.4001 valDice(vein)=0.0737 valClDice(vein)=0.0979
  [train_av_refine] batch 1/560 loss=0.8992
  [train_av_refine] batch 50/560 loss=1.2139
  [train_av_refine] batch 100/560 loss=0.7836
  [train_av_refine] batch 150/560 loss=1.1807
  [train_av_refine] batch 200/560 loss=1.3152
  [train_av_refine] batch 250/560 loss=1.2577
  [train_av_refine] batch 300/560 loss=0.8439
  [train_av_refine] batch 350/560 loss=1.3541
  [train_av_refine] batch 400/560 loss=1.3047
  [train_av_refine] batch 450/560 loss=1.0289
  [train_av_refine] batch 500/560 loss=0.8797
  [train_av_refine] batch 550/560 loss=1.1015
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][3/25] loss=1.0845 valDice(A∪V)=0.5826 valClDice(A∪V)=0.5946 valDice(art)=0.4467 valClDice(art)=0.3793 valDice(vein)=0.1809 valClDice(vein)=0.2377
  [train_av_refine] batch 1/560 loss=0.9888
  [train_av_refine] batch 50/560 loss=0.8897
  [train_av_refine] batch 100/560 loss=0.8298
  [train_av_refine] batch 150/560 loss=0.8167
  [train_av_refine] batch 200/560 loss=0.7378
  [train_av_refine] batch 250/560 loss=0.7663
  [train_av_refine] batch 300/560 loss=0.8009
  [train_av_refine] batch 350/560 loss=0.6634
  [train_av_refine] batch 400/560 loss=0.6713
  [train_av_refine] batch 450/560 loss=1.1615
  [train_av_refine] batch 500/560 loss=0.8810
  [train_av_refine] batch 550/560 loss=0.9740
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][4/25] loss=0.8947 valDice(A∪V)=0.5783 valClDice(A∪V)=0.5909 valDice(art)=0.4362 valClDice(art)=0.2731 valDice(vein)=0.2476 valClDice(vein)=0.3170
  [train_av_refine] batch 1/560 loss=0.9640
  [train_av_refine] batch 50/560 loss=0.8439
  [train_av_refine] batch 100/560 loss=0.6987
  [train_av_refine] batch 150/560 loss=0.8424
  [train_av_refine] batch 200/560 loss=0.7356
  [train_av_refine] batch 250/560 loss=0.7917
  [train_av_refine] batch 300/560 loss=0.9732
  [train_av_refine] batch 350/560 loss=0.8039
  [train_av_refine] batch 400/560 loss=0.8563
  [train_av_refine] batch 450/560 loss=0.7941
  [train_av_refine] batch 500/560 loss=0.8184
  [train_av_refine] batch 550/560 loss=0.8202
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][5/25] loss=0.8395 valDice(A∪V)=0.5778 valClDice(A∪V)=0.5905 valDice(art)=0.4291 valClDice(art)=0.2427 valDice(vein)=0.2698 valClDice(vein)=0.3352
  [train_av_refine] batch 1/560 loss=0.7556
  [train_av_refine] batch 50/560 loss=0.7302
  [train_av_refine] batch 100/560 loss=0.8634
  [train_av_refine] batch 150/560 loss=0.7377
  [train_av_refine] batch 200/560 loss=0.8001
  [train_av_refine] batch 250/560 loss=0.6656
  [train_av_refine] batch 300/560 loss=0.6445
  [train_av_refine] batch 350/560 loss=0.8728
  [train_av_refine] batch 400/560 loss=0.9166
  [train_av_refine] batch 450/560 loss=0.8653
  [train_av_refine] batch 500/560 loss=0.7683
  [train_av_refine] batch 550/560 loss=0.8135
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][6/25] loss=0.8075 valDice(A∪V)=0.5790 valClDice(A∪V)=0.5914 valDice(art)=0.4224 valClDice(art)=0.2331 valDice(vein)=0.2786 valClDice(vein)=0.3434
  [train_av_refine] batch 1/560 loss=0.7920
  [train_av_refine] batch 50/560 loss=0.7228
  [train_av_refine] batch 100/560 loss=0.8532
  [train_av_refine] batch 150/560 loss=0.7178
  [train_av_refine] batch 200/560 loss=0.9349
  [train_av_refine] batch 250/560 loss=0.7343
  [train_av_refine] batch 300/560 loss=0.8452
  [train_av_refine] batch 350/560 loss=0.7033
  [train_av_refine] batch 400/560 loss=0.7260
  [train_av_refine] batch 450/560 loss=0.7423
  [train_av_refine] batch 500/560 loss=0.6503
  [train_av_refine] batch 550/560 loss=0.6594
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][7/25] loss=0.7798 valDice(A∪V)=0.5817 valClDice(A∪V)=0.5941 valDice(art)=0.4216 valClDice(art)=0.2348 valDice(vein)=0.2813 valClDice(vein)=0.3438
  [train_av_refine] batch 1/560 loss=0.7656
  [train_av_refine] batch 50/560 loss=0.8628
  [train_av_refine] batch 100/560 loss=0.7815
  [train_av_refine] batch 150/560 loss=0.8554
  [train_av_refine] batch 200/560 loss=0.7264
  [train_av_refine] batch 250/560 loss=0.8568
  [train_av_refine] batch 300/560 loss=0.7220
  [train_av_refine] batch 350/560 loss=0.6850
  [train_av_refine] batch 400/560 loss=0.7470
  [train_av_refine] batch 450/560 loss=0.6672
  [train_av_refine] batch 500/560 loss=0.8037
  [train_av_refine] batch 550/560 loss=0.6887
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][8/25] loss=0.7573 valDice(A∪V)=0.5842 valClDice(A∪V)=0.5959 valDice(art)=0.4159 valClDice(art)=0.2298 valDice(vein)=0.2876 valClDice(vein)=0.3479
  [train_av_refine] batch 1/560 loss=0.7290
  [train_av_refine] batch 50/560 loss=0.7960
  [train_av_refine] batch 100/560 loss=0.7651
  [train_av_refine] batch 150/560 loss=0.7570
  [train_av_refine] batch 200/560 loss=0.8083
  [train_av_refine] batch 250/560 loss=0.6792
  [train_av_refine] batch 300/560 loss=0.7611
  [train_av_refine] batch 350/560 loss=0.7766
  [train_av_refine] batch 400/560 loss=0.6353
  [train_av_refine] batch 450/560 loss=0.7856
  [train_av_refine] batch 500/560 loss=0.7468
  [train_av_refine] batch 550/560 loss=0.6976
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][9/25] loss=0.7355 valDice(A∪V)=0.5871 valClDice(A∪V)=0.5985 valDice(art)=0.4096 valClDice(art)=0.2250 valDice(vein)=0.2937 valClDice(vein)=0.3529
  [train_av_refine] batch 1/560 loss=0.7686
  [train_av_refine] batch 50/560 loss=0.7405
  [train_av_refine] batch 100/560 loss=0.6303
  [train_av_refine] batch 150/560 loss=0.6931
  [train_av_refine] batch 200/560 loss=0.6619
  [train_av_refine] batch 250/560 loss=0.7275
  [train_av_refine] batch 300/560 loss=0.7842
  [train_av_refine] batch 350/560 loss=0.6089
  [train_av_refine] batch 400/560 loss=0.6916
  [train_av_refine] batch 450/560 loss=0.6942
  [train_av_refine] batch 500/560 loss=0.7157
  [train_av_refine] batch 550/560 loss=0.7605
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][10/25] loss=0.7182 valDice(A∪V)=0.5905 valClDice(A∪V)=0.6018 valDice(art)=0.4065 valClDice(art)=0.2266 valDice(vein)=0.2972 valClDice(vein)=0.3532
  [train_av_refine] batch 1/560 loss=0.6721
  [train_av_refine] batch 50/560 loss=0.7067
  [train_av_refine] batch 100/560 loss=0.6982
  [train_av_refine] batch 150/560 loss=0.6609
  [train_av_refine] batch 200/560 loss=0.7376
  [train_av_refine] batch 250/560 loss=0.6935
  [train_av_refine] batch 300/560 loss=0.6543
  [train_av_refine] batch 350/560 loss=0.7529
  [train_av_refine] batch 400/560 loss=0.6223
  [train_av_refine] batch 450/560 loss=0.6582
  [train_av_refine] batch 500/560 loss=0.7006
  [train_av_refine] batch 550/560 loss=0.6637
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][11/25] loss=0.7054 valDice(A∪V)=0.5937 valClDice(A∪V)=0.6045 valDice(art)=0.4038 valClDice(art)=0.2307 valDice(vein)=0.3000 valClDice(vein)=0.3518
  [train_av_refine] batch 1/560 loss=0.6859
  [train_av_refine] batch 50/560 loss=0.7051
  [train_av_refine] batch 100/560 loss=0.6695
  [train_av_refine] batch 150/560 loss=0.7427
  [train_av_refine] batch 200/560 loss=0.6522
  [train_av_refine] batch 250/560 loss=0.7165
  [train_av_refine] batch 300/560 loss=0.7668
  [train_av_refine] batch 350/560 loss=0.6767
  [train_av_refine] batch 400/560 loss=0.6933
  [train_av_refine] batch 450/560 loss=0.6998
  [train_av_refine] batch 500/560 loss=0.6945
  [train_av_refine] batch 550/560 loss=0.7463
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][12/25] loss=0.6976 valDice(A∪V)=0.5963 valClDice(A∪V)=0.6067 valDice(art)=0.3972 valClDice(art)=0.2302 valDice(vein)=0.3053 valClDice(vein)=0.3538
  [train_av_refine] batch 1/560 loss=0.6825
  [train_av_refine] batch 50/560 loss=0.6629
  [train_av_refine] batch 100/560 loss=0.6763
  [train_av_refine] batch 150/560 loss=0.7139
  [train_av_refine] batch 200/560 loss=0.7261
  [train_av_refine] batch 250/560 loss=0.7095
  [train_av_refine] batch 300/560 loss=0.6517
  [train_av_refine] batch 350/560 loss=0.7404
  [train_av_refine] batch 400/560 loss=0.6847
  [train_av_refine] batch 450/560 loss=0.6924
  [train_av_refine] batch 500/560 loss=0.6668
  [train_av_refine] batch 550/560 loss=0.7276
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][13/25] loss=0.6907 valDice(A∪V)=0.5995 valClDice(A∪V)=0.6099 valDice(art)=0.4164 valClDice(art)=0.3078 valDice(vein)=0.2922 valClDice(vein)=0.3185
  [train_av_refine] batch 1/560 loss=0.6920
  [train_av_refine] batch 50/560 loss=0.6270
  [train_av_refine] batch 100/560 loss=0.6879
  [train_av_refine] batch 150/560 loss=0.7079
  [train_av_refine] batch 200/560 loss=0.6809
  [train_av_refine] batch 250/560 loss=0.7311
  [train_av_refine] batch 300/560 loss=0.6705
  [train_av_refine] batch 350/560 loss=0.7244
  [train_av_refine] batch 400/560 loss=0.7151
  [train_av_refine] batch 450/560 loss=0.7126
  [train_av_refine] batch 500/560 loss=0.6824
  [train_av_refine] batch 550/560 loss=0.6748
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][14/25] loss=0.6873 valDice(A∪V)=0.6013 valClDice(A∪V)=0.6112 valDice(art)=0.4160 valClDice(art)=0.3237 valDice(vein)=0.2926 valClDice(vein)=0.3109
  [train_av_refine] batch 1/560 loss=0.7328
  [train_av_refine] batch 50/560 loss=0.6728
  [train_av_refine] batch 100/560 loss=0.7116
  [train_av_refine] batch 150/560 loss=0.6677
  [train_av_refine] batch 200/560 loss=0.7163
  [train_av_refine] batch 250/560 loss=0.6555
  [train_av_refine] batch 300/560 loss=0.6397
  [train_av_refine] batch 350/560 loss=0.6875
  [train_av_refine] batch 400/560 loss=0.6761
  [train_av_refine] batch 450/560 loss=0.6808
  [train_av_refine] batch 500/560 loss=0.7009
  [train_av_refine] batch 550/560 loss=0.6982
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][15/25] loss=0.6846 valDice(A∪V)=0.6035 valClDice(A∪V)=0.6127 valDice(art)=0.4331 valClDice(art)=0.3773 valDice(vein)=0.2716 valClDice(vein)=0.2646
  [train_av_refine] batch 1/560 loss=0.7112
  [train_av_refine] batch 50/560 loss=0.6807
  [train_av_refine] batch 100/560 loss=0.6826
  [train_av_refine] batch 150/560 loss=0.6547
  [train_av_refine] batch 200/560 loss=0.6677
  [train_av_refine] batch 250/560 loss=0.7159
  [train_av_refine] batch 300/560 loss=0.6540
  [train_av_refine] batch 350/560 loss=0.7055
  [train_av_refine] batch 400/560 loss=0.6555
  [train_av_refine] batch 450/560 loss=0.6992
  [train_av_refine] batch 500/560 loss=0.6779
  [train_av_refine] batch 550/560 loss=0.6605
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][16/25] loss=0.6824 valDice(A∪V)=0.6048 valClDice(A∪V)=0.6139 valDice(art)=0.4356 valClDice(art)=0.3867 valDice(vein)=0.2633 valClDice(vein)=0.2446
  [train_av_refine] batch 1/560 loss=0.7324
  [train_av_refine] batch 50/560 loss=0.6406
  [train_av_refine] batch 100/560 loss=0.7268
  [train_av_refine] batch 150/560 loss=0.6767
  [train_av_refine] batch 200/560 loss=0.6856
  [train_av_refine] batch 250/560 loss=0.7004
  [train_av_refine] batch 300/560 loss=0.6678
  [train_av_refine] batch 350/560 loss=0.6486
  [train_av_refine] batch 400/560 loss=0.6914
  [train_av_refine] batch 450/560 loss=0.7141
  [train_av_refine] batch 500/560 loss=0.6658
  [train_av_refine] batch 550/560 loss=0.7284
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][17/25] loss=0.6838 valDice(A∪V)=0.6052 valClDice(A∪V)=0.6142 valDice(art)=0.4238 valClDice(art)=0.3660 valDice(vein)=0.2786 valClDice(vein)=0.2713
  [train_av_refine] batch 1/560 loss=0.7043
  [train_av_refine] batch 50/560 loss=0.7066
  [train_av_refine] batch 100/560 loss=0.6625
  [train_av_refine] batch 150/560 loss=0.6851
  [train_av_refine] batch 200/560 loss=0.6630
  [train_av_refine] batch 250/560 loss=0.6576
  [train_av_refine] batch 300/560 loss=0.7224
  [train_av_refine] batch 350/560 loss=0.6802
  [train_av_refine] batch 400/560 loss=0.6835
  [train_av_refine] batch 450/560 loss=0.6974
  [train_av_refine] batch 500/560 loss=0.7041
  [train_av_refine] batch 550/560 loss=0.7047
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][18/25] loss=0.6822 valDice(A∪V)=0.6057 valClDice(A∪V)=0.6146 valDice(art)=0.4173 valClDice(art)=0.3559 valDice(vein)=0.2842 valClDice(vein)=0.2775
  [train_av_refine] batch 1/560 loss=0.6792
  [train_av_refine] batch 50/560 loss=0.6820
  [train_av_refine] batch 100/560 loss=0.6776
  [train_av_refine] batch 150/560 loss=0.6524
  [train_av_refine] batch 200/560 loss=0.6678
  [train_av_refine] batch 250/560 loss=0.6926
  [train_av_refine] batch 300/560 loss=0.6570
  [train_av_refine] batch 350/560 loss=0.6722
  [train_av_refine] batch 400/560 loss=0.6817
  [train_av_refine] batch 450/560 loss=0.6679
  [train_av_refine] batch 500/560 loss=0.6633
  [train_av_refine] batch 550/560 loss=0.6658
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][19/25] loss=0.6834 valDice(A∪V)=0.6069 valClDice(A∪V)=0.6155 valDice(art)=0.4253 valClDice(art)=0.3772 valDice(vein)=0.2675 valClDice(vein)=0.2421
  [train_av_refine] batch 1/560 loss=0.6700
  [train_av_refine] batch 50/560 loss=0.6698
  [train_av_refine] batch 100/560 loss=0.6550
  [train_av_refine] batch 150/560 loss=0.6773
  [train_av_refine] batch 200/560 loss=0.6908
  [train_av_refine] batch 250/560 loss=0.6416
  [train_av_refine] batch 300/560 loss=0.6599
  [train_av_refine] batch 350/560 loss=0.6637
  [train_av_refine] batch 400/560 loss=0.6450
  [train_av_refine] batch 450/560 loss=0.6723
  [train_av_refine] batch 500/560 loss=0.6766
  [train_av_refine] batch 550/560 loss=0.6859
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][20/25] loss=0.6835 valDice(A∪V)=0.6071 valClDice(A∪V)=0.6157 valDice(art)=0.4210 valClDice(art)=0.3706 valDice(vein)=0.2729 valClDice(vein)=0.2522
  [train_av_refine] batch 1/560 loss=0.6765
  [train_av_refine] batch 50/560 loss=0.7006
  [train_av_refine] batch 100/560 loss=0.7095
  [train_av_refine] batch 150/560 loss=0.6724
  [train_av_refine] batch 200/560 loss=0.6330
  [train_av_refine] batch 250/560 loss=0.7018
  [train_av_refine] batch 300/560 loss=0.6906
  [train_av_refine] batch 350/560 loss=0.6768
  [train_av_refine] batch 400/560 loss=0.6472
  [train_av_refine] batch 450/560 loss=0.6966
  [train_av_refine] batch 500/560 loss=0.6793
  [train_av_refine] batch 550/560 loss=0.6775
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][21/25] loss=0.6810 valDice(A∪V)=0.6075 valClDice(A∪V)=0.6161 valDice(art)=0.4231 valClDice(art)=0.3737 valDice(vein)=0.2682 valClDice(vein)=0.2430
  [train_av_refine] batch 1/560 loss=0.7166
  [train_av_refine] batch 50/560 loss=0.6719
  [train_av_refine] batch 100/560 loss=0.6773
  [train_av_refine] batch 150/560 loss=0.6913
  [train_av_refine] batch 200/560 loss=0.6733
  [train_av_refine] batch 250/560 loss=0.6986
  [train_av_refine] batch 300/560 loss=0.6546
  [train_av_refine] batch 350/560 loss=0.6473
  [train_av_refine] batch 400/560 loss=0.6989
  [train_av_refine] batch 450/560 loss=0.6926
  [train_av_refine] batch 500/560 loss=0.6604
  [train_av_refine] batch 550/560 loss=0.6840
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][22/25] loss=0.6814 valDice(A∪V)=0.6081 valClDice(A∪V)=0.6165 valDice(art)=0.4282 valClDice(art)=0.3854 valDice(vein)=0.2537 valClDice(vein)=0.2135
  [train_av_refine] batch 1/560 loss=0.6705
  [train_av_refine] batch 50/560 loss=0.6700
  [train_av_refine] batch 100/560 loss=0.6668
  [train_av_refine] batch 150/560 loss=0.6791
  [train_av_refine] batch 200/560 loss=0.6917
  [train_av_refine] batch 250/560 loss=0.6746
  [train_av_refine] batch 300/560 loss=0.6707
  [train_av_refine] batch 350/560 loss=0.6684
  [train_av_refine] batch 400/560 loss=0.7023
  [train_av_refine] batch 450/560 loss=0.7070
  [train_av_refine] batch 500/560 loss=0.6811
  [train_av_refine] batch 550/560 loss=0.6775
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][23/25] loss=0.6825 valDice(A∪V)=0.6081 valClDice(A∪V)=0.6165 valDice(art)=0.4224 valClDice(art)=0.3737 valDice(vein)=0.2653 valClDice(vein)=0.2353
  [train_av_refine] batch 1/560 loss=0.6766
  [train_av_refine] batch 50/560 loss=0.6589
  [train_av_refine] batch 100/560 loss=0.6740
  [train_av_refine] batch 150/560 loss=0.6885
  [train_av_refine] batch 200/560 loss=0.6734
  [train_av_refine] batch 250/560 loss=0.6699
  [train_av_refine] batch 300/560 loss=0.6850
  [train_av_refine] batch 350/560 loss=0.6863
  [train_av_refine] batch 400/560 loss=0.6799
  [train_av_refine] batch 450/560 loss=0.6822
  [train_av_refine] batch 500/560 loss=0.6716
  [train_av_refine] batch 550/560 loss=0.6615
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][24/25] loss=0.6816 valDice(A∪V)=0.6083 valClDice(A∪V)=0.6166 valDice(art)=0.4206 valClDice(art)=0.3705 valDice(vein)=0.2679 valClDice(vein)=0.2407
  [train_av_refine] batch 1/560 loss=0.6730
  [train_av_refine] batch 50/560 loss=0.6685
  [train_av_refine] batch 100/560 loss=0.6951
  [train_av_refine] batch 150/560 loss=0.6976
  [train_av_refine] batch 200/560 loss=0.7110
  [train_av_refine] batch 250/560 loss=0.6886
  [train_av_refine] batch 300/560 loss=0.6508
  [train_av_refine] batch 350/560 loss=0.7246
  [train_av_refine] batch 400/560 loss=0.6801
  [train_av_refine] batch 450/560 loss=0.6573
  [train_av_refine] batch 500/560 loss=0.6771
  [train_av_refine] batch 550/560 loss=0.6490
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[eval_epoch] clDice over 35 cases with 11 workers
[S3][25/25] loss=0.6824 valDice(A∪V)=0.6085 valClDice(A∪V)=0.6168 valDice(art)=0.4214 valClDice(art)=0.3722 valDice(vein)=0.2652 valClDice(vein)=0.2353

From here we get the final av_ct_best_cldice.pt checkpoint that is then fed into inference.py/eval_inference.py (they do the same thing, one just works independently and does image preprocessing automatically for use on the docker container).